# Governance-Aware Hybrid Decision Engine

**Companion notebook for:**  
*Governance-Aware Hybrid Decision Fusion for Data Quality Anomaly Detection in Cloud Lakehouses: A Multi-Domain Evaluation*

**Author:** Ramesh Babu Kallam  
**ORCID:** 0009-0008-5220-1775

## Purpose

This notebook loads the baseline artifacts, constructs the record-level evidence-fusion dataset, evaluates the hybrid policies, performs nested experiment-level calibration, runs statistical tests, and exports publication-ready artifacts.

## How to Run

1. Run `01_Baseline_Prototype.ipynb` first.
2. Confirm that its generated artifacts are present under `results/`.
3. Run all cells in this notebook from top to bottom.
4. Inspect the generated files under `figures/`, `tables/`, and `results/`.

The default path configuration supports a local repository clone, GitHub Codespaces, and Jupyter. Google Colab remains optional.

---

## 1. Environment Setup and Artifact Validation


In [ ]:
# ================================================================
# 02_Hybrid_Decision_Engine.ipynb
# Environment setup and baseline artifact validation
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

try:
    from google.colab import drive
    IN_GOOGLE_COLAB = True
except ImportError:
    drive = None
    IN_GOOGLE_COLAB = False


# ----------------------------------------------------------------
# Reproducibility configuration
# ----------------------------------------------------------------
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)


# ----------------------------------------------------------------
# Project paths
# ----------------------------------------------------------------
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/governance-aware-hybrid-data-quality"
)

if USE_GOOGLE_DRIVE:
    if not IN_GOOGLE_COLAB:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE=True requires a Google Colab runtime."
        )
    drive.mount("/content/drive")
    PROJECT_ROOT = GOOGLE_DRIVE_PROJECT_ROOT
else:
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = (
        current_dir.parent
        if current_dir.name == "notebooks"
        else current_dir
    )

DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
EXPERIMENT_DIR = DATA_DIR / "experiments"

RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"

HYBRID_ROOT = RESULTS_DIR / "hybrid_decision_engine"
HYBRID_DATA_DIR = HYBRID_ROOT / "data"
HYBRID_RESULTS_DIR = HYBRID_ROOT / "results"
HYBRID_CONFIG_DIR = PROJECT_ROOT / "configs"
HYBRID_AUDIT_DIR = HYBRID_ROOT / "audit"


# ----------------------------------------------------------------
# Create hybrid module directories
# ----------------------------------------------------------------
for directory in [
    HYBRID_ROOT,
    HYBRID_DATA_DIR,
    HYBRID_RESULTS_DIR,
    HYBRID_CONFIG_DIR,
    HYBRID_AUDIT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ----------------------------------------------------------------
# Required baseline artifacts
# ----------------------------------------------------------------
BASELINE_ARTIFACTS = {
    "rule_record_results": (
        RESULTS_DIR / "rule_record_results.parquet"
    ),
    "rule_violations": (
        RESULTS_DIR / "rule_violations.parquet"
    ),
    "anomaly_ground_truth": (
        RESULTS_DIR / "anomaly_ground_truth.parquet"
    ),
    "experiment_registry": (
        RESULTS_DIR / "experiment_registry.csv"
    ),
    "ai_scoring_completed": (
        RESULTS_DIR / "ai_scoring_completed.csv"
    ),
}

AI_RESULT_DIR = (
    RESULTS_DIR / "ai_results_by_experiment"
)


# ----------------------------------------------------------------
# Validate required files
# ----------------------------------------------------------------
validation_rows = []

for artifact_name, artifact_path in (
    BASELINE_ARTIFACTS.items()
):
    validation_rows.append({
        "artifact": artifact_name,
        "path": str(artifact_path),
        "exists": artifact_path.exists(),
        "size_mb": (
            round(
                artifact_path.stat().st_size
                / (1024 ** 2),
                3
            )
            if artifact_path.exists()
            else None
        ),
    })

validation_df = pd.DataFrame(
    validation_rows
)

display(validation_df)


# ----------------------------------------------------------------
# Validate AI experiment output folder
# ----------------------------------------------------------------
ai_result_files = sorted(
    AI_RESULT_DIR.glob(
        "*_ai_results.parquet"
    )
)

print(
    f"AI result files found: "
    f"{len(ai_result_files)}"
)


# ----------------------------------------------------------------
# Fail early if required artifacts are missing
# ----------------------------------------------------------------
missing_artifacts = validation_df.loc[
    ~validation_df["exists"],
    "artifact"
].tolist()

if missing_artifacts:
    raise FileNotFoundError(
        "Missing required baseline artifacts: "
        + ", ".join(missing_artifacts)
    )

if len(ai_result_files) == 0:
    raise FileNotFoundError(
        f"No AI result files found in: "
        f"{AI_RESULT_DIR}"
    )


# ----------------------------------------------------------------
# Execution metadata
# ----------------------------------------------------------------
RUN_TIMESTAMP_UTC = datetime.now(
    timezone.utc
).isoformat()

HYBRID_RUN_ID = (
    "HYBRID_"
    + datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%SZ")
)

execution_metadata = {
    "hybrid_run_id": HYBRID_RUN_ID,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "random_seed": RANDOM_SEED,
    "project_root": str(PROJECT_ROOT),
    "ai_result_directory": str(
        AI_RESULT_DIR
    ),
    "ai_result_file_count": len(
        ai_result_files
    ),
}

metadata_path = (
    HYBRID_AUDIT_DIR
    / "hybrid_execution_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as metadata_file:
    json.dump(
        execution_metadata,
        metadata_file,
        indent=2
    )


# ----------------------------------------------------------------
# Final setup confirmation
# ----------------------------------------------------------------
print("-" * 80)
print("Hybrid decision-engine setup completed.")
print(f"Hybrid run ID: {HYBRID_RUN_ID}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Hybrid workspace: {HYBRID_ROOT}")
print(
    f"Validated baseline artifacts: "
    f"{len(validation_df)}"
)
print(
    f"AI result files available: "
    f"{len(ai_result_files)}"
)
print(
    f"Execution metadata saved to: "
    f"{metadata_path}"
)

## 2. Load and Standardize Baseline Artifacts

This section loads deterministic-rule, anomaly-ground-truth, experiment-registry, and AI-detector outputs and verifies their schemas before evidence fusion.


In [ ]:
# ================================================================
# Cell 2 — Load and standardize baseline artifacts
# ================================================================

import gc
import time

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# Load baseline artifacts
# ----------------------------------------------------------------
load_started = time.perf_counter()

rule_record_results_df = pd.read_parquet(
    BASELINE_ARTIFACTS["rule_record_results"]
)

rule_violations_df = pd.read_parquet(
    BASELINE_ARTIFACTS["rule_violations"]
)

anomaly_ground_truth_df = pd.read_parquet(
    BASELINE_ARTIFACTS["anomaly_ground_truth"]
)

experiment_registry_df = pd.read_csv(
    BASELINE_ARTIFACTS["experiment_registry"]
)

ai_scoring_completed_df = pd.read_csv(
    BASELINE_ARTIFACTS["ai_scoring_completed"]
)

print("Loaded baseline tabular artifacts.")


# ----------------------------------------------------------------
# Standardize key identifier columns
# ----------------------------------------------------------------
for df_name, df in [
    ("rule_record_results_df", rule_record_results_df),
    ("rule_violations_df", rule_violations_df),
    ("anomaly_ground_truth_df", anomaly_ground_truth_df),
    ("experiment_registry_df", experiment_registry_df),
    ("ai_scoring_completed_df", ai_scoring_completed_df),
]:
    for column in [
        "dataset",
        "experiment_id",
        "record_id",
        "_record_id",
    ]:
        if column in df.columns:
            df[column] = df[column].astype("string")

print("Standardized identifier columns.")


# ----------------------------------------------------------------
# Inspect baseline schemas
# ----------------------------------------------------------------
schema_summary_rows = []

for dataframe_name, dataframe in [
    ("rule_record_results", rule_record_results_df),
    ("rule_violations", rule_violations_df),
    ("anomaly_ground_truth", anomaly_ground_truth_df),
    ("experiment_registry", experiment_registry_df),
    ("ai_scoring_completed", ai_scoring_completed_df),
]:
    schema_summary_rows.append({
        "dataframe": dataframe_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "column_names": ", ".join(
            dataframe.columns.astype(str).tolist()
        ),
    })

schema_summary_df = pd.DataFrame(
    schema_summary_rows
)

display(schema_summary_df)


# ----------------------------------------------------------------
# Load all AI result files incrementally
# ----------------------------------------------------------------
ai_result_frames = []

total_ai_files = len(ai_result_files)

print(
    f"Loading {total_ai_files} AI result files..."
)

for position, file_path in enumerate(
    ai_result_files,
    start=1
):
    ai_part_df = pd.read_parquet(
        file_path
    )

    for column in [
        "dataset",
        "experiment_id",
        "record_id",
        "detector",
        "anomaly_type",
    ]:
        if column in ai_part_df.columns:
            ai_part_df[column] = (
                ai_part_df[column]
                .astype("string")
            )

    ai_part_df["source_file"] = (
        file_path.name
    )

    ai_result_frames.append(
        ai_part_df
    )

    if (
        position == 1
        or position % 10 == 0
        or position == total_ai_files
    ):
        print(
            f"Loaded {position}/{total_ai_files} files"
        )


ai_results_df = pd.concat(
    ai_result_frames,
    ignore_index=True
)

del ai_result_frames
gc.collect()

print(
    f"Combined AI results: "
    f"{len(ai_results_df):,} rows"
)


# ----------------------------------------------------------------
# Validate required AI columns
# ----------------------------------------------------------------
required_ai_columns = {
    "experiment_id",
    "dataset",
    "record_id",
    "detector",
    "anomaly_score",
    "anomaly_prediction",
}

missing_ai_columns = (
    required_ai_columns
    - set(ai_results_df.columns)
)

if missing_ai_columns:
    raise KeyError(
        "Missing required AI-result columns: "
        + ", ".join(
            sorted(missing_ai_columns)
        )
    )


# ----------------------------------------------------------------
# Normalize ground-truth label
# ----------------------------------------------------------------
if "ground_truth_label" not in anomaly_ground_truth_df.columns:
    raise KeyError(
        "ground_truth_label is missing from "
        "anomaly_ground_truth_df"
    )

anomaly_ground_truth_df[
    "ground_truth_label"
] = (
    pd.to_numeric(
        anomaly_ground_truth_df[
            "ground_truth_label"
        ],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)


# ----------------------------------------------------------------
# Standardize experiment metadata
# ----------------------------------------------------------------
if "anomaly_rate" in experiment_registry_df.columns:
    experiment_registry_df[
        "anomaly_rate"
    ] = pd.to_numeric(
        experiment_registry_df[
            "anomaly_rate"
        ],
        errors="coerce"
    )

if "anomaly_rate" in ai_results_df.columns:
    ai_results_df[
        "anomaly_rate"
    ] = pd.to_numeric(
        ai_results_df[
            "anomaly_rate"
        ],
        errors="coerce"
    )


# ----------------------------------------------------------------
# Join AI results with explicit ground truth
# ----------------------------------------------------------------
ground_truth_keys_df = (
    anomaly_ground_truth_df[
        [
            "experiment_id",
            "record_id",
            "ground_truth_label",
        ]
    ]
    .drop_duplicates(
        subset=[
            "experiment_id",
            "record_id",
        ]
    )
)

ai_with_truth_df = ai_results_df.merge(
    ground_truth_keys_df,
    on=[
        "experiment_id",
        "record_id",
    ],
    how="left",
    validate="many_to_one",
)

ai_with_truth_df[
    "ground_truth_label"
] = (
    ai_with_truth_df[
        "ground_truth_label"
    ]
    .fillna(0)
    .astype(int)
)


# ----------------------------------------------------------------
# Add experiment metadata
# ----------------------------------------------------------------
experiment_metadata_columns = [
    column
    for column in [
        "experiment_id",
        "dataset",
        "anomaly_type",
        "anomaly_rate",
        "seed",
    ]
    if column in experiment_registry_df.columns
]

experiment_metadata_df = (
    experiment_registry_df[
        experiment_metadata_columns
    ]
    .drop_duplicates(
        subset=["experiment_id"]
    )
)

metadata_merge_columns = [
    column
    for column in experiment_metadata_columns
    if column != "experiment_id"
    and column not in ai_with_truth_df.columns
]

if metadata_merge_columns:
    ai_with_truth_df = (
        ai_with_truth_df.merge(
            experiment_metadata_df[
                ["experiment_id"]
                + metadata_merge_columns
            ],
            on="experiment_id",
            how="left",
            validate="many_to_one",
        )
    )


# ----------------------------------------------------------------
# Summarize AI data coverage
# ----------------------------------------------------------------
ai_coverage_df = (
    ai_with_truth_df
    .groupby(
        [
            "dataset",
            "detector",
        ],
        dropna=False
    )
    .agg(
        experiments=(
            "experiment_id",
            "nunique"
        ),
        result_rows=(
            "record_id",
            "size"
        ),
        unique_records=(
            "record_id",
            "nunique"
        ),
        true_anomalies=(
            "ground_truth_label",
            "sum"
        ),
        predicted_anomalies=(
            "anomaly_prediction",
            "sum"
        ),
    )
    .reset_index()
)

display(ai_coverage_df)


# ----------------------------------------------------------------
# Rule-output diagnostics
# ----------------------------------------------------------------
rule_diagnostic_rows = []

for dataframe_name, dataframe in [
    (
        "rule_record_results",
        rule_record_results_df
    ),
    (
        "rule_violations",
        rule_violations_df
    ),
]:
    rule_diagnostic_rows.append({
        "dataframe": dataframe_name,
        "rows": len(dataframe),
        "columns": len(
            dataframe.columns
        ),
        "has_dataset": (
            "dataset"
            in dataframe.columns
        ),
        "has_record_id": (
            "record_id"
            in dataframe.columns
            or "_record_id"
            in dataframe.columns
        ),
        "has_rule_status": any(
            column in dataframe.columns
            for column in [
                "rule_passed",
                "rule_failed",
                "is_valid",
                "violation_flag",
                "rule_result",
            ]
        ),
        "has_trust_score": any(
            column in dataframe.columns
            for column in [
                "trust_score",
                "record_trust_score",
                "quality_score",
            ]
        ),
    })

rule_diagnostics_df = pd.DataFrame(
    rule_diagnostic_rows
)

display(rule_diagnostics_df)


# ----------------------------------------------------------------
# Save standardized AI-with-ground-truth artifact
# ----------------------------------------------------------------
AI_STANDARDIZED_PATH = (
    HYBRID_DATA_DIR
    / "ai_results_with_ground_truth.parquet"
)

ai_with_truth_df.to_parquet(
    AI_STANDARDIZED_PATH,
    index=False
)


# ----------------------------------------------------------------
# Save loading audit
# ----------------------------------------------------------------
load_runtime_seconds = (
    time.perf_counter()
    - load_started
)

load_audit_df = pd.DataFrame([
    {
        "hybrid_run_id": HYBRID_RUN_ID,
        "loaded_timestamp_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "rule_record_rows": len(
            rule_record_results_df
        ),
        "rule_violation_rows": len(
            rule_violations_df
        ),
        "ground_truth_rows": len(
            anomaly_ground_truth_df
        ),
        "experiment_registry_rows": len(
            experiment_registry_df
        ),
        "ai_result_rows": len(
            ai_results_df
        ),
        "ai_with_truth_rows": len(
            ai_with_truth_df
        ),
        "ai_result_files": len(
            ai_result_files
        ),
        "load_runtime_seconds": (
            load_runtime_seconds
        ),
    }
])

LOAD_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "artifact_loading_audit.csv"
)

load_audit_df.to_csv(
    LOAD_AUDIT_PATH,
    index=False
)


# ----------------------------------------------------------------
# Final validation
# ----------------------------------------------------------------
checks = {
    "all_63_ai_files_loaded": (
        len(ai_result_files) == 63
    ),
    "ai_results_not_empty": (
        len(ai_results_df) > 0
    ),
    "ground_truth_join_complete": (
        ai_with_truth_df[
            "ground_truth_label"
        ].notna().all()
    ),
    "two_detectors_present": (
        ai_with_truth_df[
            "detector"
        ].nunique() == 2
    ),
    "three_datasets_present": (
        ai_with_truth_df[
            "dataset"
        ].nunique() == 3
    ),
}

checks_df = pd.DataFrame({
    "check": checks.keys(),
    "passed": checks.values(),
})

display(checks_df)

assert checks_df["passed"].all(), (
    "One or more artifact-loading checks failed."
)


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print("Baseline artifacts loaded successfully.")
print(
    f"Rule record rows: "
    f"{len(rule_record_results_df):,}"
)
print(
    f"Rule violation rows: "
    f"{len(rule_violations_df):,}"
)
print(
    f"Ground-truth rows: "
    f"{len(anomaly_ground_truth_df):,}"
)
print(
    f"AI result rows: "
    f"{len(ai_results_df):,}"
)
print(
    f"AI-with-truth rows: "
    f"{len(ai_with_truth_df):,}"
)
print(
    f"Standardized AI artifact: "
    f"{AI_STANDARDIZED_PATH}"
)
print(
    f"Loading audit: "
    f"{LOAD_AUDIT_PATH}"
)
print(
    f"Load runtime: "
    f"{load_runtime_seconds:.2f} seconds"
)

## 3. Construct the Record-Level Hybrid Evidence Dataset

This section aligns rule evidence, detector scores, detector votes, consensus, uncertainty, and ground truth at record level.


In [ ]:
# ================================================================
# Cell 3 — Build the record-level hybrid decision dataset
# ================================================================

import gc
import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# Display source schemas for traceability
# ----------------------------------------------------------------
print("Rule-record columns:")
print(rule_record_results_df.columns.tolist())

print("\nRule-violation columns:")
print(rule_violations_df.columns.tolist())

print("\nAI-with-ground-truth columns:")
print(ai_with_truth_df.columns.tolist())


# ----------------------------------------------------------------
# Standardize rule-record identifier
# ----------------------------------------------------------------
rule_records_df = rule_record_results_df.copy()

if "_record_id" in rule_records_df.columns:
    rule_records_df = rule_records_df.rename(
        columns={"_record_id": "record_id"}
    )
elif "record_id" not in rule_records_df.columns:
    raise KeyError(
        "No record identifier found in rule_record_results_df."
    )

rule_records_df["record_id"] = (
    rule_records_df["record_id"]
    .astype("string")
)

rule_records_df["dataset"] = (
    rule_records_df["dataset"]
    .astype("string")
)


# ----------------------------------------------------------------
# Detect and standardize rule-result columns
# ----------------------------------------------------------------
failed_rule_count_candidates = [
    "failed_rule_count",
    "rule_failure_count",
    "violation_count",
    "failed_rules",
]

trust_score_candidates = [
    "trust_score",
    "record_trust_score",
    "quality_score",
]

severity_candidates = [
    "maximum_severity",
    "max_severity",
    "severity_score",
    "highest_severity",
]

rule_flag_candidates = [
    "rule_prediction",
    "rule_anomaly_prediction",
    "violation_flag",
    "is_invalid",
    "any_rule_failed",
]


def first_existing_column(
    dataframe: pd.DataFrame,
    candidates: list[str],
):
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate
    return None


failed_rule_count_column = first_existing_column(
    rule_records_df,
    failed_rule_count_candidates,
)

trust_score_column = first_existing_column(
    rule_records_df,
    trust_score_candidates,
)

severity_column = first_existing_column(
    rule_records_df,
    severity_candidates,
)

rule_flag_column = first_existing_column(
    rule_records_df,
    rule_flag_candidates,
)


# ----------------------------------------------------------------
# Create standardized rule features
# ----------------------------------------------------------------
if failed_rule_count_column is not None:
    rule_records_df["failed_rule_count_std"] = (
        pd.to_numeric(
            rule_records_df[
                failed_rule_count_column
            ],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )
else:
    rule_records_df["failed_rule_count_std"] = 0


if trust_score_column is not None:
    rule_records_df["rule_trust_score"] = (
        pd.to_numeric(
            rule_records_df[
                trust_score_column
            ],
            errors="coerce",
        )
        .fillna(1.0)
        .clip(0.0, 1.0)
    )
else:
    rule_records_df["rule_trust_score"] = (
        1.0
        - np.minimum(
            rule_records_df[
                "failed_rule_count_std"
            ],
            5,
        )
        / 5.0
    )


if severity_column is not None:
    rule_records_df["rule_severity_raw"] = (
        rule_records_df[
            severity_column
        ]
        .astype("string")
        .str.upper()
    )
else:
    rule_records_df["rule_severity_raw"] = (
        "NONE"
    )


severity_map = {
    "NONE": 0.00,
    "INFO": 0.10,
    "LOW": 0.25,
    "MEDIUM": 0.50,
    "HIGH": 0.75,
    "CRITICAL": 1.00,
    "1": 0.25,
    "2": 0.50,
    "3": 0.75,
    "4": 1.00,
}

rule_records_df["rule_severity_score"] = (
    rule_records_df[
        "rule_severity_raw"
    ]
    .map(severity_map)
)

numeric_severity = pd.to_numeric(
    rule_records_df[
        "rule_severity_raw"
    ],
    errors="coerce",
)

rule_records_df[
    "rule_severity_score"
] = (
    rule_records_df[
        "rule_severity_score"
    ]
    .fillna(
        numeric_severity
    )
    .fillna(0.0)
    .clip(0.0, 1.0)
)


if rule_flag_column is not None:
    rule_flag_values = (
        rule_records_df[
            rule_flag_column
        ]
    )

    if pd.api.types.is_bool_dtype(
        rule_flag_values
    ):
        rule_records_df[
            "rule_anomaly_prediction"
        ] = (
            rule_flag_values
            .fillna(False)
            .astype(int)
        )
    else:
        rule_records_df[
            "rule_anomaly_prediction"
        ] = (
            pd.to_numeric(
                rule_flag_values,
                errors="coerce",
            )
            .fillna(
                (
                    rule_records_df[
                        "failed_rule_count_std"
                    ] > 0
                ).astype(int)
            )
            .clip(0, 1)
            .astype(int)
        )
else:
    rule_records_df[
        "rule_anomaly_prediction"
    ] = (
        rule_records_df[
            "failed_rule_count_std"
        ] > 0
    ).astype(int)


# ----------------------------------------------------------------
# Standardized rule risk
# ----------------------------------------------------------------
rule_failure_component = np.minimum(
    rule_records_df[
        "failed_rule_count_std"
    ],
    5,
) / 5.0

rule_trust_penalty = (
    1.0
    - rule_records_df[
        "rule_trust_score"
    ]
)

rule_records_df["rule_risk_score"] = (
    0.45 * rule_failure_component
    + 0.35 * rule_trust_penalty
    + 0.20 * rule_records_df[
        "rule_severity_score"
    ]
).clip(0.0, 1.0)


# ----------------------------------------------------------------
# Keep one rule record per dataset and record
# ----------------------------------------------------------------
standardized_rule_df = (
    rule_records_df[
        [
            "dataset",
            "record_id",
            "failed_rule_count_std",
            "rule_trust_score",
            "rule_severity_score",
            "rule_anomaly_prediction",
            "rule_risk_score",
        ]
    ]
    .sort_values(
        [
            "dataset",
            "record_id",
            "rule_risk_score",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .drop_duplicates(
        subset=[
            "dataset",
            "record_id",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "\nStandardized rule records:",
    f"{len(standardized_rule_df):,}",
)


# ----------------------------------------------------------------
# Normalize detector names
# ----------------------------------------------------------------
ai_long_df = ai_with_truth_df.copy()

ai_long_df["detector"] = (
    ai_long_df["detector"]
    .astype("string")
    .str.upper()
    .str.strip()
)

ai_long_df["record_id"] = (
    ai_long_df["record_id"]
    .astype("string")
)

ai_long_df["dataset"] = (
    ai_long_df["dataset"]
    .astype("string")
)

ai_long_df["experiment_id"] = (
    ai_long_df["experiment_id"]
    .astype("string")
)


# ----------------------------------------------------------------
# Convert AI fields to numeric
# ----------------------------------------------------------------
ai_long_df["anomaly_score"] = (
    pd.to_numeric(
        ai_long_df[
            "anomaly_score"
        ],
        errors="coerce",
    )
)

ai_long_df["anomaly_prediction"] = (
    pd.to_numeric(
        ai_long_df[
            "anomaly_prediction"
        ],
        errors="coerce",
    )
    .fillna(0)
    .clip(0, 1)
    .astype(int)
)


# ----------------------------------------------------------------
# Normalize anomaly scores within each experiment and detector
# ----------------------------------------------------------------
# Percentile-rank normalization avoids assuming that Isolation
# Forest and LOF scores share the same numerical scale.
ai_long_df[
    "normalized_anomaly_score"
] = (
    ai_long_df
    .groupby(
        [
            "experiment_id",
            "detector",
        ],
        dropna=False,
    )[
        "anomaly_score"
    ]
    .rank(
        method="average",
        pct=True,
    )
    .fillna(0.0)
    .clip(0.0, 1.0)
)


# ----------------------------------------------------------------
# Pivot detector scores into one row per experiment record
# ----------------------------------------------------------------
score_wide_df = (
    ai_long_df
    .pivot_table(
        index=[
            "experiment_id",
            "dataset",
            "record_id",
        ],
        columns="detector",
        values="normalized_anomaly_score",
        aggfunc="max",
    )
    .reset_index()
)

prediction_wide_df = (
    ai_long_df
    .pivot_table(
        index=[
            "experiment_id",
            "dataset",
            "record_id",
        ],
        columns="detector",
        values="anomaly_prediction",
        aggfunc="max",
    )
    .reset_index()
)


# ----------------------------------------------------------------
# Flatten pivoted column names
# ----------------------------------------------------------------
score_wide_df.columns.name = None
prediction_wide_df.columns.name = None

score_rename_map = {
    "ISOLATION_FOREST": (
        "iforest_normalized_score"
    ),
    "LOCAL_OUTLIER_FACTOR": (
        "lof_normalized_score"
    ),
}

prediction_rename_map = {
    "ISOLATION_FOREST": (
        "iforest_prediction"
    ),
    "LOCAL_OUTLIER_FACTOR": (
        "lof_prediction"
    ),
}

score_wide_df = score_wide_df.rename(
    columns=score_rename_map
)

prediction_wide_df = (
    prediction_wide_df.rename(
        columns=prediction_rename_map
    )
)


# ----------------------------------------------------------------
# Merge detector scores and predictions
# ----------------------------------------------------------------
hybrid_base_df = score_wide_df.merge(
    prediction_wide_df,
    on=[
        "experiment_id",
        "dataset",
        "record_id",
    ],
    how="outer",
    validate="one_to_one",
)


# ----------------------------------------------------------------
# Add experiment-level metadata and ground truth
# ----------------------------------------------------------------
record_metadata_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ground_truth_label",
]

for optional_column in [
    "anomaly_type",
    "anomaly_rate",
]:
    if optional_column in ai_long_df.columns:
        record_metadata_columns.append(
            optional_column
        )

record_metadata_df = (
    ai_long_df[
        record_metadata_columns
    ]
    .drop_duplicates(
        subset=[
            "experiment_id",
            "dataset",
            "record_id",
        ]
    )
)

hybrid_base_df = hybrid_base_df.merge(
    record_metadata_df,
    on=[
        "experiment_id",
        "dataset",
        "record_id",
    ],
    how="left",
    validate="one_to_one",
)


# ----------------------------------------------------------------
# Merge deterministic rule features
# ----------------------------------------------------------------
hybrid_base_df = hybrid_base_df.merge(
    standardized_rule_df,
    on=[
        "dataset",
        "record_id",
    ],
    how="left",
    validate="many_to_one",
)


# ----------------------------------------------------------------
# Handle records that do not have deterministic rule results
# ----------------------------------------------------------------
# Some injected or sampled records may not have a corresponding
# baseline rule record. Missing rule evidence is treated as neutral,
# not as a rule violation.
hybrid_base_df[
    "rule_evidence_available"
] = (
    hybrid_base_df[
        "rule_trust_score"
    ]
    .notna()
    .astype(int)
)

hybrid_base_df[
    "failed_rule_count_std"
] = (
    hybrid_base_df[
        "failed_rule_count_std"
    ]
    .fillna(0)
    .astype(int)
)

hybrid_base_df[
    "rule_trust_score"
] = (
    hybrid_base_df[
        "rule_trust_score"
    ]
    .fillna(1.0)
    .clip(0.0, 1.0)
)

hybrid_base_df[
    "rule_severity_score"
] = (
    hybrid_base_df[
        "rule_severity_score"
    ]
    .fillna(0.0)
    .clip(0.0, 1.0)
)

hybrid_base_df[
    "rule_anomaly_prediction"
] = (
    hybrid_base_df[
        "rule_anomaly_prediction"
    ]
    .fillna(0)
    .astype(int)
)

hybrid_base_df[
    "rule_risk_score"
] = (
    hybrid_base_df[
        "rule_risk_score"
    ]
    .fillna(0.0)
    .clip(0.0, 1.0)
)


# ----------------------------------------------------------------
# Handle missing detector evidence
# ----------------------------------------------------------------
for column in [
    "iforest_normalized_score",
    "lof_normalized_score",
]:
    if column not in hybrid_base_df.columns:
        hybrid_base_df[column] = np.nan

for column in [
    "iforest_prediction",
    "lof_prediction",
]:
    if column not in hybrid_base_df.columns:
        hybrid_base_df[column] = 0

hybrid_base_df[
    "iforest_prediction"
] = (
    hybrid_base_df[
        "iforest_prediction"
    ]
    .fillna(0)
    .astype(int)
)

hybrid_base_df[
    "lof_prediction"
] = (
    hybrid_base_df[
        "lof_prediction"
    ]
    .fillna(0)
    .astype(int)
)


# ----------------------------------------------------------------
# Detector availability
# ----------------------------------------------------------------
hybrid_base_df[
    "iforest_available"
] = (
    hybrid_base_df[
        "iforest_normalized_score"
    ]
    .notna()
    .astype(int)
)

hybrid_base_df[
    "lof_available"
] = (
    hybrid_base_df[
        "lof_normalized_score"
    ]
    .notna()
    .astype(int)
)

hybrid_base_df[
    "detector_count_available"
] = (
    hybrid_base_df[
        "iforest_available"
    ]
    + hybrid_base_df[
        "lof_available"
    ]
)


# ----------------------------------------------------------------
# AI consensus score
# ----------------------------------------------------------------
hybrid_base_df[
    "ai_consensus_score"
] = (
    hybrid_base_df[
        [
            "iforest_normalized_score",
            "lof_normalized_score",
        ]
    ]
    .mean(
        axis=1,
        skipna=True,
    )
    .fillna(0.0)
    .clip(0.0, 1.0)
)


# ----------------------------------------------------------------
# AI detector agreement
# ----------------------------------------------------------------
hybrid_base_df[
    "ai_detector_agreement"
] = np.where(
    hybrid_base_df[
        "detector_count_available"
    ] == 2,
    (
        hybrid_base_df[
            "iforest_prediction"
        ]
        == hybrid_base_df[
            "lof_prediction"
        ]
    ).astype(float),
    0.5,
)


# ----------------------------------------------------------------
# AI disagreement magnitude
# ----------------------------------------------------------------
hybrid_base_df[
    "ai_score_disagreement"
] = (
    hybrid_base_df[
        "iforest_normalized_score"
    ]
    - hybrid_base_df[
        "lof_normalized_score"
    ]
).abs()

hybrid_base_df[
    "ai_score_disagreement"
] = (
    hybrid_base_df[
        "ai_score_disagreement"
    ]
    .fillna(0.5)
    .clip(0.0, 1.0)
)


# ----------------------------------------------------------------
# Record-level evidence completeness
# ----------------------------------------------------------------
hybrid_base_df[
    "evidence_completeness"
] = (
    hybrid_base_df[
        "rule_evidence_available"
    ]
    + hybrid_base_df[
        "iforest_available"
    ]
    + hybrid_base_df[
        "lof_available"
    ]
) / 3.0


# ----------------------------------------------------------------
# Ground-truth normalization
# ----------------------------------------------------------------
hybrid_base_df[
    "ground_truth_label"
] = (
    pd.to_numeric(
        hybrid_base_df[
            "ground_truth_label"
        ],
        errors="coerce",
    )
    .fillna(0)
    .clip(0, 1)
    .astype(int)
)


# ----------------------------------------------------------------
# Sort deterministically
# ----------------------------------------------------------------
hybrid_base_df = (
    hybrid_base_df
    .sort_values(
        [
            "dataset",
            "experiment_id",
            "record_id",
        ]
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------------
# Save standardized hybrid input
# ----------------------------------------------------------------
HYBRID_BASE_PATH = (
    HYBRID_DATA_DIR
    / "hybrid_record_level_input.parquet"
)

hybrid_base_df.to_parquet(
    HYBRID_BASE_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Dataset-level coverage summary
# ----------------------------------------------------------------
hybrid_coverage_df = (
    hybrid_base_df
    .groupby(
        "dataset",
        dropna=False,
    )
    .agg(
        experiments=(
            "experiment_id",
            "nunique",
        ),
        record_rows=(
            "record_id",
            "size",
        ),
        unique_records=(
            "record_id",
            "nunique",
        ),
        ground_truth_anomalies=(
            "ground_truth_label",
            "sum",
        ),
        rule_evidence_rate=(
            "rule_evidence_available",
            "mean",
        ),
        iforest_evidence_rate=(
            "iforest_available",
            "mean",
        ),
        lof_evidence_rate=(
            "lof_available",
            "mean",
        ),
        mean_rule_risk=(
            "rule_risk_score",
            "mean",
        ),
        mean_ai_consensus=(
            "ai_consensus_score",
            "mean",
        ),
        mean_evidence_completeness=(
            "evidence_completeness",
            "mean",
        ),
    )
    .reset_index()
)

display(hybrid_coverage_df)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
validation_checks = {
    "hybrid_input_not_empty": (
        len(hybrid_base_df) > 0
    ),
    "one_row_per_experiment_record": (
        not hybrid_base_df.duplicated(
            subset=[
                "experiment_id",
                "dataset",
                "record_id",
            ]
        ).any()
    ),
    "three_datasets_present": (
        hybrid_base_df[
            "dataset"
        ].nunique() == 3
    ),
    "all_63_experiments_present": (
        hybrid_base_df[
            "experiment_id"
        ].nunique() == 63
    ),
    "rule_risk_in_valid_range": (
        hybrid_base_df[
            "rule_risk_score"
        ].between(
            0.0,
            1.0,
        ).all()
    ),
    "ai_consensus_in_valid_range": (
        hybrid_base_df[
            "ai_consensus_score"
        ].between(
            0.0,
            1.0,
        ).all()
    ),
    "ground_truth_binary": (
        set(
            hybrid_base_df[
                "ground_truth_label"
            ].unique()
        ).issubset({0, 1})
    ),
}

hybrid_validation_df = pd.DataFrame({
    "check": validation_checks.keys(),
    "passed": validation_checks.values(),
})

display(hybrid_validation_df)

assert hybrid_validation_df[
    "passed"
].all(), (
    "One or more hybrid-dataset checks failed."
)


# ----------------------------------------------------------------
# Save construction audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

construction_audit = {
    "hybrid_run_id": HYBRID_RUN_ID,
    "created_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "hybrid_rows": int(
        len(hybrid_base_df)
    ),
    "experiments": int(
        hybrid_base_df[
            "experiment_id"
        ].nunique()
    ),
    "datasets": int(
        hybrid_base_df[
            "dataset"
        ].nunique()
    ),
    "rule_records_available": int(
        hybrid_base_df[
            "rule_evidence_available"
        ].sum()
    ),
    "iforest_records_available": int(
        hybrid_base_df[
            "iforest_available"
        ].sum()
    ),
    "lof_records_available": int(
        hybrid_base_df[
            "lof_available"
        ].sum()
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "output_path": str(
        HYBRID_BASE_PATH
    ),
}

HYBRID_CONSTRUCTION_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "hybrid_dataset_construction_audit.json"
)

with open(
    HYBRID_CONSTRUCTION_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        construction_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Clean temporary objects
# ----------------------------------------------------------------
del ai_long_df
del score_wide_df
del prediction_wide_df
del record_metadata_df
del rule_records_df

gc.collect()


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Hybrid record-level input created successfully."
)
print(
    f"Hybrid rows: "
    f"{len(hybrid_base_df):,}"
)
print(
    f"Experiments: "
    f"{hybrid_base_df['experiment_id'].nunique()}"
)
print(
    f"Datasets: "
    f"{hybrid_base_df['dataset'].nunique()}"
)
print(
    f"Output artifact: "
    f"{HYBRID_BASE_PATH}"
)
print(
    f"Construction audit: "
    f"{HYBRID_CONSTRUCTION_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

display(
    hybrid_base_df.head(10)
)

## 4. Hybrid Risk Scoring and Governed Decisions

This section applies the explicit policy-based fusion layer and maps combined evidence into auditable operational decisions.


In [ ]:
# ================================================================
# Cell 4 — Hybrid risk scoring and governed decision policy
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# Hybrid policy configuration
# ----------------------------------------------------------------
POLICY_VERSION = "HYBRID_POLICY_V1.0"

HYBRID_POLICY = {
    "policy_version": POLICY_VERSION,

    # Hybrid risk weights
    "weights": {
        "rule_risk": 0.40,
        "ai_consensus": 0.35,
        "ai_prediction_vote": 0.15,
        "evidence_uncertainty": 0.10,
    },

    # Decision thresholds
    "thresholds": {
        "accept_max_risk": 0.30,
        "repair_max_risk": 0.55,
        "quarantine_max_risk": 0.78,
        "critical_rule_severity": 0.75,
        "high_rule_risk": 0.70,
        "high_ai_consensus": 0.80,
        "high_disagreement": 0.50,
        "minimum_evidence_completeness": 2 / 3,
    },

    # Trust-score bands
    "trust_bands": {
        "HIGH_TRUST": 0.75,
        "MODERATE_TRUST": 0.50,
        "LOW_TRUST": 0.25,
    },

    # Decision meanings
    "decision_definitions": {
        "ACCEPT": (
            "Record has sufficiently low combined risk and no "
            "critical deterministic-rule evidence."
        ),
        "REPAIR": (
            "Record has deterministic quality issues that appear "
            "repairable and does not require immediate isolation."
        ),
        "QUARANTINE": (
            "Record presents strong rule-based or AI-based anomaly "
            "evidence and should be isolated from trusted publication."
        ),
        "ESCALATE": (
            "Available evidence is conflicting, incomplete, or "
            "operationally ambiguous and requires human review."
        ),
    },
}


# ----------------------------------------------------------------
# Save policy configuration
# ----------------------------------------------------------------
HYBRID_POLICY_PATH = (
    HYBRID_CONFIG_DIR
    / "hybrid_policy_v1.json"
)

with open(
    HYBRID_POLICY_PATH,
    "w",
    encoding="utf-8",
) as policy_file:
    json.dump(
        HYBRID_POLICY,
        policy_file,
        indent=2,
    )

print(
    f"Hybrid policy saved to: "
    f"{HYBRID_POLICY_PATH}"
)


# ----------------------------------------------------------------
# Create working decision dataset
# ----------------------------------------------------------------
hybrid_decision_df = hybrid_base_df.copy()


# ----------------------------------------------------------------
# Ensure required fields exist and are numeric
# ----------------------------------------------------------------
required_numeric_columns = [
    "rule_risk_score",
    "rule_trust_score",
    "rule_severity_score",
    "rule_anomaly_prediction",
    "iforest_normalized_score",
    "lof_normalized_score",
    "iforest_prediction",
    "lof_prediction",
    "ai_consensus_score",
    "ai_detector_agreement",
    "ai_score_disagreement",
    "evidence_completeness",
    "ground_truth_label",
]

missing_columns = [
    column
    for column in required_numeric_columns
    if column not in hybrid_decision_df.columns
]

if missing_columns:
    raise KeyError(
        "Missing required hybrid fields: "
        + ", ".join(missing_columns)
    )

for column in required_numeric_columns:
    hybrid_decision_df[column] = pd.to_numeric(
        hybrid_decision_df[column],
        errors="coerce",
    )


# ----------------------------------------------------------------
# Fill neutral defaults
# ----------------------------------------------------------------
neutral_defaults = {
    "rule_risk_score": 0.0,
    "rule_trust_score": 1.0,
    "rule_severity_score": 0.0,
    "rule_anomaly_prediction": 0,
    "iforest_normalized_score": 0.0,
    "lof_normalized_score": 0.0,
    "iforest_prediction": 0,
    "lof_prediction": 0,
    "ai_consensus_score": 0.0,
    "ai_detector_agreement": 0.5,
    "ai_score_disagreement": 0.5,
    "evidence_completeness": 0.0,
    "ground_truth_label": 0,
}

for column, default_value in neutral_defaults.items():
    hybrid_decision_df[column] = (
        hybrid_decision_df[column]
        .fillna(default_value)
    )


# ----------------------------------------------------------------
# Create AI detector-vote component
# ----------------------------------------------------------------
hybrid_decision_df[
    "ai_prediction_vote"
] = (
    hybrid_decision_df[
        "iforest_prediction"
    ]
    + hybrid_decision_df[
        "lof_prediction"
    ]
) / 2.0

hybrid_decision_df[
    "ai_prediction_vote"
] = (
    hybrid_decision_df[
        "ai_prediction_vote"
    ]
    .clip(0.0, 1.0)
)


# ----------------------------------------------------------------
# Create uncertainty component
# ----------------------------------------------------------------
# Uncertainty increases when:
# 1. Evidence is incomplete.
# 2. AI score disagreement is high.
# 3. Detector predictions disagree.
evidence_missingness = (
    1.0
    - hybrid_decision_df[
        "evidence_completeness"
    ]
)

prediction_disagreement = (
    1.0
    - hybrid_decision_df[
        "ai_detector_agreement"
    ]
)

hybrid_decision_df[
    "evidence_uncertainty_score"
] = (
    0.40 * evidence_missingness
    + 0.35 * hybrid_decision_df[
        "ai_score_disagreement"
    ]
    + 0.25 * prediction_disagreement
).clip(0.0, 1.0)


# ----------------------------------------------------------------
# Calculate weighted hybrid risk
# ----------------------------------------------------------------
weights = HYBRID_POLICY["weights"]

hybrid_decision_df[
    "hybrid_risk_score"
] = (
    weights["rule_risk"]
    * hybrid_decision_df[
        "rule_risk_score"
    ]
    + weights["ai_consensus"]
    * hybrid_decision_df[
        "ai_consensus_score"
    ]
    + weights["ai_prediction_vote"]
    * hybrid_decision_df[
        "ai_prediction_vote"
    ]
    + weights["evidence_uncertainty"]
    * hybrid_decision_df[
        "evidence_uncertainty_score"
    ]
).clip(0.0, 1.0)


# ----------------------------------------------------------------
# Calculate trust score
# ----------------------------------------------------------------
# Trust is expressed as the complement of hybrid risk, adjusted by
# evidence completeness. A high-risk or weakly evidenced record has
# lower operational trust.
hybrid_decision_df[
    "hybrid_trust_score"
] = (
    (
        1.0
        - hybrid_decision_df[
            "hybrid_risk_score"
        ]
    )
    * (
        0.80
        + 0.20
        * hybrid_decision_df[
            "evidence_completeness"
        ]
    )
).clip(0.0, 1.0)


# ----------------------------------------------------------------
# Assign trust bands
# ----------------------------------------------------------------
trust_thresholds = (
    HYBRID_POLICY[
        "trust_bands"
    ]
)

hybrid_decision_df[
    "trust_band"
] = np.select(
    [
        hybrid_decision_df[
            "hybrid_trust_score"
        ] >= trust_thresholds[
            "HIGH_TRUST"
        ],

        hybrid_decision_df[
            "hybrid_trust_score"
        ] >= trust_thresholds[
            "MODERATE_TRUST"
        ],

        hybrid_decision_df[
            "hybrid_trust_score"
        ] >= trust_thresholds[
            "LOW_TRUST"
        ],
    ],
    [
        "HIGH_TRUST",
        "MODERATE_TRUST",
        "LOW_TRUST",
    ],
    default="VERY_LOW_TRUST",
)


# ----------------------------------------------------------------
# Decision-condition flags
# ----------------------------------------------------------------
thresholds = HYBRID_POLICY["thresholds"]

hybrid_decision_df[
    "critical_rule_flag"
] = (
    hybrid_decision_df[
        "rule_severity_score"
    ]
    >= thresholds[
        "critical_rule_severity"
    ]
).astype(int)

hybrid_decision_df[
    "high_rule_risk_flag"
] = (
    hybrid_decision_df[
        "rule_risk_score"
    ]
    >= thresholds[
        "high_rule_risk"
    ]
).astype(int)

hybrid_decision_df[
    "high_ai_risk_flag"
] = (
    hybrid_decision_df[
        "ai_consensus_score"
    ]
    >= thresholds[
        "high_ai_consensus"
    ]
).astype(int)

hybrid_decision_df[
    "high_disagreement_flag"
] = (
    hybrid_decision_df[
        "ai_score_disagreement"
    ]
    >= thresholds[
        "high_disagreement"
    ]
).astype(int)

hybrid_decision_df[
    "insufficient_evidence_flag"
] = (
    hybrid_decision_df[
        "evidence_completeness"
    ]
    < thresholds[
        "minimum_evidence_completeness"
    ]
).astype(int)

hybrid_decision_df[
    "rule_ai_conflict_flag"
] = (
    (
        hybrid_decision_df[
            "rule_anomaly_prediction"
        ]
        != (
            hybrid_decision_df[
                "ai_prediction_vote"
            ] >= 0.5
        ).astype(int)
    )
).astype(int)


# ----------------------------------------------------------------
# Governed decision function
# ----------------------------------------------------------------
def assign_hybrid_decision(
    row: pd.Series,
) -> str:
    """
    Assign one governed action to a record.

    Decision precedence:
    1. ESCALATE for insufficient or conflicting evidence.
    2. QUARANTINE for critical/high-confidence risk.
    3. REPAIR for moderate deterministic-rule risk.
    4. ACCEPT for low combined risk.
    """

    risk = float(
        row["hybrid_risk_score"]
    )

    rule_risk = float(
        row["rule_risk_score"]
    )

    ai_risk = float(
        row["ai_consensus_score"]
    )

    rule_failed = int(
        row["rule_anomaly_prediction"]
    ) == 1

    critical_rule = int(
        row["critical_rule_flag"]
    ) == 1

    high_rule_risk = int(
        row["high_rule_risk_flag"]
    ) == 1

    high_ai_risk = int(
        row["high_ai_risk_flag"]
    ) == 1

    insufficient_evidence = int(
        row["insufficient_evidence_flag"]
    ) == 1

    high_disagreement = int(
        row["high_disagreement_flag"]
    ) == 1

    rule_ai_conflict = int(
        row["rule_ai_conflict_flag"]
    ) == 1

    # Escalate when evidence is insufficient or materially conflicting.
    if insufficient_evidence:
        return "ESCALATE"

    if (
        high_disagreement
        and rule_ai_conflict
        and risk
        >= thresholds[
            "repair_max_risk"
        ]
    ):
        return "ESCALATE"

    # Quarantine records with critical deterministic evidence.
    if critical_rule:
        return "QUARANTINE"

    # Quarantine high-risk records supported by strong rule or AI evidence.
    if (
        risk
        > thresholds[
            "quarantine_max_risk"
        ]
    ):
        return "QUARANTINE"

    if (
        high_rule_risk
        and high_ai_risk
    ):
        return "QUARANTINE"

    # Repair records with rule failures but without critical risk.
    if (
        rule_failed
        and risk
        <= thresholds[
            "quarantine_max_risk"
        ]
    ):
        return "REPAIR"

    # Moderate-risk AI anomalies without rule failures require escalation.
    if (
        not rule_failed
        and ai_risk
        >= thresholds[
            "high_ai_consensus"
        ]
    ):
        return "ESCALATE"

    # Low-risk records are accepted.
    if (
        risk
        <= thresholds[
            "accept_max_risk"
        ]
    ):
        return "ACCEPT"

    # Intermediate risk with no rule failure is escalated.
    if (
        risk
        <= thresholds[
            "repair_max_risk"
        ]
    ):
        return "ESCALATE"

    # Remaining elevated-risk records are quarantined.
    return "QUARANTINE"


# ----------------------------------------------------------------
# Execute policy
# ----------------------------------------------------------------
hybrid_decision_df[
    "hybrid_decision"
] = hybrid_decision_df.apply(
    assign_hybrid_decision,
    axis=1,
)


# ----------------------------------------------------------------
# Binary hybrid anomaly prediction
# ----------------------------------------------------------------
# For comparison against ground truth:
# ACCEPT = non-anomalous
# REPAIR, QUARANTINE, ESCALATE = potentially anomalous
hybrid_decision_df[
    "hybrid_anomaly_prediction"
] = (
    hybrid_decision_df[
        "hybrid_decision"
    ]
    .isin(
        [
            "REPAIR",
            "QUARANTINE",
            "ESCALATE",
        ]
    )
    .astype(int)
)


# ----------------------------------------------------------------
# Create primary decision reason
# ----------------------------------------------------------------
def assign_primary_reason(
    row: pd.Series,
) -> str:
    decision = row[
        "hybrid_decision"
    ]

    if (
        int(
            row[
                "insufficient_evidence_flag"
            ]
        )
        == 1
    ):
        return (
            "Insufficient evidence completeness"
        )

    if (
        int(
            row[
                "critical_rule_flag"
            ]
        )
        == 1
    ):
        return (
            "Critical deterministic-rule severity"
        )

    if (
        int(
            row[
                "high_rule_risk_flag"
            ]
        )
        == 1
        and int(
            row[
                "high_ai_risk_flag"
            ]
        )
        == 1
    ):
        return (
            "High rule risk and high AI anomaly evidence"
        )

    if (
        int(
            row[
                "high_disagreement_flag"
            ]
        )
        == 1
        and int(
            row[
                "rule_ai_conflict_flag"
            ]
        )
        == 1
    ):
        return (
            "Conflicting rule and AI evidence"
        )

    if (
        decision == "REPAIR"
        and int(
            row[
                "rule_anomaly_prediction"
            ]
        )
        == 1
    ):
        return (
            "Repairable deterministic-rule violation"
        )

    if (
        decision == "ESCALATE"
        and float(
            row[
                "ai_consensus_score"
            ]
        )
        >= thresholds[
            "high_ai_consensus"
        ]
    ):
        return (
            "High AI anomaly evidence without matching rule failure"
        )

    if decision == "ACCEPT":
        return (
            "Low combined rule and AI risk"
        )

    if decision == "QUARANTINE":
        return (
            "Elevated combined hybrid risk"
        )

    return (
        "Policy-defined hybrid decision"
    )


hybrid_decision_df[
    "primary_decision_reason"
] = hybrid_decision_df.apply(
    assign_primary_reason,
    axis=1,
)


# ----------------------------------------------------------------
# Add policy and execution metadata
# ----------------------------------------------------------------
decision_timestamp_utc = (
    datetime.now(
        timezone.utc
    ).isoformat()
)

hybrid_decision_df[
    "hybrid_run_id"
] = HYBRID_RUN_ID

hybrid_decision_df[
    "policy_version"
] = POLICY_VERSION

hybrid_decision_df[
    "decision_timestamp_utc"
] = decision_timestamp_utc


# ----------------------------------------------------------------
# Select ordered publication fields
# ----------------------------------------------------------------
preferred_column_order = [
    "hybrid_run_id",
    "policy_version",
    "decision_timestamp_utc",
    "experiment_id",
    "dataset",
    "record_id",
    "anomaly_type",
    "anomaly_rate",
    "ground_truth_label",

    "failed_rule_count_std",
    "rule_trust_score",
    "rule_severity_score",
    "rule_anomaly_prediction",
    "rule_risk_score",

    "iforest_normalized_score",
    "iforest_prediction",
    "lof_normalized_score",
    "lof_prediction",

    "ai_consensus_score",
    "ai_prediction_vote",
    "ai_detector_agreement",
    "ai_score_disagreement",

    "evidence_completeness",
    "evidence_uncertainty_score",

    "hybrid_risk_score",
    "hybrid_trust_score",
    "trust_band",

    "critical_rule_flag",
    "high_rule_risk_flag",
    "high_ai_risk_flag",
    "high_disagreement_flag",
    "insufficient_evidence_flag",
    "rule_ai_conflict_flag",

    "hybrid_decision",
    "hybrid_anomaly_prediction",
    "primary_decision_reason",
]

ordered_columns = [
    column
    for column in preferred_column_order
    if column in hybrid_decision_df.columns
]

remaining_columns = [
    column
    for column in hybrid_decision_df.columns
    if column not in ordered_columns
]

hybrid_decision_df = (
    hybrid_decision_df[
        ordered_columns
        + remaining_columns
    ]
)


# ----------------------------------------------------------------
# Save record-level hybrid decisions
# ----------------------------------------------------------------
HYBRID_DECISIONS_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_record_decisions.parquet"
)

hybrid_decision_df.to_parquet(
    HYBRID_DECISIONS_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Decision-distribution summary
# ----------------------------------------------------------------
decision_summary_df = (
    hybrid_decision_df
    .groupby(
        [
            "dataset",
            "hybrid_decision",
        ],
        dropna=False,
    )
    .agg(
        records=(
            "record_id",
            "size",
        ),
        unique_records=(
            "record_id",
            "nunique",
        ),
        true_anomalies=(
            "ground_truth_label",
            "sum",
        ),
        mean_hybrid_risk=(
            "hybrid_risk_score",
            "mean",
        ),
        mean_hybrid_trust=(
            "hybrid_trust_score",
            "mean",
        ),
        mean_rule_risk=(
            "rule_risk_score",
            "mean",
        ),
        mean_ai_consensus=(
            "ai_consensus_score",
            "mean",
        ),
    )
    .reset_index()
)

decision_summary_df[
    "decision_rate"
] = (
    decision_summary_df[
        "records"
    ]
    / decision_summary_df
    .groupby(
        "dataset"
    )[
        "records"
    ]
    .transform("sum")
)

HYBRID_DECISION_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_decision_summary.csv"
)

decision_summary_df.to_csv(
    HYBRID_DECISION_SUMMARY_PATH,
    index=False,
)

display(decision_summary_df)


# ----------------------------------------------------------------
# Trust-band summary
# ----------------------------------------------------------------
trust_summary_df = (
    hybrid_decision_df
    .groupby(
        [
            "dataset",
            "trust_band",
        ],
        dropna=False,
    )
    .agg(
        records=(
            "record_id",
            "size",
        ),
        true_anomalies=(
            "ground_truth_label",
            "sum",
        ),
        predicted_anomalies=(
            "hybrid_anomaly_prediction",
            "sum",
        ),
        mean_hybrid_risk=(
            "hybrid_risk_score",
            "mean",
        ),
        mean_hybrid_trust=(
            "hybrid_trust_score",
            "mean",
        ),
    )
    .reset_index()
)

HYBRID_TRUST_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_trust_band_summary.csv"
)

trust_summary_df.to_csv(
    HYBRID_TRUST_SUMMARY_PATH,
    index=False,
)

display(trust_summary_df)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
valid_decisions = {
    "ACCEPT",
    "REPAIR",
    "QUARANTINE",
    "ESCALATE",
}

validation_checks = {
    "decision_dataset_not_empty": (
        len(hybrid_decision_df) > 0
    ),

    "all_records_have_decision": (
        hybrid_decision_df[
            "hybrid_decision"
        ]
        .notna()
        .all()
    ),

    "only_valid_decisions": (
        set(
            hybrid_decision_df[
                "hybrid_decision"
            ]
            .unique()
        )
        .issubset(
            valid_decisions
        )
    ),

    "risk_scores_valid": (
        hybrid_decision_df[
            "hybrid_risk_score"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),

    "trust_scores_valid": (
        hybrid_decision_df[
            "hybrid_trust_score"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),

    "binary_prediction_valid": (
        set(
            hybrid_decision_df[
                "hybrid_anomaly_prediction"
            ]
            .unique()
        )
        .issubset(
            {0, 1}
        )
    ),

    "all_63_experiments_present": (
        hybrid_decision_df[
            "experiment_id"
        ]
        .nunique()
        == 63
    ),

    "three_datasets_present": (
        hybrid_decision_df[
            "dataset"
        ]
        .nunique()
        == 3
    ),
}

policy_validation_df = pd.DataFrame({
    "check": validation_checks.keys(),
    "passed": validation_checks.values(),
})

display(policy_validation_df)

assert policy_validation_df[
    "passed"
].all(), (
    "One or more hybrid-policy checks failed."
)


# ----------------------------------------------------------------
# Save policy-execution audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

decision_counts = (
    hybrid_decision_df[
        "hybrid_decision"
    ]
    .value_counts()
    .to_dict()
)

policy_execution_audit = {
    "hybrid_run_id": HYBRID_RUN_ID,
    "policy_version": POLICY_VERSION,
    "execution_timestamp_utc": (
        decision_timestamp_utc
    ),
    "records_processed": int(
        len(hybrid_decision_df)
    ),
    "experiments_processed": int(
        hybrid_decision_df[
            "experiment_id"
        ]
        .nunique()
    ),
    "datasets_processed": int(
        hybrid_decision_df[
            "dataset"
        ]
        .nunique()
    ),
    "decision_counts": {
        str(key): int(value)
        for key, value
        in decision_counts.items()
    },
    "mean_hybrid_risk": float(
        hybrid_decision_df[
            "hybrid_risk_score"
        ]
        .mean()
    ),
    "mean_hybrid_trust": float(
        hybrid_decision_df[
            "hybrid_trust_score"
        ]
        .mean()
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "decision_output_path": str(
        HYBRID_DECISIONS_PATH
    ),
}

POLICY_EXECUTION_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "hybrid_policy_execution_audit.json"
)

with open(
    POLICY_EXECUTION_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        policy_execution_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Hybrid decision policy executed successfully."
)
print(
    f"Policy version: "
    f"{POLICY_VERSION}"
)
print(
    f"Records processed: "
    f"{len(hybrid_decision_df):,}"
)
print(
    f"Experiments processed: "
    f"{hybrid_decision_df['experiment_id'].nunique()}"
)
print(
    f"Decisions artifact: "
    f"{HYBRID_DECISIONS_PATH}"
)
print(
    f"Decision summary: "
    f"{HYBRID_DECISION_SUMMARY_PATH}"
)
print(
    f"Trust summary: "
    f"{HYBRID_TRUST_SUMMARY_PATH}"
)
print(
    f"Execution audit: "
    f"{POLICY_EXECUTION_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

display(
    hybrid_decision_df[
        [
            "experiment_id",
            "dataset",
            "record_id",
            "ground_truth_label",
            "rule_risk_score",
            "ai_consensus_score",
            "hybrid_risk_score",
            "hybrid_trust_score",
            "trust_band",
            "hybrid_decision",
            "primary_decision_reason",
        ]
    ]
    .head(20)
)

## 5. Comparative Evaluation

This section compares rule-only, individual AI detectors, AI consensus, and the initial hybrid policy using experiment-level performance metrics.


In [ ]:
# ================================================================
# Cell 5 — Comparative evaluation of rule-only, AI-only,
#          consensus-AI, and hybrid methods
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# Evaluation helpers
# ----------------------------------------------------------------
def safe_divide(
    numerator: float,
    denominator: float,
) -> float:
    """
    Divide safely and return 0 when the denominator is zero.
    """
    if denominator == 0:
        return 0.0

    return float(
        numerator / denominator
    )


def calculate_binary_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
    y_score: pd.Series,
) -> dict:
    """
    Calculate binary-classification metrics from labels, predictions,
    and continuous anomaly scores.
    """
    y_true_array = (
        pd.to_numeric(
            y_true,
            errors="coerce",
        )
        .fillna(0)
        .clip(0, 1)
        .astype(int)
        .to_numpy()
    )

    y_pred_array = (
        pd.to_numeric(
            y_pred,
            errors="coerce",
        )
        .fillna(0)
        .clip(0, 1)
        .astype(int)
        .to_numpy()
    )

    y_score_array = (
        pd.to_numeric(
            y_score,
            errors="coerce",
        )
        .fillna(0.0)
        .astype(float)
        .to_numpy()
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true_array,
        y_pred_array,
        labels=[0, 1],
    ).ravel()

    precision = precision_score(
        y_true_array,
        y_pred_array,
        zero_division=0,
    )

    recall = recall_score(
        y_true_array,
        y_pred_array,
        zero_division=0,
    )

    f1 = f1_score(
        y_true_array,
        y_pred_array,
        zero_division=0,
    )

    false_positive_rate = safe_divide(
        fp,
        fp + tn,
    )

    false_negative_rate = safe_divide(
        fn,
        fn + tp,
    )

    specificity = safe_divide(
        tn,
        tn + fp,
    )

    accuracy = safe_divide(
        tp + tn,
        tp + tn + fp + fn,
    )

    balanced_accuracy = (
        recall
        + specificity
    ) / 2.0

    if len(
        np.unique(
            y_true_array
        )
    ) > 1:
        pr_auc = average_precision_score(
            y_true_array,
            y_score_array,
        )
    else:
        pr_auc = np.nan

    return {
        "records": int(
            len(y_true_array)
        ),
        "true_anomalies": int(
            y_true_array.sum()
        ),
        "predicted_anomalies": int(
            y_pred_array.sum()
        ),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "false_negative_rate": float(
            false_negative_rate
        ),
        "specificity": float(
            specificity
        ),
        "accuracy": float(
            accuracy
        ),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "pr_auc": (
            float(pr_auc)
            if not pd.isna(pr_auc)
            else np.nan
        ),
    }


# ----------------------------------------------------------------
# Prepare evaluation dataset
# ----------------------------------------------------------------
evaluation_df = hybrid_decision_df.copy()

required_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ground_truth_label",
    "rule_anomaly_prediction",
    "rule_risk_score",
    "iforest_prediction",
    "iforest_normalized_score",
    "lof_prediction",
    "lof_normalized_score",
    "ai_prediction_vote",
    "ai_consensus_score",
    "hybrid_anomaly_prediction",
    "hybrid_risk_score",
]

missing_columns = [
    column
    for column in required_columns
    if column not in evaluation_df.columns
]

if missing_columns:
    raise KeyError(
        "Missing evaluation fields: "
        + ", ".join(
            missing_columns
        )
    )


# ----------------------------------------------------------------
# Standardize numeric fields
# ----------------------------------------------------------------
numeric_columns = [
    "ground_truth_label",
    "rule_anomaly_prediction",
    "rule_risk_score",
    "iforest_prediction",
    "iforest_normalized_score",
    "lof_prediction",
    "lof_normalized_score",
    "ai_prediction_vote",
    "ai_consensus_score",
    "hybrid_anomaly_prediction",
    "hybrid_risk_score",
]

for column in numeric_columns:
    evaluation_df[column] = pd.to_numeric(
        evaluation_df[column],
        errors="coerce",
    )


# ----------------------------------------------------------------
# Create AI-consensus binary prediction
# ----------------------------------------------------------------
# A majority vote of the two detectors is represented by >= 0.5.
evaluation_df[
    "ai_consensus_prediction"
] = (
    evaluation_df[
        "ai_prediction_vote"
    ]
    >= 0.5
).astype(int)


# ----------------------------------------------------------------
# Method definitions
# ----------------------------------------------------------------
METHODS = {
    "RULE_ONLY": {
        "prediction_column": (
            "rule_anomaly_prediction"
        ),
        "score_column": (
            "rule_risk_score"
        ),
    },

    "ISOLATION_FOREST": {
        "prediction_column": (
            "iforest_prediction"
        ),
        "score_column": (
            "iforest_normalized_score"
        ),
    },

    "LOCAL_OUTLIER_FACTOR": {
        "prediction_column": (
            "lof_prediction"
        ),
        "score_column": (
            "lof_normalized_score"
        ),
    },

    "AI_CONSENSUS": {
        "prediction_column": (
            "ai_consensus_prediction"
        ),
        "score_column": (
            "ai_consensus_score"
        ),
    },

    "HYBRID_V1": {
        "prediction_column": (
            "hybrid_anomaly_prediction"
        ),
        "score_column": (
            "hybrid_risk_score"
        ),
    },
}


# ----------------------------------------------------------------
# Evaluate each method by experiment
# ----------------------------------------------------------------
metric_rows = []

experiment_groups = (
    evaluation_df
    .groupby(
        [
            "experiment_id",
            "dataset",
        ],
        sort=True,
        dropna=False,
    )
)

total_experiments = (
    evaluation_df[
        "experiment_id"
    ]
    .nunique()
)

print(
    f"Evaluating {len(METHODS)} methods "
    f"across {total_experiments} experiments..."
)

for (
    experiment_id,
    dataset,
), experiment_df in experiment_groups:

    anomaly_type = (
        experiment_df[
            "anomaly_type"
        ].iloc[0]
        if "anomaly_type"
        in experiment_df.columns
        else None
    )

    anomaly_rate = (
        experiment_df[
            "anomaly_rate"
        ].iloc[0]
        if "anomaly_rate"
        in experiment_df.columns
        else None
    )

    for method_name, method_config in (
        METHODS.items()
    ):
        prediction_column = (
            method_config[
                "prediction_column"
            ]
        )

        score_column = (
            method_config[
                "score_column"
            ]
        )

        method_metrics = (
            calculate_binary_metrics(
                y_true=experiment_df[
                    "ground_truth_label"
                ],
                y_pred=experiment_df[
                    prediction_column
                ],
                y_score=experiment_df[
                    score_column
                ],
            )
        )

        method_metrics.update({
            "hybrid_run_id": (
                HYBRID_RUN_ID
            ),
            "policy_version": (
                POLICY_VERSION
            ),
            "experiment_id": (
                experiment_id
            ),
            "dataset": (
                dataset
            ),
            "anomaly_type": (
                anomaly_type
            ),
            "anomaly_rate": (
                anomaly_rate
            ),
            "method": (
                method_name
            ),
        })

        metric_rows.append(
            method_metrics
        )


experiment_metrics_df = pd.DataFrame(
    metric_rows
)


# ----------------------------------------------------------------
# Order columns
# ----------------------------------------------------------------
metric_column_order = [
    "hybrid_run_id",
    "policy_version",
    "experiment_id",
    "dataset",
    "anomaly_type",
    "anomaly_rate",
    "method",
    "records",
    "true_anomalies",
    "predicted_anomalies",
    "tp",
    "tn",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "false_negative_rate",
    "specificity",
    "accuracy",
    "balanced_accuracy",
    "pr_auc",
]

experiment_metrics_df = (
    experiment_metrics_df[
        metric_column_order
    ]
)


# ----------------------------------------------------------------
# Save experiment-level metrics
# ----------------------------------------------------------------
EXPERIMENT_METRICS_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_experiment_metrics.csv"
)

experiment_metrics_df.to_csv(
    EXPERIMENT_METRICS_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Dataset-level macro summaries
# ----------------------------------------------------------------
metric_summary_columns = [
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "false_negative_rate",
    "specificity",
    "accuracy",
    "balanced_accuracy",
    "pr_auc",
]

dataset_macro_summary_df = (
    experiment_metrics_df
    .groupby(
        [
            "dataset",
            "method",
        ],
        dropna=False,
    )[
        metric_summary_columns
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

dataset_macro_summary_df.columns = [
    (
        "_".join(
            [
                str(part)
                for part in column
                if str(part)
                not in [
                    "",
                    "None",
                ]
            ]
        )
        if isinstance(
            column,
            tuple,
        )
        else str(column)
    )
    for column in (
        dataset_macro_summary_df.columns
    )
]

DATASET_MACRO_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_dataset_macro_summary.csv"
)

dataset_macro_summary_df.to_csv(
    DATASET_MACRO_SUMMARY_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Overall macro summary
# ----------------------------------------------------------------
overall_macro_summary_df = (
    experiment_metrics_df
    .groupby(
        "method",
        dropna=False,
    )[
        metric_summary_columns
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

overall_macro_summary_df.columns = [
    (
        "_".join(
            [
                str(part)
                for part in column
                if str(part)
                not in [
                    "",
                    "None",
                ]
            ]
        )
        if isinstance(
            column,
            tuple,
        )
        else str(column)
    )
    for column in (
        overall_macro_summary_df.columns
    )
]

OVERALL_MACRO_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_overall_macro_summary.csv"
)

overall_macro_summary_df.to_csv(
    OVERALL_MACRO_SUMMARY_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Compact publication-oriented table
# ----------------------------------------------------------------
publication_summary_df = (
    experiment_metrics_df
    .groupby(
        "method",
        dropna=False,
    )
    .agg(
        experiments=(
            "experiment_id",
            "nunique",
        ),
        macro_precision=(
            "precision",
            "mean",
        ),
        macro_recall=(
            "recall",
            "mean",
        ),
        macro_f1=(
            "f1",
            "mean",
        ),
        macro_fpr=(
            "false_positive_rate",
            "mean",
        ),
        macro_fnr=(
            "false_negative_rate",
            "mean",
        ),
        macro_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        macro_pr_auc=(
            "pr_auc",
            "mean",
        ),
    )
    .reset_index()
)

publication_summary_df = (
    publication_summary_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

PUBLICATION_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_publication_summary.csv"
)

publication_summary_df.to_csv(
    PUBLICATION_SUMMARY_PATH,
    index=False,
)

display(publication_summary_df)


# ----------------------------------------------------------------
# Dataset-specific compact table
# ----------------------------------------------------------------
dataset_publication_summary_df = (
    experiment_metrics_df
    .groupby(
        [
            "dataset",
            "method",
        ],
        dropna=False,
    )
    .agg(
        experiments=(
            "experiment_id",
            "nunique",
        ),
        macro_precision=(
            "precision",
            "mean",
        ),
        macro_recall=(
            "recall",
            "mean",
        ),
        macro_f1=(
            "f1",
            "mean",
        ),
        macro_fpr=(
            "false_positive_rate",
            "mean",
        ),
        macro_fnr=(
            "false_negative_rate",
            "mean",
        ),
        macro_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        macro_pr_auc=(
            "pr_auc",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "macro_f1",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

DATASET_PUBLICATION_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_dataset_publication_summary.csv"
)

dataset_publication_summary_df.to_csv(
    DATASET_PUBLICATION_SUMMARY_PATH,
    index=False,
)

display(dataset_publication_summary_df)


# ----------------------------------------------------------------
# Hybrid improvement relative to baselines
# ----------------------------------------------------------------
comparison_metrics = [
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "macro_fpr",
    "macro_fnr",
    "macro_balanced_accuracy",
    "macro_pr_auc",
]

hybrid_row_df = (
    publication_summary_df.loc[
        publication_summary_df[
            "method"
        ].eq(
            "HYBRID_V1"
        )
    ]
)

if hybrid_row_df.empty:
    raise ValueError(
        "HYBRID_V1 summary row was not created."
    )

hybrid_row = (
    hybrid_row_df.iloc[0]
)

improvement_rows = []

for _, baseline_row in (
    publication_summary_df.loc[
        ~publication_summary_df[
            "method"
        ].eq(
            "HYBRID_V1"
        )
    ]
    .iterrows()
):
    result_row = {
        "hybrid_method": (
            "HYBRID_V1"
        ),
        "baseline_method": (
            baseline_row[
                "method"
            ]
        ),
    }

    for metric in comparison_metrics:
        result_row[
            f"{metric}_difference"
        ] = (
            hybrid_row[
                metric
            ]
            - baseline_row[
                metric
            ]
        )

    improvement_rows.append(
        result_row
    )

hybrid_improvement_df = pd.DataFrame(
    improvement_rows
)

HYBRID_IMPROVEMENT_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v1_improvement_vs_baselines.csv"
)

hybrid_improvement_df.to_csv(
    HYBRID_IMPROVEMENT_PATH,
    index=False,
)

display(hybrid_improvement_df)


# ----------------------------------------------------------------
# Detect current best method by macro-F1
# ----------------------------------------------------------------
best_method_row = (
    publication_summary_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0]
)

best_method = (
    best_method_row[
        "method"
    ]
)

best_macro_f1 = float(
    best_method_row[
        "macro_f1"
    ]
)


# ----------------------------------------------------------------
# Policy diagnostic flags
# ----------------------------------------------------------------
hybrid_summary_row = (
    publication_summary_df.loc[
        publication_summary_df[
            "method"
        ].eq(
            "HYBRID_V1"
        )
    ]
    .iloc[0]
)

policy_diagnostics = {
    "hybrid_outperforms_rule_only_f1": (
        hybrid_summary_row[
            "macro_f1"
        ]
        >
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "RULE_ONLY"
            ),
            "macro_f1",
        ].iloc[0]
    ),

    "hybrid_outperforms_iforest_f1": (
        hybrid_summary_row[
            "macro_f1"
        ]
        >
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "ISOLATION_FOREST"
            ),
            "macro_f1",
        ].iloc[0]
    ),

    "hybrid_outperforms_lof_f1": (
        hybrid_summary_row[
            "macro_f1"
        ]
        >
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "LOCAL_OUTLIER_FACTOR"
            ),
            "macro_f1",
        ].iloc[0]
    ),

    "hybrid_fnr_below_rule_only": (
        hybrid_summary_row[
            "macro_fnr"
        ]
        <
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "RULE_ONLY"
            ),
            "macro_fnr",
        ].iloc[0]
    ),

    "hybrid_fpr_below_ai_consensus": (
        hybrid_summary_row[
            "macro_fpr"
        ]
        <
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "AI_CONSENSUS"
            ),
            "macro_fpr",
        ].iloc[0]
    ),
}

policy_diagnostics_df = pd.DataFrame({
    "diagnostic": (
        policy_diagnostics.keys()
    ),
    "passed": (
        policy_diagnostics.values()
    ),
})

display(policy_diagnostics_df)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
validation_checks = {
    "all_63_experiments_evaluated": (
        experiment_metrics_df[
            "experiment_id"
        ].nunique()
        == 63
    ),

    "all_five_methods_evaluated": (
        experiment_metrics_df[
            "method"
        ].nunique()
        == 5
    ),

    "expected_metric_rows": (
        len(
            experiment_metrics_df
        )
        == 63 * 5
    ),

    "three_datasets_present": (
        experiment_metrics_df[
            "dataset"
        ].nunique()
        == 3
    ),

    "all_f1_values_valid": (
        experiment_metrics_df[
            "f1"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),

    "all_fpr_values_valid": (
        experiment_metrics_df[
            "false_positive_rate"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),

    "all_fnr_values_valid": (
        experiment_metrics_df[
            "false_negative_rate"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),
}

evaluation_validation_df = pd.DataFrame({
    "check": (
        validation_checks.keys()
    ),
    "passed": (
        validation_checks.values()
    ),
})

display(evaluation_validation_df)

assert evaluation_validation_df[
    "passed"
].all(), (
    "One or more comparative-evaluation checks failed."
)


# ----------------------------------------------------------------
# Save evaluation audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

evaluation_audit = {
    "hybrid_run_id": (
        HYBRID_RUN_ID
    ),
    "policy_version": (
        POLICY_VERSION
    ),
    "evaluation_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "experiment_count": int(
        experiment_metrics_df[
            "experiment_id"
        ].nunique()
    ),
    "dataset_count": int(
        experiment_metrics_df[
            "dataset"
        ].nunique()
    ),
    "method_count": int(
        experiment_metrics_df[
            "method"
        ].nunique()
    ),
    "metric_row_count": int(
        len(
            experiment_metrics_df
        )
    ),
    "best_method_by_macro_f1": str(
        best_method
    ),
    "best_macro_f1": float(
        best_macro_f1
    ),
    "hybrid_macro_f1": float(
        hybrid_summary_row[
            "macro_f1"
        ]
    ),
    "hybrid_macro_precision": float(
        hybrid_summary_row[
            "macro_precision"
        ]
    ),
    "hybrid_macro_recall": float(
        hybrid_summary_row[
            "macro_recall"
        ]
    ),
    "hybrid_macro_fpr": float(
        hybrid_summary_row[
            "macro_fpr"
        ]
    ),
    "hybrid_macro_fnr": float(
        hybrid_summary_row[
            "macro_fnr"
        ]
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "experiment_metrics_path": str(
        EXPERIMENT_METRICS_PATH
    ),
    "publication_summary_path": str(
        PUBLICATION_SUMMARY_PATH
    ),
}

EVALUATION_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "comparative_evaluation_audit.json"
)

with open(
    EVALUATION_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        evaluation_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Comparative evaluation completed successfully."
)
print(
    f"Experiments evaluated: "
    f"{experiment_metrics_df['experiment_id'].nunique()}"
)
print(
    f"Methods evaluated: "
    f"{experiment_metrics_df['method'].nunique()}"
)
print(
    f"Metric rows created: "
    f"{len(experiment_metrics_df):,}"
)
print(
    f"Best current method by macro-F1: "
    f"{best_method}"
)
print(
    f"Best current macro-F1: "
    f"{best_macro_f1:.4f}"
)
print(
    f"Experiment metrics: "
    f"{EXPERIMENT_METRICS_PATH}"
)
print(
    f"Publication summary: "
    f"{PUBLICATION_SUMMARY_PATH}"
)
print(
    f"Hybrid improvement table: "
    f"{HYBRID_IMPROVEMENT_PATH}"
)
print(
    f"Evaluation audit: "
    f"{EVALUATION_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

## 6. Constrained Calibration of Hybrid V2

This section calibrates candidate policy configurations using experiment-level splits while controlling operational false-positive burden.


In [ ]:
# ================================================================
# Cell 6 — Calibrate Hybrid V2 using experiment-level holdout
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# Calibration configuration
# ----------------------------------------------------------------
CALIBRATION_VERSION = "HYBRID_CALIBRATION_V2.0"
CALIBRATION_RANDOM_SEED = 42
CALIBRATION_EXPERIMENT_FRACTION = 0.67

# The optimization objective rewards macro-F1 while penalizing FPR.
FPR_PENALTY_WEIGHT = 0.20

# Candidate binary decision thresholds.
CANDIDATE_THRESHOLDS = np.round(
    np.arange(
        0.20,
        0.71,
        0.025,
    ),
    3,
).tolist()


# ----------------------------------------------------------------
# Candidate weight configurations
# ----------------------------------------------------------------
# Each candidate sums to 1.0.
#
# The initial V1 policy relied heavily on deterministic rule risk.
# The calibration grid tests lower rule weights and stronger AI
# evidence while retaining uncertainty as an auditable component.
CANDIDATE_WEIGHTS = [
    {
        "candidate_id": "W01",
        "rule_risk": 0.00,
        "ai_consensus": 0.80,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W02",
        "rule_risk": 0.05,
        "ai_consensus": 0.75,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W03",
        "rule_risk": 0.10,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W04",
        "rule_risk": 0.15,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W05",
        "rule_risk": 0.10,
        "ai_consensus": 0.75,
        "ai_prediction_vote": 0.10,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W06",
        "rule_risk": 0.15,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.10,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W07",
        "rule_risk": 0.20,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.10,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W08",
        "rule_risk": 0.10,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W09",
        "rule_risk": 0.15,
        "ai_consensus": 0.60,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W10",
        "rule_risk": 0.20,
        "ai_consensus": 0.55,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W11",
        "rule_risk": 0.05,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.15,
        "uncertainty": 0.10,
    },
    {
        "candidate_id": "W12",
        "rule_risk": 0.10,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.15,
        "uncertainty": 0.10,
    },
]


# ----------------------------------------------------------------
# Validation of candidate weights
# ----------------------------------------------------------------
for candidate in CANDIDATE_WEIGHTS:
    weight_sum = (
        candidate["rule_risk"]
        + candidate["ai_consensus"]
        + candidate["ai_prediction_vote"]
        + candidate["uncertainty"]
    )

    if not np.isclose(
        weight_sum,
        1.0,
    ):
        raise ValueError(
            f"Weights for {candidate['candidate_id']} "
            f"sum to {weight_sum}, not 1.0."
        )


# ----------------------------------------------------------------
# Prepare calibration dataset
# ----------------------------------------------------------------
calibration_source_df = (
    hybrid_decision_df.copy()
)

required_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ground_truth_label",
    "rule_risk_score",
    "ai_consensus_score",
    "ai_prediction_vote",
    "evidence_uncertainty_score",
]

missing_columns = [
    column
    for column in required_columns
    if column not in calibration_source_df.columns
]

if missing_columns:
    raise KeyError(
        "Missing calibration fields: "
        + ", ".join(
            missing_columns
        )
    )


# ----------------------------------------------------------------
# Standardize numeric values
# ----------------------------------------------------------------
for column in [
    "ground_truth_label",
    "rule_risk_score",
    "ai_consensus_score",
    "ai_prediction_vote",
    "evidence_uncertainty_score",
]:
    calibration_source_df[column] = (
        pd.to_numeric(
            calibration_source_df[column],
            errors="coerce",
        )
        .fillna(0.0)
    )

calibration_source_df[
    "ground_truth_label"
] = (
    calibration_source_df[
        "ground_truth_label"
    ]
    .clip(0, 1)
    .astype(int)
)


# ----------------------------------------------------------------
# Experiment-level calibration and holdout split
# ----------------------------------------------------------------
experiment_split_df = (
    calibration_source_df[
        [
            "experiment_id",
            "dataset",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset",
            "experiment_id",
        ]
    )
    .reset_index(drop=True)
)

group_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=CALIBRATION_EXPERIMENT_FRACTION,
    random_state=CALIBRATION_RANDOM_SEED,
)

calibration_indices, holdout_indices = next(
    group_splitter.split(
        experiment_split_df,
        groups=experiment_split_df[
            "experiment_id"
        ],
    )
)

calibration_experiment_ids = set(
    experiment_split_df.iloc[
        calibration_indices
    ][
        "experiment_id"
    ].astype(str)
)

holdout_experiment_ids = set(
    experiment_split_df.iloc[
        holdout_indices
    ][
        "experiment_id"
    ].astype(str)
)

calibration_df = (
    calibration_source_df.loc[
        calibration_source_df[
            "experiment_id"
        ]
        .astype(str)
        .isin(
            calibration_experiment_ids
        )
    ]
    .copy()
)

holdout_df = (
    calibration_source_df.loc[
        calibration_source_df[
            "experiment_id"
        ]
        .astype(str)
        .isin(
            holdout_experiment_ids
        )
    ]
    .copy()
)


# ----------------------------------------------------------------
# Split diagnostics
# ----------------------------------------------------------------
split_summary_df = pd.DataFrame([
    {
        "partition": "CALIBRATION",
        "experiments": (
            calibration_df[
                "experiment_id"
            ].nunique()
        ),
        "records": len(
            calibration_df
        ),
        "true_anomalies": int(
            calibration_df[
                "ground_truth_label"
            ].sum()
        ),
        "datasets": (
            calibration_df[
                "dataset"
            ].nunique()
        ),
    },
    {
        "partition": "HOLDOUT",
        "experiments": (
            holdout_df[
                "experiment_id"
            ].nunique()
        ),
        "records": len(
            holdout_df
        ),
        "true_anomalies": int(
            holdout_df[
                "ground_truth_label"
            ].sum()
        ),
        "datasets": (
            holdout_df[
                "dataset"
            ].nunique()
        ),
    },
])

display(split_summary_df)


# ----------------------------------------------------------------
# Metric helpers
# ----------------------------------------------------------------
def safe_ratio(
    numerator: float,
    denominator: float,
) -> float:
    if denominator == 0:
        return 0.0

    return float(
        numerator / denominator
    )


def compute_metrics(
    y_true: np.ndarray,
    y_prediction: np.ndarray,
    y_score: np.ndarray,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_prediction = np.asarray(
        y_prediction,
        dtype=int,
    )

    y_score = np.asarray(
        y_score,
        dtype=float,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1],
    ).ravel()

    precision = precision_score(
        y_true,
        y_prediction,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_prediction,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_prediction,
        zero_division=0,
    )

    fpr = safe_ratio(
        fp,
        fp + tn,
    )

    fnr = safe_ratio(
        fn,
        fn + tp,
    )

    specificity = safe_ratio(
        tn,
        tn + fp,
    )

    balanced_accuracy = (
        recall
        + specificity
    ) / 2.0

    if len(
        np.unique(
            y_true
        )
    ) > 1:
        pr_auc = average_precision_score(
            y_true,
            y_score,
        )
    else:
        pr_auc = np.nan

    return {
        "records": int(
            len(y_true)
        ),
        "true_anomalies": int(
            y_true.sum()
        ),
        "predicted_anomalies": int(
            y_prediction.sum()
        ),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "false_positive_rate": float(
            fpr
        ),
        "false_negative_rate": float(
            fnr
        ),
        "specificity": float(
            specificity
        ),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "pr_auc": (
            float(pr_auc)
            if not pd.isna(
                pr_auc
            )
            else np.nan
        ),
    }


def evaluate_macro_by_experiment(
    source_df: pd.DataFrame,
    score_column: str,
    prediction_column: str,
) -> dict:
    experiment_metric_rows = []

    for (
        experiment_id,
        dataset,
    ), experiment_df in source_df.groupby(
        [
            "experiment_id",
            "dataset",
        ],
        sort=False,
    ):
        result = compute_metrics(
            y_true=experiment_df[
                "ground_truth_label"
            ].to_numpy(),
            y_prediction=experiment_df[
                prediction_column
            ].to_numpy(),
            y_score=experiment_df[
                score_column
            ].to_numpy(),
        )

        result.update({
            "experiment_id": (
                experiment_id
            ),
            "dataset": dataset,
        })

        experiment_metric_rows.append(
            result
        )

    experiment_metric_df = pd.DataFrame(
        experiment_metric_rows
    )

    macro_result = {
        "experiments": int(
            experiment_metric_df[
                "experiment_id"
            ].nunique()
        ),
        "macro_precision": float(
            experiment_metric_df[
                "precision"
            ].mean()
        ),
        "macro_recall": float(
            experiment_metric_df[
                "recall"
            ].mean()
        ),
        "macro_f1": float(
            experiment_metric_df[
                "f1"
            ].mean()
        ),
        "macro_fpr": float(
            experiment_metric_df[
                "false_positive_rate"
            ].mean()
        ),
        "macro_fnr": float(
            experiment_metric_df[
                "false_negative_rate"
            ].mean()
        ),
        "macro_balanced_accuracy": float(
            experiment_metric_df[
                "balanced_accuracy"
            ].mean()
        ),
        "macro_pr_auc": float(
            experiment_metric_df[
                "pr_auc"
            ].mean()
        ),
    }

    return {
        "macro": macro_result,
        "experiment_metrics": (
            experiment_metric_df
        ),
    }


# ----------------------------------------------------------------
# Search calibration candidates
# ----------------------------------------------------------------
calibration_candidate_rows = []

print(
    f"Testing {len(CANDIDATE_WEIGHTS)} weight configurations "
    f"and {len(CANDIDATE_THRESHOLDS)} thresholds..."
)

for candidate in CANDIDATE_WEIGHTS:
    candidate_id = (
        candidate[
            "candidate_id"
        ]
    )

    score_column = (
        f"candidate_score_{candidate_id}"
    )

    calibration_df[
        score_column
    ] = (
        candidate[
            "rule_risk"
        ]
        * calibration_df[
            "rule_risk_score"
        ]
        + candidate[
            "ai_consensus"
        ]
        * calibration_df[
            "ai_consensus_score"
        ]
        + candidate[
            "ai_prediction_vote"
        ]
        * calibration_df[
            "ai_prediction_vote"
        ]
        + candidate[
            "uncertainty"
        ]
        * calibration_df[
            "evidence_uncertainty_score"
        ]
    ).clip(
        0.0,
        1.0,
    )

    for threshold in CANDIDATE_THRESHOLDS:
        prediction_column = (
            f"candidate_prediction_{candidate_id}"
        )

        calibration_df[
            prediction_column
        ] = (
            calibration_df[
                score_column
            ]
            >= threshold
        ).astype(int)

        evaluation = (
            evaluate_macro_by_experiment(
                source_df=calibration_df,
                score_column=score_column,
                prediction_column=prediction_column,
            )
        )

        macro = evaluation[
            "macro"
        ]

        objective_score = (
            macro[
                "macro_f1"
            ]
            - FPR_PENALTY_WEIGHT
            * macro[
                "macro_fpr"
            ]
        )

        calibration_candidate_rows.append({
            "candidate_id": candidate_id,
            "threshold": float(
                threshold
            ),
            "rule_risk_weight": float(
                candidate[
                    "rule_risk"
                ]
            ),
            "ai_consensus_weight": float(
                candidate[
                    "ai_consensus"
                ]
            ),
            "ai_prediction_vote_weight": float(
                candidate[
                    "ai_prediction_vote"
                ]
            ),
            "uncertainty_weight": float(
                candidate[
                    "uncertainty"
                ]
            ),
            "experiments": int(
                macro[
                    "experiments"
                ]
            ),
            "macro_precision": float(
                macro[
                    "macro_precision"
                ]
            ),
            "macro_recall": float(
                macro[
                    "macro_recall"
                ]
            ),
            "macro_f1": float(
                macro[
                    "macro_f1"
                ]
            ),
            "macro_fpr": float(
                macro[
                    "macro_fpr"
                ]
            ),
            "macro_fnr": float(
                macro[
                    "macro_fnr"
                ]
            ),
            "macro_balanced_accuracy": float(
                macro[
                    "macro_balanced_accuracy"
                ]
            ),
            "macro_pr_auc": float(
                macro[
                    "macro_pr_auc"
                ]
            ),
            "objective_score": float(
                objective_score
            ),
        })

        del calibration_df[
            prediction_column
        ]

    del calibration_df[
        score_column
    ]


calibration_search_df = pd.DataFrame(
    calibration_candidate_rows
)


# ----------------------------------------------------------------
# Select best candidate
# ----------------------------------------------------------------
best_candidate_row = (
    calibration_search_df
    .sort_values(
        [
            "objective_score",
            "macro_f1",
            "macro_fpr",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .iloc[0]
)

BEST_CANDIDATE_ID = str(
    best_candidate_row[
        "candidate_id"
    ]
)

BEST_THRESHOLD = float(
    best_candidate_row[
        "threshold"
    ]
)

BEST_WEIGHTS = {
    "rule_risk": float(
        best_candidate_row[
            "rule_risk_weight"
        ]
    ),
    "ai_consensus": float(
        best_candidate_row[
            "ai_consensus_weight"
        ]
    ),
    "ai_prediction_vote": float(
        best_candidate_row[
            "ai_prediction_vote_weight"
        ]
    ),
    "uncertainty": float(
        best_candidate_row[
            "uncertainty_weight"
        ]
    ),
}

print("-" * 80)
print(
    f"Best calibration candidate: "
    f"{BEST_CANDIDATE_ID}"
)
print(
    f"Best threshold: "
    f"{BEST_THRESHOLD:.3f}"
)
print(
    f"Best weights: "
    f"{BEST_WEIGHTS}"
)
print(
    f"Calibration macro-F1: "
    f"{best_candidate_row['macro_f1']:.4f}"
)
print(
    f"Calibration macro-FPR: "
    f"{best_candidate_row['macro_fpr']:.4f}"
)
print(
    f"Calibration objective: "
    f"{best_candidate_row['objective_score']:.4f}"
)


# ----------------------------------------------------------------
# Show top calibration candidates
# ----------------------------------------------------------------
top_candidates_df = (
    calibration_search_df
    .sort_values(
        [
            "objective_score",
            "macro_f1",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(20)
    .reset_index(drop=True)
)

display(top_candidates_df)


# ----------------------------------------------------------------
# Apply selected V2 configuration to calibration and holdout data
# ----------------------------------------------------------------
def apply_v2_configuration(
    source_df: pd.DataFrame,
) -> pd.DataFrame:
    output_df = source_df.copy()

    output_df[
        "hybrid_v2_risk_score"
    ] = (
        BEST_WEIGHTS[
            "rule_risk"
        ]
        * output_df[
            "rule_risk_score"
        ]
        + BEST_WEIGHTS[
            "ai_consensus"
        ]
        * output_df[
            "ai_consensus_score"
        ]
        + BEST_WEIGHTS[
            "ai_prediction_vote"
        ]
        * output_df[
            "ai_prediction_vote"
        ]
        + BEST_WEIGHTS[
            "uncertainty"
        ]
        * output_df[
            "evidence_uncertainty_score"
        ]
    ).clip(
        0.0,
        1.0,
    )

    output_df[
        "hybrid_v2_anomaly_prediction"
    ] = (
        output_df[
            "hybrid_v2_risk_score"
        ]
        >= BEST_THRESHOLD
    ).astype(int)

    output_df[
        "hybrid_v2_trust_score"
    ] = (
        1.0
        - output_df[
            "hybrid_v2_risk_score"
        ]
    ).clip(
        0.0,
        1.0,
    )

    return output_df


calibration_v2_df = (
    apply_v2_configuration(
        calibration_df
    )
)

holdout_v2_df = (
    apply_v2_configuration(
        holdout_df
    )
)


# ----------------------------------------------------------------
# Evaluate selected V2 configuration
# ----------------------------------------------------------------
calibration_v2_evaluation = (
    evaluate_macro_by_experiment(
        source_df=calibration_v2_df,
        score_column=(
            "hybrid_v2_risk_score"
        ),
        prediction_column=(
            "hybrid_v2_anomaly_prediction"
        ),
    )
)

holdout_v2_evaluation = (
    evaluate_macro_by_experiment(
        source_df=holdout_v2_df,
        score_column=(
            "hybrid_v2_risk_score"
        ),
        prediction_column=(
            "hybrid_v2_anomaly_prediction"
        ),
    )
)


# ----------------------------------------------------------------
# Evaluate holdout AI-consensus baseline
# ----------------------------------------------------------------
holdout_v2_df[
    "ai_consensus_prediction_holdout"
] = (
    holdout_v2_df[
        "ai_prediction_vote"
    ]
    >= 0.5
).astype(int)

holdout_ai_evaluation = (
    evaluate_macro_by_experiment(
        source_df=holdout_v2_df,
        score_column=(
            "ai_consensus_score"
        ),
        prediction_column=(
            "ai_consensus_prediction_holdout"
        ),
    )
)


# ----------------------------------------------------------------
# Evaluate holdout Hybrid V1
# ----------------------------------------------------------------
holdout_v1_evaluation = (
    evaluate_macro_by_experiment(
        source_df=holdout_v2_df,
        score_column=(
            "hybrid_risk_score"
        ),
        prediction_column=(
            "hybrid_anomaly_prediction"
        ),
    )
)


# ----------------------------------------------------------------
# Comparative holdout summary
# ----------------------------------------------------------------
holdout_comparison_rows = []

for method_name, evaluation in [
    (
        "HYBRID_V2_CALIBRATED",
        holdout_v2_evaluation,
    ),
    (
        "HYBRID_V1",
        holdout_v1_evaluation,
    ),
    (
        "AI_CONSENSUS",
        holdout_ai_evaluation,
    ),
]:
    macro = evaluation[
        "macro"
    ]

    holdout_comparison_rows.append({
        "method": method_name,
        "experiments": (
            macro[
                "experiments"
            ]
        ),
        "macro_precision": (
            macro[
                "macro_precision"
            ]
        ),
        "macro_recall": (
            macro[
                "macro_recall"
            ]
        ),
        "macro_f1": (
            macro[
                "macro_f1"
            ]
        ),
        "macro_fpr": (
            macro[
                "macro_fpr"
            ]
        ),
        "macro_fnr": (
            macro[
                "macro_fnr"
            ]
        ),
        "macro_balanced_accuracy": (
            macro[
                "macro_balanced_accuracy"
            ]
        ),
        "macro_pr_auc": (
            macro[
                "macro_pr_auc"
            ]
        ),
    })

holdout_comparison_df = pd.DataFrame(
    holdout_comparison_rows
).sort_values(
    "macro_f1",
    ascending=False,
).reset_index(drop=True)

display(holdout_comparison_df)


# ----------------------------------------------------------------
# Dataset-level holdout evaluation
# ----------------------------------------------------------------
dataset_holdout_rows = []

for dataset_name, dataset_df in (
    holdout_v2_df.groupby(
        "dataset",
        sort=True,
    )
):
    for method_name, score_column, prediction_column in [
        (
            "HYBRID_V2_CALIBRATED",
            "hybrid_v2_risk_score",
            "hybrid_v2_anomaly_prediction",
        ),
        (
            "HYBRID_V1",
            "hybrid_risk_score",
            "hybrid_anomaly_prediction",
        ),
        (
            "AI_CONSENSUS",
            "ai_consensus_score",
            "ai_consensus_prediction_holdout",
        ),
    ]:
        dataset_evaluation = (
            evaluate_macro_by_experiment(
                source_df=dataset_df,
                score_column=score_column,
                prediction_column=prediction_column,
            )
        )

        macro = dataset_evaluation[
            "macro"
        ]

        dataset_holdout_rows.append({
            "dataset": dataset_name,
            "method": method_name,
            "experiments": (
                macro[
                    "experiments"
                ]
            ),
            "macro_precision": (
                macro[
                    "macro_precision"
                ]
            ),
            "macro_recall": (
                macro[
                    "macro_recall"
                ]
            ),
            "macro_f1": (
                macro[
                    "macro_f1"
                ]
            ),
            "macro_fpr": (
                macro[
                    "macro_fpr"
                ]
            ),
            "macro_fnr": (
                macro[
                    "macro_fnr"
                ]
            ),
            "macro_balanced_accuracy": (
                macro[
                    "macro_balanced_accuracy"
                ]
            ),
            "macro_pr_auc": (
                macro[
                    "macro_pr_auc"
                ]
            ),
        })

dataset_holdout_comparison_df = (
    pd.DataFrame(
        dataset_holdout_rows
    )
    .sort_values(
        [
            "dataset",
            "macro_f1",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(dataset_holdout_comparison_df)


# ----------------------------------------------------------------
# Apply calibrated V2 policy to the full hybrid dataset
# ----------------------------------------------------------------
hybrid_v2_full_df = (
    apply_v2_configuration(
        hybrid_decision_df
    )
)

hybrid_v2_full_df[
    "calibration_version"
] = (
    CALIBRATION_VERSION
)

hybrid_v2_full_df[
    "calibrated_candidate_id"
] = (
    BEST_CANDIDATE_ID
)

hybrid_v2_full_df[
    "calibrated_threshold"
] = (
    BEST_THRESHOLD
)

hybrid_v2_full_df[
    "calibration_partition"
] = np.where(
    hybrid_v2_full_df[
        "experiment_id"
    ]
    .astype(str)
    .isin(
        calibration_experiment_ids
    ),
    "CALIBRATION",
    "HOLDOUT",
)


# ----------------------------------------------------------------
# Assign V2 governed action categories
# ----------------------------------------------------------------
def assign_v2_decision(
    row: pd.Series,
) -> str:
    """
    Convert the calibrated binary risk into governed actions.

    ACCEPT:
        Below the calibrated anomaly threshold.

    REPAIR:
        Above threshold with deterministic rule evidence, but without
        critical rule severity.

    QUARANTINE:
        Above threshold with strong AI evidence or critical rule risk.

    ESCALATE:
        Above threshold with incomplete or conflicting evidence.
    """
    is_anomaly = int(
        row[
            "hybrid_v2_anomaly_prediction"
        ]
    ) == 1

    if not is_anomaly:
        return "ACCEPT"

    insufficient_evidence = (
        float(
            row[
                "evidence_completeness"
            ]
        )
        < 2 / 3
    )

    high_disagreement = (
        float(
            row[
                "ai_score_disagreement"
            ]
        )
        >= 0.50
    )

    rule_ai_conflict = (
        int(
            row[
                "rule_ai_conflict_flag"
            ]
        )
        == 1
    )

    critical_rule = (
        float(
            row[
                "rule_severity_score"
            ]
        )
        >= 0.75
    )

    rule_failure = (
        int(
            row[
                "rule_anomaly_prediction"
            ]
        )
        == 1
    )

    high_ai_evidence = (
        float(
            row[
                "ai_consensus_score"
            ]
        )
        >= 0.80
    )

    if insufficient_evidence:
        return "ESCALATE"

    if (
        high_disagreement
        and rule_ai_conflict
    ):
        return "ESCALATE"

    if critical_rule:
        return "QUARANTINE"

    if (
        rule_failure
        and not high_ai_evidence
    ):
        return "REPAIR"

    return "QUARANTINE"


hybrid_v2_full_df[
    "hybrid_v2_decision"
] = (
    hybrid_v2_full_df.apply(
        assign_v2_decision,
        axis=1,
    )
)


# ----------------------------------------------------------------
# Save calibrated policy
# ----------------------------------------------------------------
calibrated_policy = {
    "calibration_version": (
        CALIBRATION_VERSION
    ),
    "created_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "calibration_random_seed": (
        CALIBRATION_RANDOM_SEED
    ),
    "calibration_experiment_fraction": (
        CALIBRATION_EXPERIMENT_FRACTION
    ),
    "fpr_penalty_weight": (
        FPR_PENALTY_WEIGHT
    ),
    "selected_candidate_id": (
        BEST_CANDIDATE_ID
    ),
    "selected_threshold": (
        BEST_THRESHOLD
    ),
    "selected_weights": (
        BEST_WEIGHTS
    ),
    "calibration_experiment_count": int(
        len(
            calibration_experiment_ids
        )
    ),
    "holdout_experiment_count": int(
        len(
            holdout_experiment_ids
        )
    ),
    "calibration_experiment_ids": sorted(
        calibration_experiment_ids
    ),
    "holdout_experiment_ids": sorted(
        holdout_experiment_ids
    ),
}

CALIBRATED_POLICY_PATH = (
    HYBRID_CONFIG_DIR
    / "hybrid_policy_v2_calibrated.json"
)

with open(
    CALIBRATED_POLICY_PATH,
    "w",
    encoding="utf-8",
) as policy_file:
    json.dump(
        calibrated_policy,
        policy_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Save outputs
# ----------------------------------------------------------------
CALIBRATION_SEARCH_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v2_calibration_search.csv"
)

calibration_search_df.to_csv(
    CALIBRATION_SEARCH_PATH,
    index=False,
)

TOP_CANDIDATES_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v2_top_calibration_candidates.csv"
)

top_candidates_df.to_csv(
    TOP_CANDIDATES_PATH,
    index=False,
)

HOLDOUT_COMPARISON_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v2_holdout_comparison.csv"
)

holdout_comparison_df.to_csv(
    HOLDOUT_COMPARISON_PATH,
    index=False,
)

DATASET_HOLDOUT_COMPARISON_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v2_dataset_holdout_comparison.csv"
)

dataset_holdout_comparison_df.to_csv(
    DATASET_HOLDOUT_COMPARISON_PATH,
    index=False,
)

HYBRID_V2_FULL_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v2_calibrated_decisions.parquet"
)

hybrid_v2_full_df.to_parquet(
    HYBRID_V2_FULL_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Statistical comparison diagnostics
# ----------------------------------------------------------------
v2_holdout_row = (
    holdout_comparison_df.loc[
        holdout_comparison_df[
            "method"
        ].eq(
            "HYBRID_V2_CALIBRATED"
        )
    ]
    .iloc[0]
)

v1_holdout_row = (
    holdout_comparison_df.loc[
        holdout_comparison_df[
            "method"
        ].eq(
            "HYBRID_V1"
        )
    ]
    .iloc[0]
)

ai_holdout_row = (
    holdout_comparison_df.loc[
        holdout_comparison_df[
            "method"
        ].eq(
            "AI_CONSENSUS"
        )
    ]
    .iloc[0]
)

calibration_diagnostics = {
    "v2_f1_above_v1_holdout": (
        v2_holdout_row[
            "macro_f1"
        ]
        >
        v1_holdout_row[
            "macro_f1"
        ]
    ),

    "v2_fpr_below_v1_holdout": (
        v2_holdout_row[
            "macro_fpr"
        ]
        <
        v1_holdout_row[
            "macro_fpr"
        ]
    ),

    "v2_f1_above_ai_consensus_holdout": (
        v2_holdout_row[
            "macro_f1"
        ]
        >
        ai_holdout_row[
            "macro_f1"
        ]
    ),

    "v2_fpr_below_ai_consensus_holdout": (
        v2_holdout_row[
            "macro_fpr"
        ]
        <
        ai_holdout_row[
            "macro_fpr"
        ]
    ),

    "calibration_and_holdout_disjoint": (
        len(
            calibration_experiment_ids
            .intersection(
                holdout_experiment_ids
            )
        )
        == 0
    ),
}

calibration_diagnostics_df = pd.DataFrame({
    "diagnostic": (
        calibration_diagnostics.keys()
    ),
    "passed": (
        calibration_diagnostics.values()
    ),
})

display(calibration_diagnostics_df)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
validation_checks = {
    "all_63_experiments_partitioned": (
        len(
            calibration_experiment_ids
        )
        + len(
            holdout_experiment_ids
        )
        == 63
    ),

    "calibration_not_empty": (
        len(
            calibration_df
        )
        > 0
    ),

    "holdout_not_empty": (
        len(
            holdout_df
        )
        > 0
    ),

    "three_datasets_in_calibration": (
        calibration_df[
            "dataset"
        ]
        .nunique()
        == 3
    ),

    "three_datasets_in_holdout": (
        holdout_df[
            "dataset"
        ]
        .nunique()
        == 3
    ),

    "selected_threshold_valid": (
        0.0
        <= BEST_THRESHOLD
        <= 1.0
    ),

    "selected_weights_sum_to_one": (
        np.isclose(
            sum(
                BEST_WEIGHTS.values()
            ),
            1.0,
        )
    ),

    "v2_scores_valid": (
        hybrid_v2_full_df[
            "hybrid_v2_risk_score"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),

    "v2_predictions_binary": (
        set(
            hybrid_v2_full_df[
                "hybrid_v2_anomaly_prediction"
            ]
            .unique()
        )
        .issubset(
            {0, 1}
        )
    ),

    "valid_v2_decisions": (
        set(
            hybrid_v2_full_df[
                "hybrid_v2_decision"
            ]
            .unique()
        )
        .issubset({
            "ACCEPT",
            "REPAIR",
            "QUARANTINE",
            "ESCALATE",
        })
    ),
}

calibration_validation_df = pd.DataFrame({
    "check": (
        validation_checks.keys()
    ),
    "passed": (
        validation_checks.values()
    ),
})

display(calibration_validation_df)

assert calibration_validation_df[
    "passed"
].all(), (
    "One or more calibration checks failed."
)


# ----------------------------------------------------------------
# Save calibration audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

calibration_audit = {
    "hybrid_run_id": (
        HYBRID_RUN_ID
    ),
    "calibration_version": (
        CALIBRATION_VERSION
    ),
    "execution_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "candidate_weight_configurations": (
        len(
            CANDIDATE_WEIGHTS
        )
    ),
    "candidate_thresholds": (
        len(
            CANDIDATE_THRESHOLDS
        )
    ),
    "total_candidate_combinations": int(
        len(
            calibration_search_df
        )
    ),
    "selected_candidate_id": (
        BEST_CANDIDATE_ID
    ),
    "selected_threshold": (
        BEST_THRESHOLD
    ),
    "selected_weights": (
        BEST_WEIGHTS
    ),
    "calibration_macro_f1": float(
        best_candidate_row[
            "macro_f1"
        ]
    ),
    "calibration_macro_fpr": float(
        best_candidate_row[
            "macro_fpr"
        ]
    ),
    "holdout_v2_macro_f1": float(
        v2_holdout_row[
            "macro_f1"
        ]
    ),
    "holdout_v2_macro_fpr": float(
        v2_holdout_row[
            "macro_fpr"
        ]
    ),
    "holdout_ai_consensus_macro_f1": float(
        ai_holdout_row[
            "macro_f1"
        ]
    ),
    "holdout_ai_consensus_macro_fpr": float(
        ai_holdout_row[
            "macro_fpr"
        ]
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "calibrated_policy_path": str(
        CALIBRATED_POLICY_PATH
    ),
    "hybrid_v2_output_path": str(
        HYBRID_V2_FULL_PATH
    ),
}

CALIBRATION_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "hybrid_v2_calibration_audit.json"
)

with open(
    CALIBRATION_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        calibration_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Hybrid V2 calibration completed successfully."
)
print(
    f"Calibration version: "
    f"{CALIBRATION_VERSION}"
)
print(
    f"Selected candidate: "
    f"{BEST_CANDIDATE_ID}"
)
print(
    f"Selected threshold: "
    f"{BEST_THRESHOLD:.3f}"
)
print(
    f"Selected weights: "
    f"{BEST_WEIGHTS}"
)
print(
    f"Calibration experiments: "
    f"{len(calibration_experiment_ids)}"
)
print(
    f"Holdout experiments: "
    f"{len(holdout_experiment_ids)}"
)
print(
    f"Holdout V2 macro-F1: "
    f"{v2_holdout_row['macro_f1']:.4f}"
)
print(
    f"Holdout V2 macro-FPR: "
    f"{v2_holdout_row['macro_fpr']:.4f}"
)
print(
    f"Holdout AI-consensus macro-F1: "
    f"{ai_holdout_row['macro_f1']:.4f}"
)
print(
    f"Holdout AI-consensus macro-FPR: "
    f"{ai_holdout_row['macro_fpr']:.4f}"
)
print(
    f"Calibrated policy: "
    f"{CALIBRATED_POLICY_PATH}"
)
print(
    f"V2 decisions: "
    f"{HYBRID_V2_FULL_PATH}"
)
print(
    f"Calibration audit: "
    f"{CALIBRATION_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

## 7. Nested Calibration of Hybrid V3

This section performs nested experiment-level cross-validation so that policy selection and outer-fold evaluation remain separated.


In [ ]:
# ================================================================
# Cell 7 — Nested constrained calibration for Hybrid V3
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# V3 calibration configuration
# ----------------------------------------------------------------
V3_VERSION = "HYBRID_NESTED_CV_V3.0"

OUTER_FOLDS = 3
INNER_FOLDS = 3
RANDOM_SEED = 42

# The selected hybrid configuration must keep its validation FPR
# close to the AI-consensus FPR.
FPR_TOLERANCE_ABOVE_AI = 0.02

CANDIDATE_THRESHOLDS = np.round(
    np.arange(
        0.25,
        0.651,
        0.025,
    ),
    3,
).tolist()


CANDIDATE_WEIGHTS = [
    {
        "candidate_id": "W01",
        "rule_risk": 0.00,
        "ai_consensus": 0.80,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W02",
        "rule_risk": 0.05,
        "ai_consensus": 0.75,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W03",
        "rule_risk": 0.10,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W04",
        "rule_risk": 0.15,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W05",
        "rule_risk": 0.05,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W06",
        "rule_risk": 0.10,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W07",
        "rule_risk": 0.15,
        "ai_consensus": 0.60,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W08",
        "rule_risk": 0.20,
        "ai_consensus": 0.55,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W09",
        "rule_risk": 0.05,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.15,
        "uncertainty": 0.10,
    },
    {
        "candidate_id": "W10",
        "rule_risk": 0.10,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.15,
        "uncertainty": 0.10,
    },
]


# ----------------------------------------------------------------
# Validate candidate weights
# ----------------------------------------------------------------
for candidate in CANDIDATE_WEIGHTS:
    weight_sum = (
        candidate["rule_risk"]
        + candidate["ai_consensus"]
        + candidate["ai_prediction_vote"]
        + candidate["uncertainty"]
    )

    if not np.isclose(
        weight_sum,
        1.0,
    ):
        raise ValueError(
            f"{candidate['candidate_id']} weights sum to "
            f"{weight_sum}, not 1.0."
        )


# ----------------------------------------------------------------
# Prepare source data
# ----------------------------------------------------------------
v3_source_df = hybrid_decision_df.copy()

required_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ground_truth_label",
    "rule_risk_score",
    "ai_consensus_score",
    "ai_prediction_vote",
    "evidence_uncertainty_score",
]

missing_columns = [
    column
    for column in required_columns
    if column not in v3_source_df.columns
]

if missing_columns:
    raise KeyError(
        "Missing V3 fields: "
        + ", ".join(missing_columns)
    )


for column in [
    "ground_truth_label",
    "rule_risk_score",
    "ai_consensus_score",
    "ai_prediction_vote",
    "evidence_uncertainty_score",
]:
    v3_source_df[column] = (
        pd.to_numeric(
            v3_source_df[column],
            errors="coerce",
        )
        .fillna(0.0)
    )

v3_source_df["ground_truth_label"] = (
    v3_source_df["ground_truth_label"]
    .clip(0, 1)
    .astype(int)
)


# ----------------------------------------------------------------
# Experiment table for stratified splitting
# ----------------------------------------------------------------
experiment_table_df = (
    v3_source_df[
        [
            "experiment_id",
            "dataset",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset",
            "experiment_id",
        ]
    )
    .reset_index(drop=True)
)

if len(experiment_table_df) != 63:
    raise ValueError(
        "Expected 63 unique experiments, found "
        f"{len(experiment_table_df)}."
    )


# ----------------------------------------------------------------
# Metric helpers
# ----------------------------------------------------------------
def safe_divide(
    numerator: float,
    denominator: float,
) -> float:
    if denominator == 0:
        return 0.0

    return float(
        numerator / denominator
    )


def calculate_metrics(
    y_true,
    y_pred,
    y_score,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    y_score = np.asarray(
        y_score,
        dtype=float,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    fpr = safe_divide(
        fp,
        fp + tn,
    )

    fnr = safe_divide(
        fn,
        fn + tp,
    )

    specificity = safe_divide(
        tn,
        tn + fp,
    )

    balanced_accuracy = (
        recall
        + specificity
    ) / 2.0

    if len(np.unique(y_true)) > 1:
        pr_auc = average_precision_score(
            y_true,
            y_score,
        )
    else:
        pr_auc = np.nan

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "fnr": float(fnr),
        "specificity": float(specificity),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "pr_auc": float(pr_auc),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }


def calculate_macro_experiment_metrics(
    source_df: pd.DataFrame,
    score_values: np.ndarray,
    prediction_values: np.ndarray,
) -> dict:
    working_df = source_df[
        [
            "experiment_id",
            "dataset",
            "ground_truth_label",
        ]
    ].copy()

    working_df["_score"] = score_values
    working_df["_prediction"] = (
        prediction_values
    )

    rows = []

    for (
        experiment_id,
        dataset,
    ), experiment_df in working_df.groupby(
        [
            "experiment_id",
            "dataset",
        ],
        sort=False,
    ):
        metrics = calculate_metrics(
            y_true=experiment_df[
                "ground_truth_label"
            ],
            y_pred=experiment_df[
                "_prediction"
            ],
            y_score=experiment_df[
                "_score"
            ],
        )

        metrics.update({
            "experiment_id": experiment_id,
            "dataset": dataset,
        })

        rows.append(metrics)

    metrics_df = pd.DataFrame(rows)

    return {
        "macro_precision": float(
            metrics_df["precision"].mean()
        ),
        "macro_recall": float(
            metrics_df["recall"].mean()
        ),
        "macro_f1": float(
            metrics_df["f1"].mean()
        ),
        "macro_fpr": float(
            metrics_df["fpr"].mean()
        ),
        "macro_fnr": float(
            metrics_df["fnr"].mean()
        ),
        "macro_balanced_accuracy": float(
            metrics_df[
                "balanced_accuracy"
            ].mean()
        ),
        "macro_pr_auc": float(
            metrics_df["pr_auc"].mean()
        ),
        "experiment_metrics": metrics_df,
    }


def build_candidate_score(
    source_df: pd.DataFrame,
    candidate: dict,
) -> np.ndarray:
    return (
        candidate["rule_risk"]
        * source_df[
            "rule_risk_score"
        ].to_numpy()
        + candidate["ai_consensus"]
        * source_df[
            "ai_consensus_score"
        ].to_numpy()
        + candidate["ai_prediction_vote"]
        * source_df[
            "ai_prediction_vote"
        ].to_numpy()
        + candidate["uncertainty"]
        * source_df[
            "evidence_uncertainty_score"
        ].to_numpy()
    ).clip(
        0.0,
        1.0,
    )


# ----------------------------------------------------------------
# Outer stratified cross-validation
# ----------------------------------------------------------------
outer_splitter = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

outer_result_rows = []
inner_search_rows = []
selected_configuration_rows = []
outer_record_predictions = []

outer_split_iterator = outer_splitter.split(
    experiment_table_df[
        "experiment_id"
    ],
    experiment_table_df[
        "dataset"
    ],
)


for outer_fold, (
    outer_train_indices,
    outer_test_indices,
) in enumerate(
    outer_split_iterator,
    start=1,
):
    print(
        "-" * 80
    )
    print(
        f"Outer fold {outer_fold}/{OUTER_FOLDS}"
    )

    outer_train_experiments_df = (
        experiment_table_df.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_experiments_df = (
        experiment_table_df.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_train_ids = set(
        outer_train_experiments_df[
            "experiment_id"
        ]
    )

    outer_test_ids = set(
        outer_test_experiments_df[
            "experiment_id"
        ]
    )

    outer_train_df = (
        v3_source_df.loc[
            v3_source_df[
                "experiment_id"
            ].isin(
                outer_train_ids
            )
        ]
        .copy()
    )

    outer_test_df = (
        v3_source_df.loc[
            v3_source_df[
                "experiment_id"
            ].isin(
                outer_test_ids
            )
        ]
        .copy()
    )


    # ------------------------------------------------------------
    # Inner stratified cross-validation
    # ------------------------------------------------------------
    inner_splitter = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + outer_fold
        ),
    )

    candidate_fold_results = []

    inner_iterator = inner_splitter.split(
        outer_train_experiments_df[
            "experiment_id"
        ],
        outer_train_experiments_df[
            "dataset"
        ],
    )

    inner_splits = list(
        inner_iterator
    )

    for candidate in CANDIDATE_WEIGHTS:
        candidate_id = (
            candidate[
                "candidate_id"
            ]
        )

        for threshold in CANDIDATE_THRESHOLDS:
            inner_validation_rows = []

            for inner_fold, (
                inner_train_indices,
                inner_validation_indices,
            ) in enumerate(
                inner_splits,
                start=1,
            ):
                validation_experiment_ids = set(
                    outer_train_experiments_df
                    .iloc[
                        inner_validation_indices
                    ][
                        "experiment_id"
                    ]
                )

                validation_df = (
                    outer_train_df.loc[
                        outer_train_df[
                            "experiment_id"
                        ].isin(
                            validation_experiment_ids
                        )
                    ]
                    .copy()
                )

                hybrid_score = (
                    build_candidate_score(
                        validation_df,
                        candidate,
                    )
                )

                hybrid_prediction = (
                    hybrid_score
                    >= threshold
                ).astype(int)

                hybrid_metrics = (
                    calculate_macro_experiment_metrics(
                        source_df=validation_df,
                        score_values=hybrid_score,
                        prediction_values=(
                            hybrid_prediction
                        ),
                    )
                )

                ai_score = (
                    validation_df[
                        "ai_consensus_score"
                    ].to_numpy()
                )

                ai_prediction = (
                    validation_df[
                        "ai_prediction_vote"
                    ].to_numpy()
                    >= 0.5
                ).astype(int)

                ai_metrics = (
                    calculate_macro_experiment_metrics(
                        source_df=validation_df,
                        score_values=ai_score,
                        prediction_values=(
                            ai_prediction
                        ),
                    )
                )

                inner_validation_rows.append({
                    "outer_fold": outer_fold,
                    "inner_fold": inner_fold,
                    "candidate_id": (
                        candidate_id
                    ),
                    "threshold": float(
                        threshold
                    ),
                    "hybrid_macro_precision": (
                        hybrid_metrics[
                            "macro_precision"
                        ]
                    ),
                    "hybrid_macro_recall": (
                        hybrid_metrics[
                            "macro_recall"
                        ]
                    ),
                    "hybrid_macro_f1": (
                        hybrid_metrics[
                            "macro_f1"
                        ]
                    ),
                    "hybrid_macro_fpr": (
                        hybrid_metrics[
                            "macro_fpr"
                        ]
                    ),
                    "hybrid_macro_fnr": (
                        hybrid_metrics[
                            "macro_fnr"
                        ]
                    ),
                    "hybrid_macro_pr_auc": (
                        hybrid_metrics[
                            "macro_pr_auc"
                        ]
                    ),
                    "ai_macro_f1": (
                        ai_metrics[
                            "macro_f1"
                        ]
                    ),
                    "ai_macro_fpr": (
                        ai_metrics[
                            "macro_fpr"
                        ]
                    ),
                })

            candidate_fold_df = pd.DataFrame(
                inner_validation_rows
            )

            candidate_summary = {
                "outer_fold": outer_fold,
                "candidate_id": (
                    candidate_id
                ),
                "threshold": float(
                    threshold
                ),
                "rule_risk_weight": float(
                    candidate[
                        "rule_risk"
                    ]
                ),
                "ai_consensus_weight": float(
                    candidate[
                        "ai_consensus"
                    ]
                ),
                "ai_prediction_vote_weight": float(
                    candidate[
                        "ai_prediction_vote"
                    ]
                ),
                "uncertainty_weight": float(
                    candidate[
                        "uncertainty"
                    ]
                ),
                "mean_validation_precision": float(
                    candidate_fold_df[
                        "hybrid_macro_precision"
                    ].mean()
                ),
                "mean_validation_recall": float(
                    candidate_fold_df[
                        "hybrid_macro_recall"
                    ].mean()
                ),
                "mean_validation_f1": float(
                    candidate_fold_df[
                        "hybrid_macro_f1"
                    ].mean()
                ),
                "mean_validation_fpr": float(
                    candidate_fold_df[
                        "hybrid_macro_fpr"
                    ].mean()
                ),
                "mean_validation_fnr": float(
                    candidate_fold_df[
                        "hybrid_macro_fnr"
                    ].mean()
                ),
                "mean_validation_pr_auc": float(
                    candidate_fold_df[
                        "hybrid_macro_pr_auc"
                    ].mean()
                ),
                "mean_ai_validation_f1": float(
                    candidate_fold_df[
                        "ai_macro_f1"
                    ].mean()
                ),
                "mean_ai_validation_fpr": float(
                    candidate_fold_df[
                        "ai_macro_fpr"
                    ].mean()
                ),
            }

            candidate_summary[
                "maximum_allowed_fpr"
            ] = (
                candidate_summary[
                    "mean_ai_validation_fpr"
                ]
                + FPR_TOLERANCE_ABOVE_AI
            )

            candidate_summary[
                "fpr_constraint_satisfied"
            ] = (
                candidate_summary[
                    "mean_validation_fpr"
                ]
                <= candidate_summary[
                    "maximum_allowed_fpr"
                ]
            )

            candidate_summary[
                "fpr_constraint_violation"
            ] = max(
                0.0,
                candidate_summary[
                    "mean_validation_fpr"
                ]
                - candidate_summary[
                    "maximum_allowed_fpr"
                ],
            )

            candidate_fold_results.append(
                candidate_summary
            )

            inner_search_rows.extend(
                inner_validation_rows
            )


    # ------------------------------------------------------------
    # Select configuration using inner validation only
    # ------------------------------------------------------------
    candidate_results_df = pd.DataFrame(
        candidate_fold_results
    )

    feasible_candidates_df = (
        candidate_results_df.loc[
            candidate_results_df[
                "fpr_constraint_satisfied"
            ]
        ]
        .copy()
    )

    if not feasible_candidates_df.empty:
        selected_row = (
            feasible_candidates_df
            .sort_values(
                [
                    "mean_validation_f1",
                    "mean_validation_fpr",
                    "mean_validation_precision",
                ],
                ascending=[
                    False,
                    True,
                    False,
                ],
            )
            .iloc[0]
        )

        selection_mode = (
            "FPR_CONSTRAINED"
        )

    else:
        selected_row = (
            candidate_results_df
            .sort_values(
                [
                    "fpr_constraint_violation",
                    "mean_validation_f1",
                    "mean_validation_fpr",
                ],
                ascending=[
                    True,
                    False,
                    True,
                ],
            )
            .iloc[0]
        )

        selection_mode = (
            "MINIMUM_FPR_VIOLATION"
        )

    selected_candidate_id = str(
        selected_row[
            "candidate_id"
        ]
    )

    selected_threshold = float(
        selected_row[
            "threshold"
        ]
    )

    selected_candidate = next(
        candidate
        for candidate in CANDIDATE_WEIGHTS
        if candidate[
            "candidate_id"
        ]
        == selected_candidate_id
    )

    selected_configuration_rows.append({
        "outer_fold": outer_fold,
        "selection_mode": selection_mode,
        "candidate_id": (
            selected_candidate_id
        ),
        "threshold": (
            selected_threshold
        ),
        "rule_risk_weight": (
            selected_candidate[
                "rule_risk"
            ]
        ),
        "ai_consensus_weight": (
            selected_candidate[
                "ai_consensus"
            ]
        ),
        "ai_prediction_vote_weight": (
            selected_candidate[
                "ai_prediction_vote"
            ]
        ),
        "uncertainty_weight": (
            selected_candidate[
                "uncertainty"
            ]
        ),
        "inner_validation_f1": float(
            selected_row[
                "mean_validation_f1"
            ]
        ),
        "inner_validation_fpr": float(
            selected_row[
                "mean_validation_fpr"
            ]
        ),
        "inner_ai_f1": float(
            selected_row[
                "mean_ai_validation_f1"
            ]
        ),
        "inner_ai_fpr": float(
            selected_row[
                "mean_ai_validation_fpr"
            ]
        ),
        "maximum_allowed_fpr": float(
            selected_row[
                "maximum_allowed_fpr"
            ]
        ),
        "fpr_constraint_satisfied": bool(
            selected_row[
                "fpr_constraint_satisfied"
            ]
        ),
    })

    print(
        f"Selected {selected_candidate_id}, "
        f"threshold={selected_threshold:.3f}, "
        f"mode={selection_mode}"
    )


    # ------------------------------------------------------------
    # Evaluate selected configuration on untouched outer fold
    # ------------------------------------------------------------
    outer_hybrid_score = (
        build_candidate_score(
            outer_test_df,
            selected_candidate,
        )
    )

    outer_hybrid_prediction = (
        outer_hybrid_score
        >= selected_threshold
    ).astype(int)

    outer_ai_score = (
        outer_test_df[
            "ai_consensus_score"
        ].to_numpy()
    )

    outer_ai_prediction = (
        outer_test_df[
            "ai_prediction_vote"
        ].to_numpy()
        >= 0.5
    ).astype(int)

    outer_v1_score = (
        outer_test_df[
            "hybrid_risk_score"
        ].to_numpy()
    )

    outer_v1_prediction = (
        outer_test_df[
            "hybrid_anomaly_prediction"
        ].to_numpy()
    )

    method_definitions = [
        (
            "HYBRID_V3_NESTED",
            outer_hybrid_score,
            outer_hybrid_prediction,
        ),
        (
            "AI_CONSENSUS",
            outer_ai_score,
            outer_ai_prediction,
        ),
        (
            "HYBRID_V1",
            outer_v1_score,
            outer_v1_prediction,
        ),
    ]

    for (
        method_name,
        score_values,
        prediction_values,
    ) in method_definitions:
        evaluation = (
            calculate_macro_experiment_metrics(
                source_df=outer_test_df,
                score_values=score_values,
                prediction_values=(
                    prediction_values
                ),
            )
        )

        outer_result_rows.append({
            "outer_fold": outer_fold,
            "method": method_name,
            "experiments": (
                outer_test_df[
                    "experiment_id"
                ].nunique()
            ),
            "records": len(
                outer_test_df
            ),
            "macro_precision": (
                evaluation[
                    "macro_precision"
                ]
            ),
            "macro_recall": (
                evaluation[
                    "macro_recall"
                ]
            ),
            "macro_f1": (
                evaluation[
                    "macro_f1"
                ]
            ),
            "macro_fpr": (
                evaluation[
                    "macro_fpr"
                ]
            ),
            "macro_fnr": (
                evaluation[
                    "macro_fnr"
                ]
            ),
            "macro_balanced_accuracy": (
                evaluation[
                    "macro_balanced_accuracy"
                ]
            ),
            "macro_pr_auc": (
                evaluation[
                    "macro_pr_auc"
                ]
            ),
        })

    fold_prediction_df = outer_test_df[
        [
            "experiment_id",
            "dataset",
            "record_id",
            "ground_truth_label",
        ]
    ].copy()

    fold_prediction_df[
        "outer_fold"
    ] = outer_fold

    fold_prediction_df[
        "hybrid_v3_risk_score"
    ] = outer_hybrid_score

    fold_prediction_df[
        "hybrid_v3_prediction"
    ] = outer_hybrid_prediction

    fold_prediction_df[
        "selected_candidate_id"
    ] = selected_candidate_id

    fold_prediction_df[
        "selected_threshold"
    ] = selected_threshold

    outer_record_predictions.append(
        fold_prediction_df
    )


# ----------------------------------------------------------------
# Combine nested-CV outputs
# ----------------------------------------------------------------
outer_results_df = pd.DataFrame(
    outer_result_rows
)

selected_configurations_df = pd.DataFrame(
    selected_configuration_rows
)

inner_fold_search_df = pd.DataFrame(
    inner_search_rows
)

nested_record_predictions_df = pd.concat(
    outer_record_predictions,
    ignore_index=True,
)


# ----------------------------------------------------------------
# Nested-CV publication summary
# ----------------------------------------------------------------
nested_cv_summary_df = (
    outer_results_df
    .groupby(
        "method",
        dropna=False,
    )
    .agg(
        outer_folds=(
            "outer_fold",
            "nunique",
        ),
        mean_macro_precision=(
            "macro_precision",
            "mean",
        ),
        std_macro_precision=(
            "macro_precision",
            "std",
        ),
        mean_macro_recall=(
            "macro_recall",
            "mean",
        ),
        std_macro_recall=(
            "macro_recall",
            "std",
        ),
        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),
        std_macro_f1=(
            "macro_f1",
            "std",
        ),
        mean_macro_fpr=(
            "macro_fpr",
            "mean",
        ),
        std_macro_fpr=(
            "macro_fpr",
            "std",
        ),
        mean_macro_fnr=(
            "macro_fnr",
            "mean",
        ),
        mean_macro_balanced_accuracy=(
            "macro_balanced_accuracy",
            "mean",
        ),
        mean_macro_pr_auc=(
            "macro_pr_auc",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "mean_macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(nested_cv_summary_df)


# ----------------------------------------------------------------
# Show fold-level outcomes and selected configurations
# ----------------------------------------------------------------
display(
    outer_results_df.sort_values(
        [
            "outer_fold",
            "method",
        ]
    )
)

display(
    selected_configurations_df.sort_values(
        "outer_fold"
    )
)


# ----------------------------------------------------------------
# Dataset-specific out-of-fold evaluation
# ----------------------------------------------------------------
dataset_oof_rows = []

for dataset_name, dataset_df in (
    nested_record_predictions_df.groupby(
        "dataset",
        sort=True,
    )
):
    metrics = calculate_macro_experiment_metrics(
        source_df=dataset_df,
        score_values=dataset_df[
            "hybrid_v3_risk_score"
        ].to_numpy(),
        prediction_values=dataset_df[
            "hybrid_v3_prediction"
        ].to_numpy(),
    )

    dataset_oof_rows.append({
        "dataset": dataset_name,
        "experiments": (
            dataset_df[
                "experiment_id"
            ].nunique()
        ),
        "records": len(
            dataset_df
        ),
        "macro_precision": (
            metrics[
                "macro_precision"
            ]
        ),
        "macro_recall": (
            metrics[
                "macro_recall"
            ]
        ),
        "macro_f1": (
            metrics[
                "macro_f1"
            ]
        ),
        "macro_fpr": (
            metrics[
                "macro_fpr"
            ]
        ),
        "macro_fnr": (
            metrics[
                "macro_fnr"
            ]
        ),
        "macro_balanced_accuracy": (
            metrics[
                "macro_balanced_accuracy"
            ]
        ),
        "macro_pr_auc": (
            metrics[
                "macro_pr_auc"
            ]
        ),
    })

dataset_oof_summary_df = pd.DataFrame(
    dataset_oof_rows
)

display(dataset_oof_summary_df)


# ----------------------------------------------------------------
# Diagnostics
# ----------------------------------------------------------------
v3_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "HYBRID_V3_NESTED"
        )
    ]
    .iloc[0]
)

ai_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "AI_CONSENSUS"
        )
    ]
    .iloc[0]
)

v1_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "HYBRID_V1"
        )
    ]
    .iloc[0]
)

nested_diagnostics = {
    "v3_f1_above_v1": (
        v3_row[
            "mean_macro_f1"
        ]
        >
        v1_row[
            "mean_macro_f1"
        ]
    ),
    "v3_f1_above_ai_consensus": (
        v3_row[
            "mean_macro_f1"
        ]
        >
        ai_row[
            "mean_macro_f1"
        ]
    ),
    "v3_fpr_below_v1": (
        v3_row[
            "mean_macro_fpr"
        ]
        <
        v1_row[
            "mean_macro_fpr"
        ]
    ),
    "v3_fpr_within_ai_tolerance": (
        v3_row[
            "mean_macro_fpr"
        ]
        <= (
            ai_row[
                "mean_macro_fpr"
            ]
            + FPR_TOLERANCE_ABOVE_AI
        )
    ),
    "all_records_received_oof_prediction": (
        len(
            nested_record_predictions_df
        )
        == len(
            v3_source_df
        )
    ),
    "one_oof_prediction_per_record": (
        not nested_record_predictions_df
        .duplicated(
            subset=[
                "experiment_id",
                "dataset",
                "record_id",
            ]
        )
        .any()
    ),
}

nested_diagnostics_df = pd.DataFrame({
    "diagnostic": nested_diagnostics.keys(),
    "passed": nested_diagnostics.values(),
})

display(nested_diagnostics_df)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
validation_checks = {
    "three_outer_folds_completed": (
        outer_results_df[
            "outer_fold"
        ].nunique()
        == OUTER_FOLDS
    ),
    "three_methods_evaluated": (
        outer_results_df[
            "method"
        ].nunique()
        == 3
    ),
    "all_63_experiments_oof": (
        nested_record_predictions_df[
            "experiment_id"
        ].nunique()
        == 63
    ),
    "all_three_datasets_oof": (
        nested_record_predictions_df[
            "dataset"
        ].nunique()
        == 3
    ),
    "scores_in_valid_range": (
        nested_record_predictions_df[
            "hybrid_v3_risk_score"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),
    "predictions_binary": (
        set(
            nested_record_predictions_df[
                "hybrid_v3_prediction"
            ]
            .unique()
        )
        .issubset(
            {0, 1}
        )
    ),
    "one_configuration_per_outer_fold": (
        len(
            selected_configurations_df
        )
        == OUTER_FOLDS
    ),
}

nested_validation_df = pd.DataFrame({
    "check": validation_checks.keys(),
    "passed": validation_checks.values(),
})

display(nested_validation_df)

assert nested_validation_df[
    "passed"
].all(), (
    "One or more nested-CV validation checks failed."
)


# ----------------------------------------------------------------
# Save artifacts
# ----------------------------------------------------------------
NESTED_OUTER_RESULTS_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_nested_outer_fold_results.csv"
)

NESTED_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_nested_cv_summary.csv"
)

NESTED_CONFIGURATIONS_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_selected_configurations.csv"
)

NESTED_INNER_SEARCH_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_inner_fold_search.csv"
)

NESTED_OOF_PREDICTIONS_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_out_of_fold_predictions.parquet"
)

NESTED_DATASET_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_dataset_oof_summary.csv"
)

outer_results_df.to_csv(
    NESTED_OUTER_RESULTS_PATH,
    index=False,
)

nested_cv_summary_df.to_csv(
    NESTED_SUMMARY_PATH,
    index=False,
)

selected_configurations_df.to_csv(
    NESTED_CONFIGURATIONS_PATH,
    index=False,
)

inner_fold_search_df.to_csv(
    NESTED_INNER_SEARCH_PATH,
    index=False,
)

nested_record_predictions_df.to_parquet(
    NESTED_OOF_PREDICTIONS_PATH,
    index=False,
)

dataset_oof_summary_df.to_csv(
    NESTED_DATASET_SUMMARY_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Save audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

nested_cv_audit = {
    "hybrid_run_id": HYBRID_RUN_ID,
    "version": V3_VERSION,
    "execution_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "random_seed": RANDOM_SEED,
    "fpr_tolerance_above_ai": (
        FPR_TOLERANCE_ABOVE_AI
    ),
    "candidate_weight_count": len(
        CANDIDATE_WEIGHTS
    ),
    "candidate_threshold_count": len(
        CANDIDATE_THRESHOLDS
    ),
    "experiments": int(
        experiment_table_df[
            "experiment_id"
        ].nunique()
    ),
    "records": int(
        len(v3_source_df)
    ),
    "v3_mean_macro_f1": float(
        v3_row[
            "mean_macro_f1"
        ]
    ),
    "v3_mean_macro_fpr": float(
        v3_row[
            "mean_macro_fpr"
        ]
    ),
    "ai_mean_macro_f1": float(
        ai_row[
            "mean_macro_f1"
        ]
    ),
    "ai_mean_macro_fpr": float(
        ai_row[
            "mean_macro_fpr"
        ]
    ),
    "v1_mean_macro_f1": float(
        v1_row[
            "mean_macro_f1"
        ]
    ),
    "v1_mean_macro_fpr": float(
        v1_row[
            "mean_macro_fpr"
        ]
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "summary_path": str(
        NESTED_SUMMARY_PATH
    ),
    "oof_predictions_path": str(
        NESTED_OOF_PREDICTIONS_PATH
    ),
}

NESTED_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "hybrid_v3_nested_cv_audit.json"
)

with open(
    NESTED_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        nested_cv_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Nested constrained Hybrid V3 evaluation completed."
)
print(
    f"Version: {V3_VERSION}"
)
print(
    f"Outer folds: {OUTER_FOLDS}"
)
print(
    f"Inner folds: {INNER_FOLDS}"
)
print(
    f"Experiments evaluated out-of-fold: "
    f"{nested_record_predictions_df['experiment_id'].nunique()}"
)
print(
    f"V3 mean macro-F1: "
    f"{v3_row['mean_macro_f1']:.4f}"
)
print(
    f"V3 mean macro-FPR: "
    f"{v3_row['mean_macro_fpr']:.4f}"
)
print(
    f"AI-consensus mean macro-F1: "
    f"{ai_row['mean_macro_f1']:.4f}"
)
print(
    f"AI-consensus mean macro-FPR: "
    f"{ai_row['mean_macro_fpr']:.4f}"
)
print(
    f"Nested-CV summary: "
    f"{NESTED_SUMMARY_PATH}"
)
print(
    f"Out-of-fold predictions: "
    f"{NESTED_OOF_PREDICTIONS_PATH}"
)
print(
    f"Audit: "
    f"{NESTED_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

## 8. Statistical and Operational Analysis

This section performs paired statistical comparisons, multiple-comparison correction, omnibus testing, and operational burden analysis.


In [ ]:
# ================================================================
# Cell 8 — Statistical comparison and operational trade-off
#          analysis using out-of-fold experiment results
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from scipy.stats import (
    friedmanchisquare,
    wilcoxon,
)


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()

STATISTICAL_ANALYSIS_VERSION = (
    "HYBRID_STATISTICAL_ANALYSIS_V1.0"
)

BOOTSTRAP_ITERATIONS = 10_000
BOOTSTRAP_RANDOM_SEED = 42
SIGNIFICANCE_LEVEL = 0.05


# ----------------------------------------------------------------
# Reconstruct experiment-level out-of-fold metrics
# ----------------------------------------------------------------
# The outer-fold results shown previously are fold-level macro
# summaries. Statistical paired comparisons require one observation
# per experiment and method.
#
# Hybrid V3 predictions are available in
# nested_record_predictions_df. AI-consensus and Hybrid V1 fields
# are merged back from hybrid_decision_df.

required_oof_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ground_truth_label",
    "outer_fold",
    "hybrid_v3_risk_score",
    "hybrid_v3_prediction",
]

missing_oof_columns = [
    column
    for column in required_oof_columns
    if column not in nested_record_predictions_df.columns
]

if missing_oof_columns:
    raise KeyError(
        "Missing out-of-fold fields: "
        + ", ".join(
            missing_oof_columns
        )
    )


baseline_merge_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ai_consensus_score",
    "ai_prediction_vote",
    "hybrid_risk_score",
    "hybrid_anomaly_prediction",
    "rule_risk_score",
    "rule_anomaly_prediction",
]

missing_baseline_columns = [
    column
    for column in baseline_merge_columns
    if column not in hybrid_decision_df.columns
]

if missing_baseline_columns:
    raise KeyError(
        "Missing baseline fields: "
        + ", ".join(
            missing_baseline_columns
        )
    )


comparison_oof_df = (
    nested_record_predictions_df
    .merge(
        hybrid_decision_df[
            baseline_merge_columns
        ],
        on=[
            "experiment_id",
            "dataset",
            "record_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ----------------------------------------------------------------
# Create AI-consensus prediction
# ----------------------------------------------------------------
comparison_oof_df[
    "ai_consensus_prediction"
] = (
    comparison_oof_df[
        "ai_prediction_vote"
    ]
    >= 0.5
).astype(int)


# ----------------------------------------------------------------
# Validate merge completeness
# ----------------------------------------------------------------
merge_validation_columns = [
    "ai_consensus_score",
    "ai_consensus_prediction",
    "hybrid_risk_score",
    "hybrid_anomaly_prediction",
    "rule_risk_score",
    "rule_anomaly_prediction",
]

if comparison_oof_df[
    merge_validation_columns
].isna().any().any():
    missing_counts = (
        comparison_oof_df[
            merge_validation_columns
        ]
        .isna()
        .sum()
    )

    raise ValueError(
        "Missing values after OOF-baseline merge:\n"
        + missing_counts.to_string()
    )


# ----------------------------------------------------------------
# Metric helper
# ----------------------------------------------------------------
def calculate_experiment_metrics(
    y_true,
    y_prediction,
    y_score,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_prediction = np.asarray(
        y_prediction,
        dtype=int,
    )

    y_score = np.asarray(
        y_score,
        dtype=float,
    )

    true_positive = int(
        np.sum(
            (y_true == 1)
            & (y_prediction == 1)
        )
    )

    true_negative = int(
        np.sum(
            (y_true == 0)
            & (y_prediction == 0)
        )
    )

    false_positive = int(
        np.sum(
            (y_true == 0)
            & (y_prediction == 1)
        )
    )

    false_negative = int(
        np.sum(
            (y_true == 1)
            & (y_prediction == 0)
        )
    )

    precision_denominator = (
        true_positive
        + false_positive
    )

    recall_denominator = (
        true_positive
        + false_negative
    )

    specificity_denominator = (
        true_negative
        + false_positive
    )

    precision = (
        true_positive
        / precision_denominator
        if precision_denominator > 0
        else 0.0
    )

    recall = (
        true_positive
        / recall_denominator
        if recall_denominator > 0
        else 0.0
    )

    specificity = (
        true_negative
        / specificity_denominator
        if specificity_denominator > 0
        else 0.0
    )

    false_positive_rate = (
        false_positive
        / specificity_denominator
        if specificity_denominator > 0
        else 0.0
    )

    false_negative_rate = (
        false_negative
        / recall_denominator
        if recall_denominator > 0
        else 0.0
    )

    f1_denominator = (
        precision
        + recall
    )

    f1 = (
        2.0
        * precision
        * recall
        / f1_denominator
        if f1_denominator > 0
        else 0.0
    )

    balanced_accuracy = (
        recall
        + specificity
    ) / 2.0

    return {
        "records": int(
            len(y_true)
        ),
        "true_anomalies": int(
            y_true.sum()
        ),
        "predicted_anomalies": int(
            y_prediction.sum()
        ),
        "tp": true_positive,
        "tn": true_negative,
        "fp": false_positive,
        "fn": false_negative,
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "false_negative_rate": float(
            false_negative_rate
        ),
        "specificity": float(
            specificity
        ),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
    }


# ----------------------------------------------------------------
# Method definitions
# ----------------------------------------------------------------
STATISTICAL_METHODS = {
    "AI_CONSENSUS": {
        "score_column": (
            "ai_consensus_score"
        ),
        "prediction_column": (
            "ai_consensus_prediction"
        ),
    },
    "HYBRID_V1": {
        "score_column": (
            "hybrid_risk_score"
        ),
        "prediction_column": (
            "hybrid_anomaly_prediction"
        ),
    },
    "HYBRID_V3_NESTED": {
        "score_column": (
            "hybrid_v3_risk_score"
        ),
        "prediction_column": (
            "hybrid_v3_prediction"
        ),
    },
    "RULE_ONLY": {
        "score_column": (
            "rule_risk_score"
        ),
        "prediction_column": (
            "rule_anomaly_prediction"
        ),
    },
}


# ----------------------------------------------------------------
# Build experiment-level metric table
# ----------------------------------------------------------------
experiment_metric_rows = []

for (
    experiment_id,
    dataset,
), experiment_df in comparison_oof_df.groupby(
    [
        "experiment_id",
        "dataset",
    ],
    sort=True,
):
    outer_fold = int(
        experiment_df[
            "outer_fold"
        ].iloc[0]
    )

    for method_name, method_fields in (
        STATISTICAL_METHODS.items()
    ):
        metrics = calculate_experiment_metrics(
            y_true=experiment_df[
                "ground_truth_label"
            ],
            y_prediction=experiment_df[
                method_fields[
                    "prediction_column"
                ]
            ],
            y_score=experiment_df[
                method_fields[
                    "score_column"
                ]
            ],
        )

        metrics.update({
            "experiment_id": experiment_id,
            "dataset": dataset,
            "outer_fold": outer_fold,
            "method": method_name,
        })

        experiment_metric_rows.append(
            metrics
        )


statistical_experiment_metrics_df = (
    pd.DataFrame(
        experiment_metric_rows
    )
)


# ----------------------------------------------------------------
# Validation of paired experiment structure
# ----------------------------------------------------------------
method_count_per_experiment = (
    statistical_experiment_metrics_df
    .groupby(
        "experiment_id"
    )[
        "method"
    ]
    .nunique()
)

if not (
    method_count_per_experiment
    == len(
        STATISTICAL_METHODS
    )
).all():
    raise ValueError(
        "Not every experiment contains all statistical methods."
    )


# ----------------------------------------------------------------
# Pivot metrics for paired tests
# ----------------------------------------------------------------
metric_names = [
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "false_negative_rate",
    "balanced_accuracy",
]

metric_pivots = {}

for metric_name in metric_names:
    metric_pivots[
        metric_name
    ] = (
        statistical_experiment_metrics_df
        .pivot(
            index=[
                "experiment_id",
                "dataset",
            ],
            columns="method",
            values=metric_name,
        )
        .reset_index()
    )


# ----------------------------------------------------------------
# Paired bootstrap confidence interval helper
# ----------------------------------------------------------------
def paired_bootstrap_difference(
    values_a,
    values_b,
    iterations=BOOTSTRAP_ITERATIONS,
    random_seed=BOOTSTRAP_RANDOM_SEED,
) -> dict:
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    if len(values_a) != len(values_b):
        raise ValueError(
            "Paired bootstrap arrays must have equal length."
        )

    observed_differences = (
        values_a
        - values_b
    )

    observed_mean_difference = float(
        observed_differences.mean()
    )

    random_generator = (
        np.random.default_rng(
            random_seed
        )
    )

    sample_count = len(
        observed_differences
    )

    bootstrap_means = np.empty(
        iterations,
        dtype=float,
    )

    for iteration in range(
        iterations
    ):
        sampled_indices = (
            random_generator.integers(
                low=0,
                high=sample_count,
                size=sample_count,
            )
        )

        bootstrap_means[
            iteration
        ] = (
            observed_differences[
                sampled_indices
            ]
            .mean()
        )

    confidence_interval_lower = float(
        np.percentile(
            bootstrap_means,
            2.5,
        )
    )

    confidence_interval_upper = float(
        np.percentile(
            bootstrap_means,
            97.5,
        )
    )

    probability_difference_above_zero = float(
        np.mean(
            bootstrap_means > 0
        )
    )

    return {
        "mean_difference": (
            observed_mean_difference
        ),
        "ci_95_lower": (
            confidence_interval_lower
        ),
        "ci_95_upper": (
            confidence_interval_upper
        ),
        "bootstrap_probability_above_zero": (
            probability_difference_above_zero
        ),
    }


# ----------------------------------------------------------------
# Wilcoxon helper with zero-difference handling
# ----------------------------------------------------------------
def safe_wilcoxon(
    values_a,
    values_b,
) -> dict:
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    differences = (
        values_a
        - values_b
    )

    nonzero_differences = (
        differences[
            differences != 0
        ]
    )

    if len(
        nonzero_differences
    ) == 0:
        return {
            "wilcoxon_statistic": 0.0,
            "wilcoxon_p_value": 1.0,
            "nonzero_pairs": 0,
        }

    test_result = wilcoxon(
        values_a,
        values_b,
        alternative="two-sided",
        zero_method="wilcox",
    )

    return {
        "wilcoxon_statistic": float(
            test_result.statistic
        ),
        "wilcoxon_p_value": float(
            test_result.pvalue
        ),
        "nonzero_pairs": int(
            len(
                nonzero_differences
            )
        ),
    }


# ----------------------------------------------------------------
# Pairwise comparisons
# ----------------------------------------------------------------
PAIRWISE_COMPARISONS = [
    (
        "HYBRID_V3_NESTED",
        "AI_CONSENSUS",
    ),
    (
        "HYBRID_V3_NESTED",
        "HYBRID_V1",
    ),
    (
        "HYBRID_V3_NESTED",
        "RULE_ONLY",
    ),
    (
        "AI_CONSENSUS",
        "HYBRID_V1",
    ),
]

pairwise_result_rows = []

for metric_name in metric_names:
    pivot_df = (
        metric_pivots[
            metric_name
        ]
    )

    for (
        method_a,
        method_b,
    ) in PAIRWISE_COMPARISONS:
        values_a = (
            pivot_df[
                method_a
            ]
            .to_numpy()
        )

        values_b = (
            pivot_df[
                method_b
            ]
            .to_numpy()
        )

        bootstrap_result = (
            paired_bootstrap_difference(
                values_a=values_a,
                values_b=values_b,
                random_seed=(
                    BOOTSTRAP_RANDOM_SEED
                    + len(
                        pairwise_result_rows
                    )
                ),
            )
        )

        wilcoxon_result = (
            safe_wilcoxon(
                values_a=values_a,
                values_b=values_b,
            )
        )

        pairwise_result_rows.append({
            "metric": metric_name,
            "method_a": method_a,
            "method_b": method_b,
            "experiments": int(
                len(values_a)
            ),
            "method_a_mean": float(
                values_a.mean()
            ),
            "method_b_mean": float(
                values_b.mean()
            ),
            **bootstrap_result,
            **wilcoxon_result,
        })


pairwise_statistical_tests_df = (
    pd.DataFrame(
        pairwise_result_rows
    )
)


# ----------------------------------------------------------------
# Multiple-testing correction using Holm's method
# ----------------------------------------------------------------
def holm_adjust(
    p_values,
) -> np.ndarray:
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    test_count = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    sorted_p_values = (
        p_values[
            order
        ]
    )

    adjusted_sorted = np.empty(
        test_count,
        dtype=float,
    )

    running_maximum = 0.0

    for rank, p_value in enumerate(
        sorted_p_values
    ):
        multiplier = (
            test_count
            - rank
        )

        adjusted_value = min(
            1.0,
            p_value
            * multiplier,
        )

        running_maximum = max(
            running_maximum,
            adjusted_value,
        )

        adjusted_sorted[
            rank
        ] = running_maximum

    adjusted = np.empty(
        test_count,
        dtype=float,
    )

    adjusted[
        order
    ] = adjusted_sorted

    return adjusted


pairwise_statistical_tests_df[
    "holm_adjusted_p_value"
] = holm_adjust(
    pairwise_statistical_tests_df[
        "wilcoxon_p_value"
    ]
    .to_numpy()
)

pairwise_statistical_tests_df[
    "statistically_significant"
] = (
    pairwise_statistical_tests_df[
        "holm_adjusted_p_value"
    ]
    < SIGNIFICANCE_LEVEL
)


# ----------------------------------------------------------------
# Friedman omnibus tests
# ----------------------------------------------------------------
friedman_rows = []

for metric_name in metric_names:
    pivot_df = (
        metric_pivots[
            metric_name
        ]
    )

    method_arrays = [
        pivot_df[
            method_name
        ]
        .to_numpy()
        for method_name
        in STATISTICAL_METHODS.keys()
    ]

    friedman_result = (
        friedmanchisquare(
            *method_arrays
        )
    )

    friedman_rows.append({
        "metric": metric_name,
        "methods_compared": int(
            len(
                STATISTICAL_METHODS
            )
        ),
        "experiments": int(
            len(
                pivot_df
            )
        ),
        "friedman_statistic": float(
            friedman_result.statistic
        ),
        "friedman_p_value": float(
            friedman_result.pvalue
        ),
        "statistically_significant": (
            friedman_result.pvalue
            < SIGNIFICANCE_LEVEL
        ),
    })


friedman_tests_df = pd.DataFrame(
    friedman_rows
)


# ----------------------------------------------------------------
# Experiment-level win, tie, and loss analysis
# ----------------------------------------------------------------
f1_pivot_df = (
    metric_pivots[
        "f1"
    ]
)

fpr_pivot_df = (
    metric_pivots[
        "false_positive_rate"
    ]
)

win_loss_rows = []

for comparison_method in [
    "AI_CONSENSUS",
    "HYBRID_V1",
    "RULE_ONLY",
]:
    f1_difference = (
        f1_pivot_df[
            "HYBRID_V3_NESTED"
        ]
        - f1_pivot_df[
            comparison_method
        ]
    )

    fpr_difference = (
        fpr_pivot_df[
            "HYBRID_V3_NESTED"
        ]
        - fpr_pivot_df[
            comparison_method
        ]
    )

    win_loss_rows.append({
        "comparison": (
            "HYBRID_V3_NESTED vs "
            + comparison_method
        ),
        "f1_wins": int(
            np.sum(
                f1_difference > 0
            )
        ),
        "f1_ties": int(
            np.sum(
                f1_difference == 0
            )
        ),
        "f1_losses": int(
            np.sum(
                f1_difference < 0
            )
        ),
        "lower_fpr_wins": int(
            np.sum(
                fpr_difference < 0
            )
        ),
        "equal_fpr_ties": int(
            np.sum(
                fpr_difference == 0
            )
        ),
        "higher_fpr_losses": int(
            np.sum(
                fpr_difference > 0
            )
        ),
        "mean_f1_difference": float(
            f1_difference.mean()
        ),
        "mean_fpr_difference": float(
            fpr_difference.mean()
        ),
    })


experiment_win_loss_df = pd.DataFrame(
    win_loss_rows
)


# ----------------------------------------------------------------
# Dataset-level paired differences
# ----------------------------------------------------------------
dataset_difference_rows = []

for dataset_name in sorted(
    comparison_oof_df[
        "dataset"
    ].unique()
):
    dataset_f1_pivot = (
        f1_pivot_df.loc[
            f1_pivot_df[
                "dataset"
            ].eq(
                dataset_name
            )
        ]
    )

    dataset_fpr_pivot = (
        fpr_pivot_df.loc[
            fpr_pivot_df[
                "dataset"
            ].eq(
                dataset_name
            )
        ]
    )

    for comparison_method in [
        "AI_CONSENSUS",
        "HYBRID_V1",
        "RULE_ONLY",
    ]:
        dataset_difference_rows.append({
            "dataset": dataset_name,
            "comparison_method": (
                comparison_method
            ),
            "experiments": int(
                len(
                    dataset_f1_pivot
                )
            ),
            "v3_mean_f1": float(
                dataset_f1_pivot[
                    "HYBRID_V3_NESTED"
                ].mean()
            ),
            "comparison_mean_f1": float(
                dataset_f1_pivot[
                    comparison_method
                ].mean()
            ),
            "mean_f1_difference": float(
                (
                    dataset_f1_pivot[
                        "HYBRID_V3_NESTED"
                    ]
                    - dataset_f1_pivot[
                        comparison_method
                    ]
                ).mean()
            ),
            "v3_mean_fpr": float(
                dataset_fpr_pivot[
                    "HYBRID_V3_NESTED"
                ].mean()
            ),
            "comparison_mean_fpr": float(
                dataset_fpr_pivot[
                    comparison_method
                ].mean()
            ),
            "mean_fpr_difference": float(
                (
                    dataset_fpr_pivot[
                        "HYBRID_V3_NESTED"
                    ]
                    - dataset_fpr_pivot[
                        comparison_method
                    ]
                ).mean()
            ),
        })


dataset_paired_difference_df = (
    pd.DataFrame(
        dataset_difference_rows
    )
)


# ----------------------------------------------------------------
# Operational burden analysis
# ----------------------------------------------------------------
# A flagged record represents a potential intervention:
# repair, quarantine, escalation, or review.
#
# Lower review burden is desirable only when anomaly capture remains
# acceptable. This table makes that trade-off explicit.

operational_rows = []

for method_name, fields in (
    STATISTICAL_METHODS.items()
):
    prediction_column = (
        fields[
            "prediction_column"
        ]
    )

    total_records = int(
        len(
            comparison_oof_df
        )
    )

    total_true_anomalies = int(
        comparison_oof_df[
            "ground_truth_label"
        ].sum()
    )

    flagged_records = int(
        comparison_oof_df[
            prediction_column
        ].sum()
    )

    true_positive_records = int(
        (
            (
                comparison_oof_df[
                    prediction_column
                ]
                == 1
            )
            & (
                comparison_oof_df[
                    "ground_truth_label"
                ]
                == 1
            )
        ).sum()
    )

    false_positive_records = int(
        (
            (
                comparison_oof_df[
                    prediction_column
                ]
                == 1
            )
            & (
                comparison_oof_df[
                    "ground_truth_label"
                ]
                == 0
            )
        ).sum()
    )

    missed_anomalies = int(
        (
            (
                comparison_oof_df[
                    prediction_column
                ]
                == 0
            )
            & (
                comparison_oof_df[
                    "ground_truth_label"
                ]
                == 1
            )
        ).sum()
    )

    operational_rows.append({
        "method": method_name,
        "total_records": (
            total_records
        ),
        "total_true_anomalies": (
            total_true_anomalies
        ),
        "flagged_records": (
            flagged_records
        ),
        "flagged_record_rate": (
            flagged_records
            / total_records
        ),
        "true_positive_records": (
            true_positive_records
        ),
        "false_positive_records": (
            false_positive_records
        ),
        "missed_anomalies": (
            missed_anomalies
        ),
        "anomalies_captured_rate": (
            true_positive_records
            / total_true_anomalies
            if total_true_anomalies > 0
            else 0.0
        ),
        "false_reviews_per_true_positive": (
            false_positive_records
            / true_positive_records
            if true_positive_records > 0
            else np.nan
        ),
        "records_flagged_per_true_positive": (
            flagged_records
            / true_positive_records
            if true_positive_records > 0
            else np.nan
        ),
    })


operational_burden_df = (
    pd.DataFrame(
        operational_rows
    )
    .sort_values(
        "false_reviews_per_true_positive",
        ascending=True,
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------------
# Display key outputs
# ----------------------------------------------------------------
print(
    "Paired statistical tests:"
)

display(
    pairwise_statistical_tests_df[
        [
            "metric",
            "method_a",
            "method_b",
            "experiments",
            "method_a_mean",
            "method_b_mean",
            "mean_difference",
            "ci_95_lower",
            "ci_95_upper",
            "wilcoxon_p_value",
            "holm_adjusted_p_value",
            "statistically_significant",
        ]
    ]
    .sort_values(
        [
            "metric",
            "method_a",
            "method_b",
        ]
    )
)


print(
    "Friedman omnibus tests:"
)

display(
    friedman_tests_df
)


print(
    "Experiment-level win/loss analysis:"
)

display(
    experiment_win_loss_df
)


print(
    "Dataset-level paired differences:"
)

display(
    dataset_paired_difference_df
)


print(
    "Operational intervention burden:"
)

display(
    operational_burden_df
)


# ----------------------------------------------------------------
# Research-claim diagnostics
# ----------------------------------------------------------------
v3_vs_ai_f1_row = (
    pairwise_statistical_tests_df.loc[
        (
            pairwise_statistical_tests_df[
                "metric"
            ].eq(
                "f1"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_a"
            ].eq(
                "HYBRID_V3_NESTED"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_b"
            ].eq(
                "AI_CONSENSUS"
            )
        )
    ]
    .iloc[0]
)

v3_vs_v1_fpr_row = (
    pairwise_statistical_tests_df.loc[
        (
            pairwise_statistical_tests_df[
                "metric"
            ].eq(
                "false_positive_rate"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_a"
            ].eq(
                "HYBRID_V3_NESTED"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_b"
            ].eq(
                "HYBRID_V1"
            )
        )
    ]
    .iloc[0]
)

research_claim_diagnostics = {
    "v3_f1_numerically_above_ai": (
        v3_vs_ai_f1_row[
            "mean_difference"
        ]
        > 0
    ),
    "v3_f1_significantly_differs_from_ai": bool(
        v3_vs_ai_f1_row[
            "statistically_significant"
        ]
    ),
    "v3_fpr_below_v1": (
        v3_vs_v1_fpr_row[
            "mean_difference"
        ]
        < 0
    ),
    "v3_fpr_reduction_vs_v1_significant": bool(
        v3_vs_v1_fpr_row[
            "statistically_significant"
        ]
    ),
    "all_63_experiments_in_paired_tests": (
        f1_pivot_df[
            "experiment_id"
        ].nunique()
        == 63
    ),
}

research_claim_diagnostics_df = pd.DataFrame({
    "diagnostic": (
        research_claim_diagnostics.keys()
    ),
    "passed": (
        research_claim_diagnostics.values()
    ),
})

display(
    research_claim_diagnostics_df
)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
expected_metric_rows = (
    63
    * len(
        STATISTICAL_METHODS
    )
)

validation_checks = {
    "expected_experiment_metric_rows": (
        len(
            statistical_experiment_metrics_df
        )
        == expected_metric_rows
    ),
    "all_63_experiments_present": (
        statistical_experiment_metrics_df[
            "experiment_id"
        ].nunique()
        == 63
    ),
    "all_methods_present": (
        statistical_experiment_metrics_df[
            "method"
        ].nunique()
        == len(
            STATISTICAL_METHODS
        )
    ),
    "pairwise_tests_created": (
        len(
            pairwise_statistical_tests_df
        )
        == (
            len(
                metric_names
            )
            * len(
                PAIRWISE_COMPARISONS
            )
        )
    ),
    "adjusted_p_values_valid": (
        pairwise_statistical_tests_df[
            "holm_adjusted_p_value"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),
    "bootstrap_intervals_valid": (
        (
            pairwise_statistical_tests_df[
                "ci_95_lower"
            ]
            <= pairwise_statistical_tests_df[
                "ci_95_upper"
            ]
        )
        .all()
    ),
    "four_operational_methods_present": (
        operational_burden_df[
            "method"
        ].nunique()
        == 4
    ),
}

statistical_validation_df = pd.DataFrame({
    "check": validation_checks.keys(),
    "passed": validation_checks.values(),
})

display(
    statistical_validation_df
)

assert statistical_validation_df[
    "passed"
].all(), (
    "One or more statistical-analysis checks failed."
)


# ----------------------------------------------------------------
# Save output artifacts
# ----------------------------------------------------------------
STATISTICAL_EXPERIMENT_METRICS_PATH = (
    HYBRID_RESULTS_DIR
    / "statistical_experiment_metrics.csv"
)

PAIRWISE_TESTS_PATH = (
    HYBRID_RESULTS_DIR
    / "paired_statistical_tests.csv"
)

FRIEDMAN_TESTS_PATH = (
    HYBRID_RESULTS_DIR
    / "friedman_omnibus_tests.csv"
)

WIN_LOSS_PATH = (
    HYBRID_RESULTS_DIR
    / "experiment_win_loss_analysis.csv"
)

DATASET_PAIRED_DIFFERENCE_PATH = (
    HYBRID_RESULTS_DIR
    / "dataset_paired_differences.csv"
)

OPERATIONAL_BURDEN_PATH = (
    HYBRID_RESULTS_DIR
    / "operational_intervention_burden.csv"
)

statistical_experiment_metrics_df.to_csv(
    STATISTICAL_EXPERIMENT_METRICS_PATH,
    index=False,
)

pairwise_statistical_tests_df.to_csv(
    PAIRWISE_TESTS_PATH,
    index=False,
)

friedman_tests_df.to_csv(
    FRIEDMAN_TESTS_PATH,
    index=False,
)

experiment_win_loss_df.to_csv(
    WIN_LOSS_PATH,
    index=False,
)

dataset_paired_difference_df.to_csv(
    DATASET_PAIRED_DIFFERENCE_PATH,
    index=False,
)

operational_burden_df.to_csv(
    OPERATIONAL_BURDEN_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Save statistical audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

statistical_audit = {
    "hybrid_run_id": (
        HYBRID_RUN_ID
    ),
    "analysis_version": (
        STATISTICAL_ANALYSIS_VERSION
    ),
    "execution_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "experiments": int(
        statistical_experiment_metrics_df[
            "experiment_id"
        ].nunique()
    ),
    "methods": int(
        statistical_experiment_metrics_df[
            "method"
        ].nunique()
    ),
    "bootstrap_iterations": (
        BOOTSTRAP_ITERATIONS
    ),
    "significance_level": (
        SIGNIFICANCE_LEVEL
    ),
    "multiple_testing_correction": (
        "Holm"
    ),
    "v3_vs_ai_f1_mean_difference": float(
        v3_vs_ai_f1_row[
            "mean_difference"
        ]
    ),
    "v3_vs_ai_f1_ci_lower": float(
        v3_vs_ai_f1_row[
            "ci_95_lower"
        ]
    ),
    "v3_vs_ai_f1_ci_upper": float(
        v3_vs_ai_f1_row[
            "ci_95_upper"
        ]
    ),
    "v3_vs_ai_f1_adjusted_p_value": float(
        v3_vs_ai_f1_row[
            "holm_adjusted_p_value"
        ]
    ),
    "v3_vs_v1_fpr_mean_difference": float(
        v3_vs_v1_fpr_row[
            "mean_difference"
        ]
    ),
    "v3_vs_v1_fpr_adjusted_p_value": float(
        v3_vs_v1_fpr_row[
            "holm_adjusted_p_value"
        ]
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "pairwise_tests_path": str(
        PAIRWISE_TESTS_PATH
    ),
    "operational_burden_path": str(
        OPERATIONAL_BURDEN_PATH
    ),
}

STATISTICAL_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "statistical_analysis_audit.json"
)

with open(
    STATISTICAL_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        statistical_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Statistical and operational analysis completed successfully."
)
print(
    f"Analysis version: "
    f"{STATISTICAL_ANALYSIS_VERSION}"
)
print(
    f"Experiments analyzed: "
    f"{statistical_experiment_metrics_df['experiment_id'].nunique()}"
)
print(
    f"Methods analyzed: "
    f"{statistical_experiment_metrics_df['method'].nunique()}"
)
print(
    f"V3 minus AI-consensus macro-F1 difference: "
    f"{v3_vs_ai_f1_row['mean_difference']:.4f}"
)
print(
    f"V3 versus AI 95% bootstrap CI: "
    f"[{v3_vs_ai_f1_row['ci_95_lower']:.4f}, "
    f"{v3_vs_ai_f1_row['ci_95_upper']:.4f}]"
)
print(
    f"V3 versus AI Holm-adjusted p-value: "
    f"{v3_vs_ai_f1_row['holm_adjusted_p_value']:.6f}"
)
print(
    f"V3 minus V1 FPR difference: "
    f"{v3_vs_v1_fpr_row['mean_difference']:.4f}"
)
print(
    f"Pairwise tests: "
    f"{PAIRWISE_TESTS_PATH}"
)
print(
    f"Operational burden: "
    f"{OPERATIONAL_BURDEN_PATH}"
)
print(
    f"Statistical audit: "
    f"{STATISTICAL_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

## 9. Publication Artifact Generation

This section exports machine-readable tables, statistical summaries, and publication-quality figures directly into repository folders.


In [ ]:
# ================================================================
# Cell 9 — Generate publication-ready tables, figures, and
#          final research-results summary
# ================================================================

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()

PUBLICATION_ARTIFACT_VERSION = (
    "HYBRID_PUBLICATION_ARTIFACTS_V1.0"
)


# ----------------------------------------------------------------
# Publication directories
# ----------------------------------------------------------------
PUBLICATION_DIR = RESULTS_DIR / "publication_artifacts"

PUBLICATION_TABLES_DIR = TABLES_DIR

PUBLICATION_FIGURES_DIR = FIGURES_DIR

PUBLICATION_SUMMARIES_DIR = RESULTS_DIR / "publication_summaries"

for directory in [
    PUBLICATION_DIR,
    PUBLICATION_TABLES_DIR,
    PUBLICATION_FIGURES_DIR,
    PUBLICATION_SUMMARIES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ----------------------------------------------------------------
# Validate required data frames
# ----------------------------------------------------------------
required_dataframes = {
    "nested_cv_summary_df": (
        nested_cv_summary_df
    ),
    "outer_results_df": (
        outer_results_df
    ),
    "selected_configurations_df": (
        selected_configurations_df
    ),
    "dataset_oof_summary_df": (
        dataset_oof_summary_df
    ),
    "pairwise_statistical_tests_df": (
        pairwise_statistical_tests_df
    ),
    "friedman_tests_df": (
        friedman_tests_df
    ),
    "experiment_win_loss_df": (
        experiment_win_loss_df
    ),
    "dataset_paired_difference_df": (
        dataset_paired_difference_df
    ),
    "operational_burden_df": (
        operational_burden_df
    ),
    "statistical_experiment_metrics_df": (
        statistical_experiment_metrics_df
    ),
}

for dataframe_name, dataframe_value in (
    required_dataframes.items()
):
    if not isinstance(
        dataframe_value,
        pd.DataFrame,
    ):
        raise TypeError(
            f"{dataframe_name} is not a pandas DataFrame."
        )

    if dataframe_value.empty:
        raise ValueError(
            f"{dataframe_name} is empty."
        )


# ----------------------------------------------------------------
# Publication naming
# ----------------------------------------------------------------
METHOD_LABELS = {
    "RULE_ONLY": "Rule-only",
    "ISOLATION_FOREST": "Isolation Forest",
    "LOCAL_OUTLIER_FACTOR": (
        "Local Outlier Factor"
    ),
    "AI_CONSENSUS": "AI consensus",
    "HYBRID_V1": "Hybrid V1",
    "HYBRID_V2_CALIBRATED": (
        "Hybrid V2"
    ),
    "HYBRID_V3_NESTED": (
        "Hybrid V3"
    ),
}


DATASET_LABELS = {
    "finance": "Finance",
    "healthcare": "Healthcare",
    "retail": "Retail",
}


# ----------------------------------------------------------------
# Formatting helpers
# ----------------------------------------------------------------
def format_decimal(
    value,
    digits=3,
):
    if pd.isna(value):
        return ""

    return f"{float(value):.{digits}f}"


def format_p_value(
    value,
):
    if pd.isna(value):
        return ""

    value = float(value)

    if value < 0.001:
        return "<0.001"

    return f"{value:.3f}"


def significance_symbol(
    adjusted_p_value,
):
    if pd.isna(
        adjusted_p_value
    ):
        return ""

    adjusted_p_value = float(
        adjusted_p_value
    )

    if adjusted_p_value < 0.001:
        return "***"

    if adjusted_p_value < 0.01:
        return "**"

    if adjusted_p_value < 0.05:
        return "*"

    return "ns"


# ----------------------------------------------------------------
# Table 1 — Nested-CV overall performance
# ----------------------------------------------------------------
table_1_df = (
    nested_cv_summary_df[
        [
            "method",
            "mean_macro_precision",
            "std_macro_precision",
            "mean_macro_recall",
            "std_macro_recall",
            "mean_macro_f1",
            "std_macro_f1",
            "mean_macro_fpr",
            "std_macro_fpr",
            "mean_macro_fnr",
            "mean_macro_balanced_accuracy",
            "mean_macro_pr_auc",
        ]
    ]
    .copy()
)

table_1_df[
    "Method"
] = (
    table_1_df[
        "method"
    ]
    .map(
        METHOD_LABELS
    )
    .fillna(
        table_1_df[
            "method"
        ]
    )
)

table_1_df[
    "Precision"
] = table_1_df.apply(
    lambda row: (
        f"{row['mean_macro_precision']:.3f} "
        f"± {row['std_macro_precision']:.3f}"
    ),
    axis=1,
)

table_1_df[
    "Recall"
] = table_1_df.apply(
    lambda row: (
        f"{row['mean_macro_recall']:.3f} "
        f"± {row['std_macro_recall']:.3f}"
    ),
    axis=1,
)

table_1_df[
    "F1"
] = table_1_df.apply(
    lambda row: (
        f"{row['mean_macro_f1']:.3f} "
        f"± {row['std_macro_f1']:.3f}"
    ),
    axis=1,
)

table_1_df[
    "FPR"
] = table_1_df.apply(
    lambda row: (
        f"{row['mean_macro_fpr']:.3f} "
        f"± {row['std_macro_fpr']:.3f}"
    ),
    axis=1,
)

table_1_df[
    "FNR"
] = table_1_df[
    "mean_macro_fnr"
].map(
    lambda value: (
        format_decimal(
            value
        )
    )
)

table_1_df[
    "Balanced accuracy"
] = table_1_df[
    "mean_macro_balanced_accuracy"
].map(
    lambda value: (
        format_decimal(
            value
        )
    )
)

table_1_df[
    "PR-AUC"
] = table_1_df[
    "mean_macro_pr_auc"
].map(
    lambda value: (
        format_decimal(
            value
        )
    )
)

table_1_publication_df = (
    table_1_df[
        [
            "Method",
            "Precision",
            "Recall",
            "F1",
            "FPR",
            "FNR",
            "Balanced accuracy",
            "PR-AUC",
        ]
    ]
    .copy()
)

TABLE_1_PATH = (
    PUBLICATION_TABLES_DIR
    / "table_1_nested_cv_performance.csv"
)

table_1_publication_df.to_csv(
    TABLE_1_PATH,
    index=False,
)

display(
    table_1_publication_df
)


# ----------------------------------------------------------------
# Table 2 — Domain-specific Hybrid V3 performance
# ----------------------------------------------------------------
table_2_df = (
    dataset_oof_summary_df.copy()
)

table_2_df[
    "Dataset"
] = (
    table_2_df[
        "dataset"
    ]
    .map(
        DATASET_LABELS
    )
    .fillna(
        table_2_df[
            "dataset"
        ]
    )
)

table_2_publication_df = pd.DataFrame({
    "Dataset": (
        table_2_df[
            "Dataset"
        ]
    ),
    "Experiments": (
        table_2_df[
            "experiments"
        ]
        .astype(int)
    ),
    "Records": (
        table_2_df[
            "records"
        ]
        .astype(int)
    ),
    "Precision": (
        table_2_df[
            "macro_precision"
        ]
        .map(
            format_decimal
        )
    ),
    "Recall": (
        table_2_df[
            "macro_recall"
        ]
        .map(
            format_decimal
        )
    ),
    "F1": (
        table_2_df[
            "macro_f1"
        ]
        .map(
            format_decimal
        )
    ),
    "FPR": (
        table_2_df[
            "macro_fpr"
        ]
        .map(
            format_decimal
        )
    ),
    "FNR": (
        table_2_df[
            "macro_fnr"
        ]
        .map(
            format_decimal
        )
    ),
    "Balanced accuracy": (
        table_2_df[
            "macro_balanced_accuracy"
        ]
        .map(
            format_decimal
        )
    ),
    "PR-AUC": (
        table_2_df[
            "macro_pr_auc"
        ]
        .map(
            format_decimal
        )
    ),
})

TABLE_2_PATH = (
    PUBLICATION_TABLES_DIR
    / "table_2_domain_specific_v3_performance.csv"
)

table_2_publication_df.to_csv(
    TABLE_2_PATH,
    index=False,
)

display(
    table_2_publication_df
)


# ----------------------------------------------------------------
# Table 3 — Key paired statistical comparisons
# ----------------------------------------------------------------
key_statistical_comparisons = [
    {
        "metric": "f1",
        "method_a": "HYBRID_V3_NESTED",
        "method_b": "AI_CONSENSUS",
    },
    {
        "metric": "precision",
        "method_a": "HYBRID_V3_NESTED",
        "method_b": "AI_CONSENSUS",
    },
    {
        "metric": "recall",
        "method_a": "HYBRID_V3_NESTED",
        "method_b": "AI_CONSENSUS",
    },
    {
        "metric": "false_positive_rate",
        "method_a": "HYBRID_V3_NESTED",
        "method_b": "AI_CONSENSUS",
    },
    {
        "metric": "false_positive_rate",
        "method_a": "HYBRID_V3_NESTED",
        "method_b": "HYBRID_V1",
    },
    {
        "metric": "f1",
        "method_a": "HYBRID_V3_NESTED",
        "method_b": "RULE_ONLY",
    },
]

table_3_rows = []

for comparison in (
    key_statistical_comparisons
):
    matching_row_df = (
        pairwise_statistical_tests_df.loc[
            (
                pairwise_statistical_tests_df[
                    "metric"
                ].eq(
                    comparison[
                        "metric"
                    ]
                )
            )
            & (
                pairwise_statistical_tests_df[
                    "method_a"
                ].eq(
                    comparison[
                        "method_a"
                    ]
                )
            )
            & (
                pairwise_statistical_tests_df[
                    "method_b"
                ].eq(
                    comparison[
                        "method_b"
                    ]
                )
            )
        ]
    )

    if matching_row_df.empty:
        raise ValueError(
            "Missing statistical comparison: "
            f"{comparison}"
        )

    row = matching_row_df.iloc[0]

    table_3_rows.append({
        "Metric": (
            comparison[
                "metric"
            ]
            .replace(
                "_",
                " ",
            )
            .title()
        ),
        "Comparison": (
            f"{METHOD_LABELS.get(row['method_a'], row['method_a'])} "
            f"− {METHOD_LABELS.get(row['method_b'], row['method_b'])}"
        ),
        "Mean difference": (
            format_decimal(
                row[
                    "mean_difference"
                ],
                digits=4,
            )
        ),
        "95% bootstrap CI": (
            f"[{row['ci_95_lower']:.4f}, "
            f"{row['ci_95_upper']:.4f}]"
        ),
        "Adjusted p-value": (
            format_p_value(
                row[
                    "holm_adjusted_p_value"
                ]
            )
        ),
        "Significance": (
            significance_symbol(
                row[
                    "holm_adjusted_p_value"
                ]
            )
        ),
    })


table_3_publication_df = pd.DataFrame(
    table_3_rows
)

TABLE_3_PATH = (
    PUBLICATION_TABLES_DIR
    / "table_3_key_statistical_comparisons.csv"
)

table_3_publication_df.to_csv(
    TABLE_3_PATH,
    index=False,
)

display(
    table_3_publication_df
)


# ----------------------------------------------------------------
# Table 4 — Operational burden
# ----------------------------------------------------------------
table_4_df = (
    operational_burden_df.copy()
)

table_4_df[
    "Method"
] = (
    table_4_df[
        "method"
    ]
    .map(
        METHOD_LABELS
    )
    .fillna(
        table_4_df[
            "method"
        ]
    )
)

table_4_publication_df = pd.DataFrame({
    "Method": (
        table_4_df[
            "Method"
        ]
    ),
    "Flagged records": (
        table_4_df[
            "flagged_records"
        ]
        .astype(int)
    ),
    "Flagged rate": (
        table_4_df[
            "flagged_record_rate"
        ]
        .map(
            lambda value: (
                f"{value:.1%}"
            )
        )
    ),
    "True positives": (
        table_4_df[
            "true_positive_records"
        ]
        .astype(int)
    ),
    "False positives": (
        table_4_df[
            "false_positive_records"
        ]
        .astype(int)
    ),
    "Missed anomalies": (
        table_4_df[
            "missed_anomalies"
        ]
        .astype(int)
    ),
    "Anomaly capture rate": (
        table_4_df[
            "anomalies_captured_rate"
        ]
        .map(
            lambda value: (
                f"{value:.1%}"
            )
        )
    ),
    "False reviews per TP": (
        table_4_df[
            "false_reviews_per_true_positive"
        ]
        .map(
            lambda value: (
                format_decimal(
                    value
                )
            )
        )
    ),
})

TABLE_4_PATH = (
    PUBLICATION_TABLES_DIR
    / "table_4_operational_burden.csv"
)

table_4_publication_df.to_csv(
    TABLE_4_PATH,
    index=False,
)

display(
    table_4_publication_df
)


# ----------------------------------------------------------------
# Table 5 — Selected nested-CV configurations
# ----------------------------------------------------------------
table_5_df = (
    selected_configurations_df.copy()
)

table_5_publication_df = pd.DataFrame({
    "Outer fold": (
        table_5_df[
            "outer_fold"
        ]
        .astype(int)
    ),
    "Candidate": (
        table_5_df[
            "candidate_id"
        ]
    ),
    "Threshold": (
        table_5_df[
            "threshold"
        ]
        .map(
            format_decimal
        )
    ),
    "Rule weight": (
        table_5_df[
            "rule_risk_weight"
        ]
        .map(
            format_decimal
        )
    ),
    "AI consensus weight": (
        table_5_df[
            "ai_consensus_weight"
        ]
        .map(
            format_decimal
        )
    ),
    "AI vote weight": (
        table_5_df[
            "ai_prediction_vote_weight"
        ]
        .map(
            format_decimal
        )
    ),
    "Uncertainty weight": (
        table_5_df[
            "uncertainty_weight"
        ]
        .map(
            format_decimal
        )
    ),
    "Inner F1": (
        table_5_df[
            "inner_validation_f1"
        ]
        .map(
            format_decimal
        )
    ),
    "Inner FPR": (
        table_5_df[
            "inner_validation_fpr"
        ]
        .map(
            format_decimal
        )
    ),
    "Constraint satisfied": (
        table_5_df[
            "fpr_constraint_satisfied"
        ]
    ),
})

TABLE_5_PATH = (
    PUBLICATION_TABLES_DIR
    / "table_5_nested_selected_configurations.csv"
)

table_5_publication_df.to_csv(
    TABLE_5_PATH,
    index=False,
)

display(
    table_5_publication_df
)


# ----------------------------------------------------------------
# Figure 1 — Overall method comparison
# ----------------------------------------------------------------
figure_1_df = (
    nested_cv_summary_df[
        [
            "method",
            "mean_macro_precision",
            "mean_macro_recall",
            "mean_macro_f1",
            "mean_macro_fpr",
        ]
    ]
    .copy()
)

figure_1_df[
    "method_label"
] = (
    figure_1_df[
        "method"
    ]
    .map(
        METHOD_LABELS
    )
    .fillna(
        figure_1_df[
            "method"
        ]
    )
)

figure_1_long_df = (
    figure_1_df
    .melt(
        id_vars=[
            "method_label",
        ],
        value_vars=[
            "mean_macro_precision",
            "mean_macro_recall",
            "mean_macro_f1",
            "mean_macro_fpr",
        ],
        var_name="metric",
        value_name="value",
    )
)

figure_1_metric_labels = {
    "mean_macro_precision": "Precision",
    "mean_macro_recall": "Recall",
    "mean_macro_f1": "F1",
    "mean_macro_fpr": "FPR",
}

figure_1_long_df[
    "metric_label"
] = (
    figure_1_long_df[
        "metric"
    ]
    .map(
        figure_1_metric_labels
    )
)

method_order = (
    figure_1_df[
        "method_label"
    ]
    .tolist()
)

metric_order = [
    "Precision",
    "Recall",
    "F1",
    "FPR",
]

x_positions = np.arange(
    len(
        method_order
    )
)

bar_width = 0.18

fig, ax = plt.subplots(
    figsize=(
        10,
        6,
    )
)

for metric_index, metric_label in enumerate(
    metric_order
):
    metric_values = []

    for method_label in method_order:
        value = (
            figure_1_long_df.loc[
                (
                    figure_1_long_df[
                        "method_label"
                    ].eq(
                        method_label
                    )
                )
                & (
                    figure_1_long_df[
                        "metric_label"
                    ].eq(
                        metric_label
                    )
                ),
                "value",
            ]
            .iloc[0]
        )

        metric_values.append(
            value
        )

    ax.bar(
        x_positions
        + (
            metric_index
            - 1.5
        )
        * bar_width,
        metric_values,
        width=bar_width,
        label=metric_label,
    )

ax.set_xlabel(
    "Method"
)

ax.set_ylabel(
    "Macro-averaged score"
)

ax.set_title(
    "Nested cross-validation performance by method"
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    method_order
)

ax.set_ylim(
    0.0,
    0.65,
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.3,
)

fig.tight_layout()

FIGURE_1_PATH = (
    PUBLICATION_FIGURES_DIR
    / "figure_1_overall_method_comparison.png"
)

fig.savefig(
    FIGURE_1_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ----------------------------------------------------------------
# Figure 2 — Domain-specific F1 comparison
# ----------------------------------------------------------------
domain_f1_source_df = (
    statistical_experiment_metrics_df
    .groupby(
        [
            "dataset",
            "method",
        ],
        dropna=False,
    )
    .agg(
        macro_f1=(
            "f1",
            "mean",
        ),
        f1_std=(
            "f1",
            "std",
        ),
    )
    .reset_index()
)

domain_methods = [
    "AI_CONSENSUS",
    "HYBRID_V1",
    "HYBRID_V3_NESTED",
]

domain_datasets = [
    "finance",
    "healthcare",
    "retail",
]

x_positions = np.arange(
    len(
        domain_datasets
    )
)

bar_width = 0.24

fig, ax = plt.subplots(
    figsize=(
        10,
        6,
    )
)

for method_index, method_name in enumerate(
    domain_methods
):
    method_values = []
    method_errors = []

    for dataset_name in domain_datasets:
        matching_row = (
            domain_f1_source_df.loc[
                (
                    domain_f1_source_df[
                        "dataset"
                    ].eq(
                        dataset_name
                    )
                )
                & (
                    domain_f1_source_df[
                        "method"
                    ].eq(
                        method_name
                    )
                )
            ]
            .iloc[0]
        )

        method_values.append(
            matching_row[
                "macro_f1"
            ]
        )

        method_errors.append(
            matching_row[
                "f1_std"
            ]
        )

    ax.bar(
        x_positions
        + (
            method_index
            - 1
        )
        * bar_width,
        method_values,
        width=bar_width,
        yerr=method_errors,
        capsize=3,
        label=METHOD_LABELS[
            method_name
        ],
    )

ax.set_xlabel(
    "Dataset"
)

ax.set_ylabel(
    "Macro-F1"
)

ax.set_title(
    "Domain-specific anomaly-detection performance"
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    [
        DATASET_LABELS[
            dataset_name
        ]
        for dataset_name
        in domain_datasets
    ]
)

ax.set_ylim(
    0.0,
    0.65,
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.3,
)

fig.tight_layout()

FIGURE_2_PATH = (
    PUBLICATION_FIGURES_DIR
    / "figure_2_domain_specific_f1.png"
)

fig.savefig(
    FIGURE_2_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ----------------------------------------------------------------
# Figure 3 — F1 versus false-positive-rate trade-off
# ----------------------------------------------------------------
tradeoff_df = (
    nested_cv_summary_df[
        [
            "method",
            "mean_macro_f1",
            "mean_macro_fpr",
        ]
    ]
    .copy()
)

tradeoff_df[
    "method_label"
] = (
    tradeoff_df[
        "method"
    ]
    .map(
        METHOD_LABELS
    )
    .fillna(
        tradeoff_df[
            "method"
        ]
    )
)

fig, ax = plt.subplots(
    figsize=(
        8,
        6,
    )
)

ax.scatter(
    tradeoff_df[
        "mean_macro_fpr"
    ],
    tradeoff_df[
        "mean_macro_f1"
    ],
    s=90,
)

for _, row in (
    tradeoff_df.iterrows()
):
    ax.annotate(
        row[
            "method_label"
        ],
        (
            row[
                "mean_macro_fpr"
            ],
            row[
                "mean_macro_f1"
            ],
        ),
        xytext=(
            6,
            6,
        ),
        textcoords="offset points",
    )

ax.set_xlabel(
    "Macro false-positive rate"
)

ax.set_ylabel(
    "Macro-F1"
)

ax.set_title(
    "Detection performance and false-positive trade-off"
)

ax.grid(
    alpha=0.3,
)

fig.tight_layout()

FIGURE_3_PATH = (
    PUBLICATION_FIGURES_DIR
    / "figure_3_f1_fpr_tradeoff.png"
)

fig.savefig(
    FIGURE_3_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ----------------------------------------------------------------
# Figure 4 — Operational intervention burden
# ----------------------------------------------------------------
figure_4_df = (
    operational_burden_df.copy()
)

figure_4_df[
    "method_label"
] = (
    figure_4_df[
        "method"
    ]
    .map(
        METHOD_LABELS
    )
    .fillna(
        figure_4_df[
            "method"
        ]
    )
)

figure_4_df = (
    figure_4_df
    .sort_values(
        "false_reviews_per_true_positive",
        ascending=True,
    )
)

fig, ax = plt.subplots(
    figsize=(
        9,
        6,
    )
)

ax.bar(
    figure_4_df[
        "method_label"
    ],
    figure_4_df[
        "false_reviews_per_true_positive"
    ],
)

ax.set_xlabel(
    "Method"
)

ax.set_ylabel(
    "False reviews per true positive"
)

ax.set_title(
    "Operational review burden by method"
)

ax.tick_params(
    axis="x",
    rotation=20,
)

ax.grid(
    axis="y",
    alpha=0.3,
)

fig.tight_layout()

FIGURE_4_PATH = (
    PUBLICATION_FIGURES_DIR
    / "figure_4_operational_review_burden.png"
)

fig.savefig(
    FIGURE_4_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ----------------------------------------------------------------
# Figure 5 — Experiment-level F1 distributions
# ----------------------------------------------------------------
figure_5_df = (
    statistical_experiment_metrics_df.loc[
        statistical_experiment_metrics_df[
            "method"
        ].isin(
            [
                "AI_CONSENSUS",
                "HYBRID_V1",
                "HYBRID_V3_NESTED",
                "RULE_ONLY",
            ]
        )
    ]
    .copy()
)

boxplot_method_order = [
    "RULE_ONLY",
    "HYBRID_V1",
    "HYBRID_V3_NESTED",
    "AI_CONSENSUS",
]

boxplot_values = [
    figure_5_df.loc[
        figure_5_df[
            "method"
        ].eq(
            method_name
        ),
        "f1",
    ].to_numpy()
    for method_name
    in boxplot_method_order
]

fig, ax = plt.subplots(
    figsize=(
        9,
        6,
    )
)

ax.boxplot(
    boxplot_values,
    labels=[
        METHOD_LABELS[
            method_name
        ]
        for method_name
        in boxplot_method_order
    ],
    showmeans=True,
)

ax.set_xlabel(
    "Method"
)

ax.set_ylabel(
    "Experiment-level F1"
)

ax.set_title(
    "Distribution of F1 scores across 63 experiments"
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.grid(
    axis="y",
    alpha=0.3,
)

fig.tight_layout()

FIGURE_5_PATH = (
    PUBLICATION_FIGURES_DIR
    / "figure_5_experiment_f1_distribution.png"
)

fig.savefig(
    FIGURE_5_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# ----------------------------------------------------------------
# Create results narrative values
# ----------------------------------------------------------------
ai_summary_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "AI_CONSENSUS"
        )
    ]
    .iloc[0]
)

v1_summary_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "HYBRID_V1"
        )
    ]
    .iloc[0]
)

v3_summary_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "HYBRID_V3_NESTED"
        )
    ]
    .iloc[0]
)

v3_ai_f1_test_row = (
    pairwise_statistical_tests_df.loc[
        (
            pairwise_statistical_tests_df[
                "metric"
            ].eq(
                "f1"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_a"
            ].eq(
                "HYBRID_V3_NESTED"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_b"
            ].eq(
                "AI_CONSENSUS"
            )
        )
    ]
    .iloc[0]
)

v3_v1_fpr_test_row = (
    pairwise_statistical_tests_df.loc[
        (
            pairwise_statistical_tests_df[
                "metric"
            ].eq(
                "false_positive_rate"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_a"
            ].eq(
                "HYBRID_V3_NESTED"
            )
        )
        & (
            pairwise_statistical_tests_df[
                "method_b"
            ].eq(
                "HYBRID_V1"
            )
        )
    ]
    .iloc[0]
)


# ----------------------------------------------------------------
# Machine-readable final result summary
# ----------------------------------------------------------------
final_result_summary = {
    "artifact_version": (
        PUBLICATION_ARTIFACT_VERSION
    ),
    "hybrid_run_id": (
        HYBRID_RUN_ID
    ),
    "created_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "evaluation_design": {
        "experiments": 63,
        "datasets": 3,
        "records": int(
            len(
                nested_record_predictions_df
            )
        ),
        "outer_folds": 3,
        "inner_folds": 3,
        "split_unit": "experiment",
        "multiple_testing_correction": (
            "Holm"
        ),
        "bootstrap_iterations": 10_000,
    },
    "selected_hybrid_configuration": {
        "candidate_id": "W08",
        "threshold": 0.45,
        "rule_risk_weight": 0.20,
        "ai_consensus_weight": 0.55,
        "ai_prediction_vote_weight": 0.20,
        "uncertainty_weight": 0.05,
        "selection_consistency": (
            "Selected independently in all three outer folds"
        ),
    },
    "overall_performance": {
        "ai_consensus": {
            "macro_precision": float(
                ai_summary_row[
                    "mean_macro_precision"
                ]
            ),
            "macro_recall": float(
                ai_summary_row[
                    "mean_macro_recall"
                ]
            ),
            "macro_f1": float(
                ai_summary_row[
                    "mean_macro_f1"
                ]
            ),
            "macro_fpr": float(
                ai_summary_row[
                    "mean_macro_fpr"
                ]
            ),
        },
        "hybrid_v1": {
            "macro_precision": float(
                v1_summary_row[
                    "mean_macro_precision"
                ]
            ),
            "macro_recall": float(
                v1_summary_row[
                    "mean_macro_recall"
                ]
            ),
            "macro_f1": float(
                v1_summary_row[
                    "mean_macro_f1"
                ]
            ),
            "macro_fpr": float(
                v1_summary_row[
                    "mean_macro_fpr"
                ]
            ),
        },
        "hybrid_v3": {
            "macro_precision": float(
                v3_summary_row[
                    "mean_macro_precision"
                ]
            ),
            "macro_recall": float(
                v3_summary_row[
                    "mean_macro_recall"
                ]
            ),
            "macro_f1": float(
                v3_summary_row[
                    "mean_macro_f1"
                ]
            ),
            "macro_fpr": float(
                v3_summary_row[
                    "mean_macro_fpr"
                ]
            ),
        },
    },
    "key_statistical_results": {
        "hybrid_v3_vs_ai_f1": {
            "mean_difference": float(
                v3_ai_f1_test_row[
                    "mean_difference"
                ]
            ),
            "ci_95_lower": float(
                v3_ai_f1_test_row[
                    "ci_95_lower"
                ]
            ),
            "ci_95_upper": float(
                v3_ai_f1_test_row[
                    "ci_95_upper"
                ]
            ),
            "holm_adjusted_p_value": float(
                v3_ai_f1_test_row[
                    "holm_adjusted_p_value"
                ]
            ),
            "significant": bool(
                v3_ai_f1_test_row[
                    "statistically_significant"
                ]
            ),
        },
        "hybrid_v3_vs_v1_fpr": {
            "mean_difference": float(
                v3_v1_fpr_test_row[
                    "mean_difference"
                ]
            ),
            "ci_95_lower": float(
                v3_v1_fpr_test_row[
                    "ci_95_lower"
                ]
            ),
            "ci_95_upper": float(
                v3_v1_fpr_test_row[
                    "ci_95_upper"
                ]
            ),
            "holm_adjusted_p_value": float(
                v3_v1_fpr_test_row[
                    "holm_adjusted_p_value"
                ]
            ),
            "significant": bool(
                v3_v1_fpr_test_row[
                    "statistically_significant"
                ]
            ),
        },
    },
    "primary_conclusion": (
        "AI consensus achieved the highest numerical macro-F1, "
        "but its F1 advantage over constrained Hybrid V3 was not "
        "statistically significant after Holm correction. Hybrid V3 "
        "significantly reduced false-positive rate relative to the "
        "initial Hybrid V1 policy, while retaining an auditable "
        "evidence-fusion and governed-decision architecture."
    ),
    "claim_boundary": (
        "The evidence does not support claiming that Hybrid V3 "
        "universally outperforms AI consensus in predictive accuracy. "
        "The supported contribution is governed evidence fusion, "
        "auditable decision routing, policy calibration, and "
        "domain-dependent performance."
    ),
}


FINAL_RESULT_SUMMARY_PATH = (
    PUBLICATION_SUMMARIES_DIR
    / "final_research_result_summary.json"
)

with open(
    FINAL_RESULT_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as summary_file:
    json.dump(
        final_result_summary,
        summary_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Create manuscript-ready results text
# ----------------------------------------------------------------
results_text = f"""
RESULTS SUMMARY

Across 63 experiment-level evaluations, AI consensus achieved the
highest numerical macro-F1 ({ai_summary_row['mean_macro_f1']:.3f}),
followed by Hybrid V1 ({v1_summary_row['mean_macro_f1']:.3f}) and the
nested, false-positive-constrained Hybrid V3 policy
({v3_summary_row['mean_macro_f1']:.3f}). The mean F1 difference between
Hybrid V3 and AI consensus was
{v3_ai_f1_test_row['mean_difference']:.4f}, with a 95% paired bootstrap
confidence interval of
[{v3_ai_f1_test_row['ci_95_lower']:.4f},
{v3_ai_f1_test_row['ci_95_upper']:.4f}]. This difference was not
statistically significant after Holm correction
(adjusted p = {v3_ai_f1_test_row['holm_adjusted_p_value']:.4f}).

Hybrid V3 achieved macro precision of
{v3_summary_row['mean_macro_precision']:.3f}, macro recall of
{v3_summary_row['mean_macro_recall']:.3f}, macro-F1 of
{v3_summary_row['mean_macro_f1']:.3f}, and macro false-positive rate of
{v3_summary_row['mean_macro_fpr']:.3f}. Compared with the initial Hybrid
V1 policy, Hybrid V3 reduced the mean false-positive rate by
{abs(v3_v1_fpr_test_row['mean_difference']):.4f}. This reduction was
statistically significant after Holm correction
(adjusted p = {v3_v1_fpr_test_row['holm_adjusted_p_value']:.6g}).

Domain-level analysis demonstrated heterogeneous performance. Hybrid V3
improved mean F1 relative to AI consensus in the finance and retail
datasets but underperformed AI consensus in healthcare. These results
show that the benefit of deterministic and AI evidence fusion is
domain-dependent and that a globally fixed policy cannot be assumed to
produce uniform predictive gains.

The nested calibration procedure independently selected the same W08
configuration in every outer fold: rule-risk weight 0.20, AI-consensus
weight 0.55, AI prediction-vote weight 0.20, uncertainty weight 0.05,
and decision threshold 0.45. This consistency indicates stable policy
selection under the evaluated experiment partitions.

Overall, the results do not establish universal predictive superiority
of the hybrid method over AI consensus. Instead, they support the hybrid
framework as an auditable governance layer that combines deterministic
rule evidence, unsupervised anomaly evidence, detector agreement, and
uncertainty into reproducible risk scores and operational decisions.
""".strip()


RESULTS_TEXT_PATH = (
    PUBLICATION_SUMMARIES_DIR
    / "manuscript_ready_results_text.txt"
)

with open(
    RESULTS_TEXT_PATH,
    "w",
    encoding="utf-8",
) as results_file:
    results_file.write(
        results_text
    )

print(
    results_text
)


# ----------------------------------------------------------------
# Create figure caption file
# ----------------------------------------------------------------
figure_captions = {
    "Figure 1": (
        "Nested cross-validation comparison of macro precision, "
        "recall, F1, and false-positive rate for AI consensus, "
        "Hybrid V1, and constrained Hybrid V3."
    ),
    "Figure 2": (
        "Domain-specific macro-F1 performance across finance, "
        "healthcare, and retail experiments. Error bars represent "
        "the standard deviation across experiment-level F1 scores."
    ),
    "Figure 3": (
        "Trade-off between macro-F1 and macro false-positive rate. "
        "Points closer to the upper-left region indicate stronger "
        "detection performance with lower false-positive burden."
    ),
    "Figure 4": (
        "Operational review burden expressed as false-positive "
        "reviews generated per true-positive anomaly identified."
    ),
    "Figure 5": (
        "Distribution of experiment-level F1 scores across the 63 "
        "evaluation experiments. Boxplots show medians, quartiles, "
        "dispersion, and means."
    ),
}


FIGURE_CAPTIONS_PATH = (
    PUBLICATION_SUMMARIES_DIR
    / "figure_captions.json"
)

with open(
    FIGURE_CAPTIONS_PATH,
    "w",
    encoding="utf-8",
) as captions_file:
    json.dump(
        figure_captions,
        captions_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Create table-caption file
# ----------------------------------------------------------------
table_captions = {
    "Table 1": (
        "Overall nested cross-validation performance. Values for "
        "precision, recall, F1, and FPR are reported as mean ± "
        "standard deviation across three outer folds."
    ),
    "Table 2": (
        "Out-of-fold Hybrid V3 performance stratified by application "
        "domain."
    ),
    "Table 3": (
        "Paired experiment-level statistical comparisons using "
        "10,000 bootstrap samples and Wilcoxon signed-rank tests "
        "with Holm correction."
    ),
    "Table 4": (
        "Operational intervention burden over all 687,855 evaluated "
        "record-experiment observations."
    ),
    "Table 5": (
        "Hybrid V3 policy configurations independently selected in "
        "the three outer folds."
    ),
}


TABLE_CAPTIONS_PATH = (
    PUBLICATION_SUMMARIES_DIR
    / "table_captions.json"
)

with open(
    TABLE_CAPTIONS_PATH,
    "w",
    encoding="utf-8",
) as captions_file:
    json.dump(
        table_captions,
        captions_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Artifact manifest
# ----------------------------------------------------------------
publication_artifacts = [
    TABLE_1_PATH,
    TABLE_2_PATH,
    TABLE_3_PATH,
    TABLE_4_PATH,
    TABLE_5_PATH,
    FIGURE_1_PATH,
    FIGURE_2_PATH,
    FIGURE_3_PATH,
    FIGURE_4_PATH,
    FIGURE_5_PATH,
    FINAL_RESULT_SUMMARY_PATH,
    RESULTS_TEXT_PATH,
    FIGURE_CAPTIONS_PATH,
    TABLE_CAPTIONS_PATH,
]

artifact_manifest_rows = []

for artifact_path in publication_artifacts:
    artifact_manifest_rows.append({
        "artifact_name": (
            artifact_path.name
        ),
        "artifact_type": (
            artifact_path.suffix
            .lower()
            .replace(
                ".",
                "",
            )
        ),
        "artifact_path": str(
            artifact_path
        ),
        "exists": (
            artifact_path.exists()
        ),
        "size_bytes": (
            artifact_path.stat().st_size
            if artifact_path.exists()
            else 0
        ),
    })


publication_manifest_df = pd.DataFrame(
    artifact_manifest_rows
)

PUBLICATION_MANIFEST_PATH = (
    PUBLICATION_DIR
    / "publication_artifact_manifest.csv"
)

publication_manifest_df.to_csv(
    PUBLICATION_MANIFEST_PATH,
    index=False,
)

display(
    publication_manifest_df
)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
validation_checks = {
    "all_five_tables_created": (
        all(
            path.exists()
            for path in [
                TABLE_1_PATH,
                TABLE_2_PATH,
                TABLE_3_PATH,
                TABLE_4_PATH,
                TABLE_5_PATH,
            ]
        )
    ),
    "all_five_figures_created": (
        all(
            path.exists()
            for path in [
                FIGURE_1_PATH,
                FIGURE_2_PATH,
                FIGURE_3_PATH,
                FIGURE_4_PATH,
                FIGURE_5_PATH,
            ]
        )
    ),
    "final_summary_created": (
        FINAL_RESULT_SUMMARY_PATH.exists()
    ),
    "manuscript_text_created": (
        RESULTS_TEXT_PATH.exists()
    ),
    "figure_captions_created": (
        FIGURE_CAPTIONS_PATH.exists()
    ),
    "table_captions_created": (
        TABLE_CAPTIONS_PATH.exists()
    ),
    "all_manifest_artifacts_exist": (
        publication_manifest_df[
            "exists"
        ].all()
    ),
    "all_artifacts_nonempty": (
        (
            publication_manifest_df[
                "size_bytes"
            ]
            > 0
        ).all()
    ),
}

publication_validation_df = pd.DataFrame({
    "check": validation_checks.keys(),
    "passed": validation_checks.values(),
})

display(
    publication_validation_df
)

assert publication_validation_df[
    "passed"
].all(), (
    "One or more publication artifact checks failed."
)


# ----------------------------------------------------------------
# Save publication audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

publication_audit = {
    "hybrid_run_id": (
        HYBRID_RUN_ID
    ),
    "artifact_version": (
        PUBLICATION_ARTIFACT_VERSION
    ),
    "execution_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "tables_created": 5,
    "figures_created": 5,
    "summary_files_created": 4,
    "total_manifest_artifacts": int(
        len(
            publication_manifest_df
        )
    ),
    "publication_directory": str(
        PUBLICATION_DIR
    ),
    "manifest_path": str(
        PUBLICATION_MANIFEST_PATH
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
}

PUBLICATION_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "publication_artifact_audit.json"
)

with open(
    PUBLICATION_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        publication_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Publication-ready artifacts created successfully."
)
print(
    f"Artifact version: "
    f"{PUBLICATION_ARTIFACT_VERSION}"
)
print(
    f"Tables created: 5"
)
print(
    f"Figures created: 5"
)
print(
    f"Publication directory: "
    f"{PUBLICATION_DIR}"
)
print(
    f"Final result summary: "
    f"{FINAL_RESULT_SUMMARY_PATH}"
)
print(
    f"Manuscript-ready results: "
    f"{RESULTS_TEXT_PATH}"
)
print(
    f"Artifact manifest: "
    f"{PUBLICATION_MANIFEST_PATH}"
)
print(
    f"Publication audit: "
    f"{PUBLICATION_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

## 10. Reproducibility Completion Report

The following paths contain the final publication and verification artifacts generated by this notebook.

In [ ]:
completion_report = {
    "selected_policy": "Hybrid V3 / W08 when selected by the nested procedure",
    "figures_directory": FIGURES_DIR,
    "tables_directory": TABLES_DIR,
    "results_directory": RESULTS_DIR,
    "configuration_directory": HYBRID_CONFIG_DIR,
    "audit_directory": HYBRID_AUDIT_DIR,
}

for item, value in completion_report.items():
    print(f"{item}: {value}")